# 01. Configurações e caminhos


In [ ]:

# ============================================================
# 00. CONFIGURAÇÃO
# ============================================================



try:
    from functions import *
except Exception:
    import time
    class Temporizador:
        def iniciar(self):
            self._inicio = time.time()
            return None
        def parar(self):
            fim = time.time()
            inicio = getattr(self, '_inicio', fim)
            return fim - inicio
        def finalizar(self):
            return self.parar()
from pathlib import Path
import time
from datetime import datetime
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
try:
    from IPython.display import display
except Exception:
    display = print
    
# Iniciando Temporizador
timer = Temporizador()
timer.iniciar()
_TEMPO_TOTAL_INICIO_V58 = time.perf_counter()

PIPELINE_VERSION = "v66_19"
OUTPUT_SOLVER_FILENAME = f"bd_SOLVER_{PIPELINE_VERSION}.xlsx"
ESTRUTURA_FILENAME = "estruturas.xlsx"

# Resolve o projeto pela localização do notebook/script, sem depender de usuário, máquina ou OneDrive.
try:
    CAMINHO_EXECUCAO = Path(__file__).resolve().parent
except NameError:
    CAMINHO_EXECUCAO = Path.cwd().resolve()

def _resolver_raiz_ltp_v58(caminho_inicial):
    # Procura a raiz real do LTP pela presença simultânea das pastas operacionais do projeto.
    candidatos = [caminho_inicial, caminho_inicial.parent, caminho_inicial.parent.parent]
    for candidato in candidatos:
        if (candidato / "01_INPUT").is_dir() and (candidato / "02_OUTPUT").is_dir():
            return candidato
    # Em execução isolada, mantém o diretório corrente como raiz para facilitar testes locais.
    return caminho_inicial

RAIZ_LTP = _resolver_raiz_ltp_v58(CAMINHO_EXECUCAO)
PASTA_INPUT = RAIZ_LTP / "01_INPUT"
PASTA_OUTPUT = RAIZ_LTP / "02_OUTPUT"

# Se o notebook já estiver dentro de 02_OUTPUT, usa a própria pasta como saída e a irmã 01_INPUT.
if CAMINHO_EXECUCAO.name.upper() == "02_OUTPUT":
    PASTA_OUTPUT = CAMINHO_EXECUCAO
    PASTA_INPUT = CAMINHO_EXECUCAO.parent / "01_INPUT"

INPUT_FILE = PASTA_OUTPUT / "bd_LTP_NEC_SOLVER.xlsx"
OUTPUT_SOLVER_FILE = PASTA_OUTPUT / OUTPUT_SOLVER_FILENAME
ESTRUTURA_FILE = PASTA_INPUT / ESTRUTURA_FILENAME

# Compatibilidade apenas para teste local quando o arquivo oficial foi anexado com sufixo numérico.
if not INPUT_FILE.exists():
    candidatos_input = sorted(CAMINHO_EXECUCAO.glob("bd_LTP_NEC_SOLVER*.xlsx"))
    if candidatos_input:
        INPUT_FILE = candidatos_input[-1]

TOL = 1e-7
DATA_HORA_EXECUCAO = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
TEMPOS_ETAPAS_V58 = {}

def _iniciar_etapa_v58(nome):
    TEMPOS_ETAPAS_V58[nome] = -time.perf_counter()

def _finalizar_etapa_v58(nome):
    TEMPOS_ETAPAS_V58[nome] = TEMPOS_ETAPAS_V58.get(nome, 0.0) + time.perf_counter()
    return TEMPOS_ETAPAS_V58[nome]

print("=" * 60)
print(" KRONA - SOLVER LTP v66_19")
print("=" * 60)


# ============================================================
# CONTROLE DE EXPORTACAO DAS GUIAS
# ============================================================
# Mantem todos os calculos internos do Solver.
# As flags abaixo controlam somente quais DataFrames serao gravados no Excel final.
GUIAS_EXPORTAR = {
    "01_RESUMO_RECURSOS": False,
    "02_RESUMO_FERRAMENTAS": False,
    "03_PLANO_PRODUCAO": True,
    "04_NAO_ATEND_ITEM": False,
    "05_AUDITORIA_CAPACIDADE": False,
    "06_ALTERNATIVAS_ITEM": False,
    "07_RASTRO_RATEIO": False,
    "08_DIAGNOSTICO_ITEM": False,
    "09_DIAGNOSTICO_ALTERNATIVAS": False,
    "10_ESTRUTURA_EXPLODIDA": False,
    "11_NEC_EXPLODIDA_COMPONENTES": False,
    "12_NEC_CONSOLIDADA_SOLVER": False,
    "13_AJUSTE_PAI_FILHO": False,
    "14_RESUMO_PAI_FILHO": False,
    "15_CAPACIDADE_LIBERADA_PF": False,
    "16_REALOCACAO_CAP_LIBERADA": False,
    "18_RESUMO_ITEM_REALOCACAO": False,
}


# 02. Leitura e preparação das bases


In [ ]:
print("[01/04] Carregando e preparando bases de entrada...")
_iniciar_etapa_v58("PREPARACAO_BASES")

# ============================================================
# 01. FUNÇÕES AUXILIARES
# ============================================================

def to_num(s):
    return pd.to_numeric(s, errors="coerce").fillna(0)


def ensure_columns(df: pd.DataFrame, cols, default=0):
    for c in cols:
        if c not in df.columns:
            df[c] = default
    return df


def ajustar_nec_lote_embalagem(nec, lote_min, qtd_emb):
    """Aplica as regras industriais diretamente na NEC, de forma vetorizada.

    Regras:
    - NEC <= 0 permanece 0;
    - NEC positiva abaixo do LOTE_MIN sobe para o LOTE_MIN;
    - depois, quando QTD_EMB > 0, arredonda para cima para o próximo múltiplo válido.
    """
    nec = pd.to_numeric(nec, errors="coerce").fillna(0.0)
    lote_min = pd.to_numeric(lote_min, errors="coerce").fillna(0.0).clip(lower=0.0)
    qtd_emb = pd.to_numeric(qtd_emb, errors="coerce").fillna(0.0).clip(lower=0.0)

    nec_ajustada = nec.clip(lower=0.0)
    mask_pos = nec_ajustada > 0
    nec_ajustada = nec_ajustada.where(~mask_pos, np.maximum(nec_ajustada, lote_min))

    mask_emb = mask_pos & (qtd_emb > 0)
    nec_ajustada.loc[mask_emb] = np.ceil((nec_ajustada.loc[mask_emb] - 1e-12) / qtd_emb.loc[mask_emb]) * qtd_emb.loc[mask_emb]
    return nec_ajustada


def is_valid_alt(s):
    """Converte a coluna ALTERNATIVA_VALIDA para booleano de forma segura.

    Aceita booleanos reais e textos comuns como TRUE, SIM, YES, 1, VALIDA e ALTERNATIVA_VALIDA.
    Qualquer valor vazio ou diferente disso vira False.
    """
    if isinstance(s, pd.Series):
        if str(s.dtype).lower() == "bool":
            return s.fillna(False)
        txt = s.fillna(False).astype(str).str.strip().str.upper()
        return txt.isin(["TRUE", "VERDADEIRO", "SIM", "YES", "Y", "1", "VALIDA", "VÁLIDA", "ALTERNATIVA_VALIDA"])
    return bool(s)


def calcular_nec_pcs_se_necessario(df: pd.DataFrame, aplicar_regras_industriais: bool = True) -> pd.DataFrame:
    """Calcula NEC_PCS e permite separar otimização contínua do fechamento industrial.

    aplicar_regras_industriais=True:
        preserva o comportamento histórico da V66, aplicando LIMIT_PCS, LOTE_MIN e QTD_EMB.
    aplicar_regras_industriais=False:
        calcula a NEC contínua usada nas iterações da Matriz, sem LIMIT_PCS, LOTE_MIN ou QTD_EMB.
    """
    df = df.copy()
    if "NEC_PCS" in df.columns and aplicar_regras_industriais:
        df = ensure_columns(df, ["LOTE_MIN", "QTD_EMB"], 0)
        df["NEC_PCS"] = to_num(df["NEC_PCS"]).clip(lower=0.0)
        df["NEC_PCS"] = ajustar_nec_lote_embalagem(df["NEC_PCS"], df["LOTE_MIN"], df["QTD_EMB"])
        if "NEC_ESTRUTURA_PAI_PCS" not in df.columns:
            df["NEC_DIRETA_BRUTA_PCS"] = df["NEC_PCS"]
            df["NEC_ESTRUTURA_PAI_PCS"] = df["NEC_PCS"]
        else:
            df["NEC_ESTRUTURA_PAI_PCS"] = to_num(df["NEC_ESTRUTURA_PAI_PCS"]).clip(lower=0.0)
            df["NEC_ESTRUTURA_PAI_PCS"] = ajustar_nec_lote_embalagem(
                df["NEC_ESTRUTURA_PAI_PCS"], df["LOTE_MIN"], df["QTD_EMB"]
            )
        return df

    # Em modo continuo, NEC_PCS preexistente nao e fonte de verdade, pois pode
    # carregar LIMIT_PCS/LOTE_MIN/QTD_EMB de uma etapa anterior. A NEC e
    # reconstruida abaixo diretamente pelos componentes LTP.

    required = [
        "LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS", "LTP_EST_SEG_PCS",
        "LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS",
        "LTP_SALDO_PREV_PROX_MES_PCS", "LTP_COMP_NEC_PCS", "LIMIT_PCS", "LOTE_MIN", "QTD_EMB"
    ]
    df = ensure_columns(df, required, 0)
    for c in required:
        df[c] = to_num(df[c])

    # Demanda bruta antes de deduzir estoque/ORI/transito.
    # Esta demanda é usada como base da explosão da estrutura, seguindo o comportamento do motor antigo
    # que explodia componentes antes de o corte/solver eliminar itens por estoque/capacidade.
    demanda_bruta = (
        df["LTP_CART_ARR_MES_ANT"] + df["LTP_CART_MES_ATUAL"] + df["LTP_SALDO_PREV_PCS"] + df["LTP_EST_SEG_PCS"]
    )
    mesma_reg_nao = df.get("MESMA_REG", "SIM").astype(str).str.upper().eq("NAO")
    demanda_bruta = np.where(mesma_reg_nao, demanda_bruta + df["LTP_SALDO_PREV_PROX_MES_PCS"], demanda_bruta)
    demanda_bruta = pd.Series(demanda_bruta, index=df.index).clip(lower=0)
    demanda_bruta = demanda_bruta + df["LTP_COMP_NEC_PCS"]
    if aplicar_regras_industriais:
        demanda_bruta = np.maximum(demanda_bruta, df["LIMIT_PCS"])

    # Demanda líquida para o solver: deduz estoque/transito/ORI/triangulação.
    base_comum = (
        df["LTP_CART_ARR_MES_ANT"] + df["LTP_CART_MES_ATUAL"] + df["LTP_SALDO_PREV_PCS"] + df["LTP_EST_SEG_PCS"]
        - df["LTP_EST_INI_PCS"] - df["LTP_EST_TRANS_PCS"] - df["ORI_TOT_PCS"] - df["TRIANG_TOT_PCS"]
    )
    nec = np.where(mesma_reg_nao, base_comum + df["LTP_SALDO_PREV_PROX_MES_PCS"], base_comum)
    nec = pd.Series(nec, index=df.index).clip(lower=0)
    nec = nec + df["LTP_COMP_NEC_PCS"]
    if aplicar_regras_industriais:
        nec = np.maximum(nec, df["LIMIT_PCS"])

    nec = pd.Series(nec, index=df.index).fillna(0.0).clip(lower=0.0)
    if aplicar_regras_industriais:
        df["NEC_PCS"] = ajustar_nec_lote_embalagem(nec, df["LOTE_MIN"], df["QTD_EMB"])
    else:
        df["NEC_PCS"] = nec

    # Demanda bruta calculada para auditoria e para exceções de explosão estrutural.
    # Por padrão, a explosão usa a NEC líquida do pai (NEC_PCS).
    # Exceções de retorno de componente/transferência são ativadas depois que a estrutura é conhecida.
    nec_estrutura_bruta = demanda_bruta.copy()
    df["NEC_DIRETA_BRUTA_PCS"] = pd.Series(nec_estrutura_bruta, index=df.index).fillna(0)
    df["NEC_ESTRUTURA_PAI_PCS"] = df["NEC_PCS"]
    return df


def preparar_base(df: pd.DataFrame):
    df = df.copy()
    for c in ["PRIOR_MATPAR", "PRIOR_ROT", "PCS_HORA", "HOR_REC", "HOR_FER", "NEC_PCS"]:
        if c in df.columns:
            df[c] = to_num(df[c])
    df["HOR_CAP"] = df[["HOR_REC", "HOR_FER"]].min(axis=1)
    mes_ref = df["MES_REF"].min()
    bd_mes = df[df["MES_REF"].eq(mes_ref)].copy()
    return bd_mes


def classificar_motivo_invalida(row):
    motivos = []
    if pd.isna(row.get("ALOC_REC")) or str(row.get("ALOC_REC", "")).strip() == "":
        motivos.append("ALOC_REC_AUSENTE")
    if pd.isna(row.get("COD_FER_UNID")) or str(row.get("COD_FER_UNID", "")).strip() == "":
        motivos.append("COD_FER_UNID_AUSENTE")
    if row.get("PCS_HORA", 0) <= TOL:
        motivos.append("PCS_HORA_ZERO_OU_INVALIDO")
    if row.get("HOR_REC", 0) <= TOL:
        motivos.append("HOR_REC_ZERO_OU_NEGATIVO")
    if row.get("HOR_FER", 0) <= TOL:
        motivos.append("HOR_FER_ZERO_OU_NEGATIVO")
    if row.get("HOR_CAP", 0) <= TOL:
        motivos.append("HOR_CAP_ZERO_OU_NEGATIVO")
    return " | ".join(dict.fromkeys(motivos)) if motivos else "ALTERNATIVA_VALIDA"


def formatar_excel(caminho_arquivo):
    wb = load_workbook(caminho_arquivo)
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True)
    thin_gray = Side(style="thin", color="D9E2F3")
    for ws in wb.worksheets:
        ws.freeze_panes = "A2"
        ws.sheet_view.showGridLines = False
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.border = Border(bottom=thin_gray)
        for col_idx, column_cells in enumerate(ws.columns, start=1):
            max_len = 0
            for cell in column_cells:
                if cell.value is not None:
                    max_len = max(max_len, len(str(cell.value)))
            ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 10), 45)
        ws.auto_filter.ref = ws.dimensions
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                if isinstance(cell.value, (int, float)):
                    cell.number_format = '#,##0.00'
    wb.save(caminho_arquivo)



# ============================================================
# 01B. FUNÇÕES - EXPLOSÃO DE ESTRUTURA COM ESTOQUE
# ============================================================

from collections import defaultdict, deque


def normalizar_codigo_serie(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
         .fillna("")
         .str.strip()
         .str.replace(r"\.0$", "", regex=True)
    )


def ler_estrutura_produto(caminho_estrutura: Path) -> pd.DataFrame:
    if not caminho_estrutura.exists():
        print("Arquivo de estrutura não encontrado. A explosão será ignorada:", caminho_estrutura)
        return pd.DataFrame()

    estrutura = pd.read_excel(
        caminho_estrutura,
        sheet_name="estruturas",
        dtype={"cod_prod_acabado": str, "cod_insumo": str, "empresa": str}
    )

    required = ["empresa", "cod_prod_acabado", "cod_insumo", "qtd_utilizada_pcs"]
    missing = [c for c in required if c not in estrutura.columns]
    if missing:
        raise ValueError(f"Colunas obrigatórias ausentes na estrutura: {missing}")

    estrutura = estrutura.copy()
    estrutura["empresa"] = estrutura["empresa"].astype("string").fillna("").str.strip()
    estrutura["cod_prod_acabado"] = normalizar_codigo_serie(estrutura["cod_prod_acabado"])
    estrutura["cod_insumo"] = normalizar_codigo_serie(estrutura["cod_insumo"])
    estrutura["qtd_utilizada_pcs"] = to_num(estrutura["qtd_utilizada_pcs"])
    estrutura = estrutura[
        (estrutura["empresa"] != "")
        & (estrutura["cod_prod_acabado"] != "")
        & (estrutura["cod_insumo"] != "")
    ].copy()

    estrutura = estrutura.drop_duplicates(["empresa", "cod_prod_acabado", "cod_insumo"], keep="first")
    return estrutura


def calcular_estoque_componentes_pos_demanda_direta(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula estoque disponível para a explosão após consumir a demanda direta/bruta do próprio item.

    Mantém a lógica do pipeline antigo: estoque total = estoque inicial + trânsito + ORI + triangulação.
    Depois abate a demanda bruta do item, sem descontar estoque nessa demanda, para evitar usar na estrutura
    o estoque que já estaria comprometido com a necessidade direta do próprio produto.
    """
    df = df.copy()
    required = [
        "LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS",
        "LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS",
        "LTP_SALDO_PREV_PROX_MES_PCS", "LTP_EST_SEG_PCS", "LIMIT_PCS", "LOTE_MIN", "QTD_EMB"
    ]
    df = ensure_columns(df, required, 0)
    for c in required:
        df[c] = to_num(df[c])

    df["COD_PROD"] = normalizar_codigo_serie(df["COD_PROD"])
    df["UNID_PROD"] = df["UNID_PROD"].astype("string").fillna("").str.strip()
    if "UNID_FAT" not in df.columns:
        df["UNID_FAT"] = df["UNID_PROD"]
    df["UNID_FAT"] = df["UNID_FAT"].astype("string").fillna("").str.strip()
    tipo_prod = df.get("TIPO_PROD", "").astype(str).str.upper()
    mesma_reg = df.get("MESMA_REG", "SIM").astype(str).str.upper()

    estoque_total = (
        df["LTP_EST_INI_PCS"]
        + df["LTP_EST_TRANS_PCS"]
        + df["ORI_TOT_PCS"]
        + df["TRIANG_TOT_PCS"]
    )

    demanda_base = (
        df["LTP_CART_ARR_MES_ANT"]
        + df["LTP_CART_MES_ATUAL"]
        + df["LTP_SALDO_PREV_PCS"]
        + df["LTP_EST_SEG_PCS"]
    )
    mask_mesma_reg_nao = mesma_reg.eq("NAO")
    demanda_base = demanda_base.where(~mask_mesma_reg_nao, demanda_base + df["LTP_SALDO_PREV_PROX_MES_PCS"])

    demanda_bruta = np.maximum(demanda_base, df["LIMIT_PCS"])

    df["ESTOQUE_TOTAL_PCS"] = estoque_total
    df["DEMANDA_BRUTA_DIRETA_PCS"] = np.maximum(demanda_bruta, 0)
    df["ESTOQUE_DISP_EXPLOSAO_PCS"] = np.maximum(df["ESTOQUE_TOTAL_PCS"] - df["DEMANDA_BRUTA_DIRETA_PCS"], 0)

    # Regra herdada do motor de cortes:
    # o estoque disponível para a explosão é localizado pela unidade de faturamento/destino.
    # Isso permite que uma demanda Nordeste do pai gere necessidade em uma linha do componente
    # produzida em Matriz, desde que a linha do componente tenha UNID_FAT = Nordeste.
    estoque = (
        df.groupby(["COD_PROD", "UNID_FAT"], as_index=False)
        .agg(
            ESTOQUE_TOTAL_PCS=("ESTOQUE_TOTAL_PCS", "max"),
            DEMANDA_BRUTA_DIRETA_PCS=("DEMANDA_BRUTA_DIRETA_PCS", "max"),
            ESTOQUE_DISP_EXPLOSAO_PCS=("ESTOQUE_DISP_EXPLOSAO_PCS", "max")
        )
        .rename(columns={"UNID_FAT": "UNID_PROD"})
    )
    return estoque


def explodir_estrutura_multinivel_com_estoque(
    bd_estrutura: pd.DataFrame,
    bd_demanda_raiz: pd.DataFrame,
    bd_estoque_componentes: pd.DataFrame,
    bd_mapa_componentes: pd.DataFrame = None,
    max_nivel: int = 30
):
    cols = [
        "MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM",
        "NEC_RAIZ_PCS", "COD_PROD_ACAB", "COD_INSUMO", "QTD_UTIL_PCS",
        "NEC_GERADA_BRUTA_PCS", "ESTOQUE_COMP_ANTES_PCS", "ESTOQUE_CONSUMIDO_PCS",
        "NEC_GERADA_LIQ_PCS", "ESTOQUE_COMP_DEPOIS_PCS", "NIVEL", "TRILHA"
    ]
    if bd_estrutura.empty or bd_demanda_raiz.empty:
        return pd.DataFrame(columns=cols), pd.DataFrame(columns=["UNID_PROD", "COD_PROD_ORIGEM", "TRILHA", "MOTIVO"])

    estrutura_idx = defaultdict(lambda: defaultdict(list))
    for emp, pai, filho, qtd in bd_estrutura[["empresa", "cod_prod_acabado", "cod_insumo", "qtd_utilizada_pcs"]].itertuples(index=False, name=None):
        estrutura_idx[str(emp)][str(pai)].append((str(filho), float(qtd or 0)))

    estoque_dict = {
        (str(unid), str(cod)): float(est or 0)
        for cod, unid, est in bd_estoque_componentes[["COD_PROD", "UNID_PROD", "ESTOQUE_DISP_EXPLOSAO_PCS"]].itertuples(index=False, name=None)
    }

    # Regra oficial de chaveamento da explosão/retorno:
    # A estrutura é explodida na UNID_PROD corrente e essa unidade é preservada
    # na necessidade gerada. No retorno ao LTP, o casamento é:
    #   COD_INSUMO -> COD_PROD
    #   UNID_PROD  -> UNID_FAT
    # Para continuar a explosão multinível, localiza-se o componente no LTP
    # por COD_PROD + UNID_FAT e usa-se a UNID_PROD dessa linha no próximo nível.
    if bd_mapa_componentes is None or bd_mapa_componentes.empty:
        mapa_comp = pd.DataFrame(columns=["COD_PROD", "UNID_FAT", "UNID_PROD"])
    else:
        mapa_comp = bd_mapa_componentes.copy()
    for _c in ["COD_PROD", "UNID_FAT", "UNID_PROD"]:
        if _c not in mapa_comp.columns:
            mapa_comp[_c] = ""
        mapa_comp[_c] = mapa_comp[_c].astype("string").fillna("").str.strip()
    mapa_comp["COD_PROD"] = normalizar_codigo_serie(mapa_comp["COD_PROD"])
    mapa_comp = mapa_comp.drop_duplicates(["COD_PROD", "UNID_FAT"], keep="first")
    comp_por_fat = {
        (str(cod), str(unid_fat)): str(unid_prod)
        for cod, unid_fat, unid_prod
        in mapa_comp[["COD_PROD", "UNID_FAT", "UNID_PROD"]].itertuples(index=False, name=None)
    }

    def resolver_unidade_prod_componente(cod_filho, unid_prod_explosao):
        cod_filho = str(cod_filho)
        unid_prod_explosao = str(unid_prod_explosao or "").strip()
        return comp_por_fat.get((cod_filho, unid_prod_explosao), unid_prod_explosao)

    resultados = []
    problemas = []

    ordem_cols = ["MES_REF", "UNID_PROD", "COD_PROD", "ID_PROD_UNID_FAT"]
    bd_demanda_raiz_ord = bd_demanda_raiz.sort_values([c for c in ordem_cols if c in bd_demanda_raiz.columns]).copy()

    for row in bd_demanda_raiz_ord[["MES_REF", "UNID_PROD", "UNID_FAT_ORIGEM", "ID_PROD_UNID_FAT", "COD_PROD", "NEC_PCS"]].itertuples(index=False, name=None):
        mes_ref, unid_estrutura, unid_fat_origem, id_origem, cod_origem, nec_raiz = row
        emp = str(unid_estrutura)
        unid_fat_origem = str(unid_fat_origem or emp).strip()
        root = str(cod_origem)
        nec_raiz = float(nec_raiz or 0)
        if nec_raiz <= TOL:
            continue
        estrutura_empresa = estrutura_idx.get(emp, {})
        if root not in estrutura_empresa:
            continue

        # Na fila entra a necessidade líquida do pai.
        # A fila carrega a unidade estrutural do nó para localizar seus filhos, mantendo separada a UNID_FAT da origem.
        fila = deque([(emp, root, 0, nec_raiz, root, {root})])
        while fila:
            emp_no, pai, nivel, nec_pai_liq, trilha, caminho = fila.popleft()
            if nivel >= max_nivel:
                problemas.append((emp_no, root, trilha, "MAX_NIVEL_ATINGIDO"))
                continue

            estrutura_empresa_no = estrutura_idx.get(str(emp_no), {})
            for filho, qtd_util in estrutura_empresa_no.get(pai, []):
                nivel_filho = nivel + 1
                trilha_filho = f"{trilha} -> {filho}"
                ciclo = filho in caminho
                nec_bruta = float(nec_pai_liq or 0) * float(qtd_util or 0)

                if ciclo:
                    problemas.append((emp_no, root, trilha_filho, "CICLO_ESTRUTURA"))
                    continue
                if nec_bruta <= TOL:
                    continue

                # A linha explodida preserva a UNID_PROD que gerou a necessidade.
                # No retorno ao LTP: UNID_PROD -> UNID_FAT e COD_INSUMO -> COD_PROD.
                unid_retorno = str(emp_no).strip()
                unid_prod_componente = resolver_unidade_prod_componente(filho, unid_retorno)
                chave_estoque = (unid_retorno, str(filho))
                estoque_antes = float(estoque_dict.get(chave_estoque, 0.0))
                estoque_consumido = min(estoque_antes, nec_bruta)
                nec_liq = max(nec_bruta - estoque_consumido, 0.0)
                estoque_depois = max(estoque_antes - estoque_consumido, 0.0)
                estoque_dict[chave_estoque] = estoque_depois

                resultados.append((
                    mes_ref, unid_retorno, id_origem, root, nec_raiz, pai, filho, qtd_util,
                    nec_bruta, estoque_antes, estoque_consumido, nec_liq, estoque_depois,
                    nivel_filho, trilha_filho
                ))

                # Só explode abaixo do componente se ainda existe necessidade líquida dele.
                # Para localizar os filhos do componente, usa a unidade produtiva do componente.
                if nec_liq > TOL:
                    fila.append((unid_prod_componente, filho, nivel_filho, nec_liq, trilha_filho, caminho | {filho}))

    bd_explodida = pd.DataFrame.from_records(resultados, columns=cols)
    bd_problemas = pd.DataFrame.from_records(problemas, columns=["UNID_PROD", "COD_PROD_ORIGEM", "TRILHA", "MOTIVO"])
    return bd_explodida, bd_problemas

def consolidar_necessidade_com_estrutura(bd_mes: pd.DataFrame, bd_estrutura_explodida: pd.DataFrame):
    """Consolida demanda direta e demanda de componente seguindo o motor antigo.

    Regra principal:
    - a explosão da estrutura nasce pela unidade produtiva do pai;
    - a necessidade líquida do componente volta para o LTP pelo chaveamento definido pelo cliente;
    - a chave de retorno é:
        componente.COD_PROD = COD_INSUMO explodido
        componente.UNID_FAT = UNID_PROD registrada na explosão
    - o solver passa a trabalhar com duas naturezas de demanda:
        NEC_DIRETA_PCS = demanda própria do item;
        NEC_COMP_PCS   = demanda líquida gerada por pais/componentes;
        NEC_PCS        = NEC_DIRETA_PCS + NEC_COMP_PCS.
    """
    bd_mes_out = bd_mes.copy()
    if "UNID_FAT" not in bd_mes_out.columns:
        bd_mes_out["UNID_FAT"] = bd_mes_out["UNID_PROD"]
    bd_mes_out["COD_PROD"] = normalizar_codigo_serie(bd_mes_out["COD_PROD"])
    bd_mes_out["UNID_PROD"] = bd_mes_out["UNID_PROD"].astype("string").fillna("").str.strip()
    bd_mes_out["UNID_FAT"] = bd_mes_out["UNID_FAT"].astype("string").fillna("").str.strip()

    bd_mes_out["NEC_PCS_ORIGINAL"] = to_num(bd_mes_out["NEC_PCS"])
    bd_mes_out["NEC_DIRETA_PCS"] = bd_mes_out["NEC_PCS_ORIGINAL"]

    if bd_estrutura_explodida is None or bd_estrutura_explodida.empty:
        nec_comp_retorno = pd.DataFrame(columns=[
            "MES_REF", "UNID_FAT", "COD_PROD",
            "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_COMP_PCS"
        ])
    else:
        nec_comp_retorno = (
            bd_estrutura_explodida
            .groupby(["MES_REF", "UNID_PROD", "COD_INSUMO"], as_index=False)
            .agg(
                NEC_EXPLODIDA_BRUTA_PCS=("NEC_GERADA_BRUTA_PCS", "sum"),
                ESTOQUE_CONSUMIDO_EXPLOSAO_PCS=("ESTOQUE_CONSUMIDO_PCS", "sum"),
                NEC_COMP_PCS=("NEC_GERADA_LIQ_PCS", "sum")
            )
            .rename(columns={"UNID_PROD": "UNID_FAT", "COD_INSUMO": "COD_PROD"})
        )
        nec_comp_retorno["COD_PROD"] = normalizar_codigo_serie(nec_comp_retorno["COD_PROD"])
        nec_comp_retorno["UNID_FAT"] = nec_comp_retorno["UNID_FAT"].astype("string").fillna("").str.strip()

    bd_mes_out = bd_mes_out.merge(
        nec_comp_retorno,
        on=["MES_REF", "UNID_FAT", "COD_PROD"],
        how="left"
    )

    for c in ["NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_COMP_PCS"]:
        bd_mes_out[c] = to_num(bd_mes_out[c])

    # Alias mantido para compatibilidade com guias antigas.
    bd_mes_out["NEC_EXPLODIDA_PCS"] = bd_mes_out["NEC_COMP_PCS"]
    bd_mes_out["LTP_COMP_NEC_PCS"] = bd_mes_out["NEC_COMP_PCS"]
    bd_mes_out["NEC_PCS_CONSOLIDADA"] = bd_mes_out["NEC_DIRETA_PCS"] + bd_mes_out["NEC_COMP_PCS"]
    bd_mes_out = ensure_columns(bd_mes_out, ["LOTE_MIN", "QTD_EMB"], 0)
    bd_mes_out["NEC_PCS_CONSOLIDADA"] = ajustar_nec_lote_embalagem(bd_mes_out["NEC_PCS_CONSOLIDADA"], bd_mes_out["LOTE_MIN"], bd_mes_out["QTD_EMB"])
    bd_mes_out["NEC_PCS"] = bd_mes_out["NEC_PCS_CONSOLIDADA"]

    chave_item = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "UNID_PROD", "UNID_FAT"]
    extras = ["DESC_PROD", "TIPO_PROD"]
    agg = {
        "NEC_DIRETA_PCS": "max",
        "NEC_COMP_PCS": "max",
        "NEC_EXPLODIDA_BRUTA_PCS": "max",
        "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS": "max",
        "NEC_EXPLODIDA_PCS": "max",
        "NEC_PCS_CONSOLIDADA": "max",
    }
    for c in extras:
        if c in bd_mes_out.columns:
            agg[c] = "first"
    nec_consolidada = (
        bd_mes_out.groupby([c for c in chave_item if c in bd_mes_out.columns], as_index=False)
        .agg(agg)
    )
    nec_consolidada["TEM_ROTEIRO_NO_SOLVER"] = nec_consolidada["ID_PROD_UNID_FAT"].notna()

    return bd_mes_out, nec_consolidada

# ============================================================
# 01C. FUNÇÕES - AJUSTE PÓS-SOLVER PAI/FILHO - rotina operacional MOTIVO AJUSTADO
# ============================================================

def calcular_ajuste_pos_solver_pai_filho(estrutura_explodida: pd.DataFrame, nec_consolidada_solver: pd.DataFrame, nao_atend_item: pd.DataFrame, plano_producao: pd.DataFrame):
    """Calcula diagnóstico pós-solver de viabilidade pai/filho.

    rotina:
    - o filho/componente sustenta o pai com estoque já consumido na explosão + produção do filho pelo solver;
    - a produção do filho continua distribuída proporcionalmente entre demanda direta e estrutural quando houver necessidade total compartilhada;
    - todo corte do pai recalcula a capacidade liberada para realocação posterior.
    """
    cols_ajuste = [
        "MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM", "COD_PROD_ACAB", "COD_INSUMO", "NIVEL", "TRILHA", "QTD_UTIL_PCS",
        "NEC_RAIZ_PCS", "QTD_PAI_SOLVER", "FATOR_ATEND_PAI_SOLVER", "NEC_GERADA_LIQ_PCS", "ESTOQUE_CONSUMIDO_PCS", "NEC_FILHO_POS_CORTE_PAI_PCS",
        "NEC_DIRETA_FILHO_PCS", "NEC_TOTAL_FILHO_SOLVER_PCS", "QTD_FILHO_SOLVER_TOTAL", "QTD_FILHO_SOLVER_ESTRUTURA_PCS", "QTD_FILHO_DISPONIVEL_ESTRUTURA_PCS",
        "TEM_ROTEIRO_FILHO_NO_SOLVER", "FATOR_ATEND_FILHO_TOTAL", "FATOR_FILHO_USADO_PARA_LIMITAR", "NEC_FILHO_ATENDIDA_ESTRUTURA_PCS",
        "QTD_PAI_FINAL_VIAVEL_EST", "FATOR_FINAL_VIAVEL_ORIGEM", "MOTIVO_AJUSTE_PAI_FILHO"
    ]
    cols_resumo = [
        "MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM", "NEC_RAIZ_PCS", "QTD_PAI_SOLVER", "QTD_PAI_FINAL_VIAVEL_EST",
        "QTD_REDUCAO_POS_PAI_FILHO", "FATOR_ATEND_PAI_SOLVER", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER", "QTD_COMPONENTES_ANALISADOS",
        "QTD_COMPONENTES_LIMITANTES", "QTD_COMPONENTES_SEM_ROTEIRO", "MOTIVO_AJUSTE_PAI_FILHO"
    ]
    if estrutura_explodida is None or estrutura_explodida.empty:
        return pd.DataFrame(columns=cols_ajuste), pd.DataFrame(columns=cols_resumo), pd.DataFrame()

    ajuste = estrutura_explodida.copy()
    for c in ["NEC_RAIZ_PCS", "NEC_GERADA_LIQ_PCS", "QTD_UTIL_PCS", "ESTOQUE_CONSUMIDO_PCS"]:
        if c not in ajuste.columns:
            ajuste[c] = 0.0
        ajuste[c] = to_num(ajuste[c])

    cols_atend_base = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "UNID_PROD", "UNID_FAT", "QTD_ATENDIDA_SOLVER", "NEC_PCS", "NEC_DIRETA_PCS", "NEC_COMP_PCS", "NEC_PCS_CONSOLIDADA"]
    cols_atend_base = [c for c in cols_atend_base if c in nao_atend_item.columns]
    atend = nao_atend_item[cols_atend_base].copy()
    for c in ["QTD_ATENDIDA_SOLVER", "NEC_PCS", "NEC_DIRETA_PCS", "NEC_COMP_PCS", "NEC_PCS_CONSOLIDADA"]:
        if c not in atend.columns:
            atend[c] = 0.0
        atend[c] = to_num(atend[c])

    if nec_consolidada_solver is not None and not nec_consolidada_solver.empty and "TEM_ROTEIRO_NO_SOLVER" in nec_consolidada_solver.columns:
        roteiro_filho = nec_consolidada_solver[["MES_REF", "COD_PROD", "UNID_FAT", "TEM_ROTEIRO_NO_SOLVER"]].copy()
        roteiro_filho = roteiro_filho.rename(columns={"COD_PROD": "COD_INSUMO", "UNID_FAT": "UNID_PROD", "TEM_ROTEIRO_NO_SOLVER": "TEM_ROTEIRO_FILHO_NO_SOLVER"})
    else:
        roteiro_filho = pd.DataFrame(columns=["MES_REF", "COD_INSUMO", "UNID_PROD", "TEM_ROTEIRO_FILHO_NO_SOLVER"])

    atend_pai = atend.rename(columns={
        "ID_PROD_UNID_FAT": "ID_PROD_UNID_FAT_ORIGEM",
        "QTD_ATENDIDA_SOLVER": "QTD_PAI_SOLVER",
        "NEC_PCS": "NEC_PAI_SOLVER_PCS"
    })[["MES_REF", "ID_PROD_UNID_FAT_ORIGEM", "QTD_PAI_SOLVER", "NEC_PAI_SOLVER_PCS"]]
    ajuste = ajuste.merge(atend_pai, on=["MES_REF", "ID_PROD_UNID_FAT_ORIGEM"], how="left")
    ajuste["QTD_PAI_SOLVER"] = ajuste["QTD_PAI_SOLVER"].fillna(0)
    ajuste["FATOR_ATEND_PAI_SOLVER"] = np.where(ajuste["NEC_RAIZ_PCS"] > TOL, (ajuste["QTD_PAI_SOLVER"] / ajuste["NEC_RAIZ_PCS"]).clip(0, 1), 0)
    ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] = ajuste["NEC_GERADA_LIQ_PCS"] * ajuste["FATOR_ATEND_PAI_SOLVER"]

    atend_filho = atend.rename(columns={
        "COD_PROD": "COD_INSUMO",
        "UNID_FAT": "UNID_PROD_ESTRUTURA",
        "QTD_ATENDIDA_SOLVER": "QTD_FILHO_SOLVER_TOTAL",
        "NEC_PCS": "NEC_TOTAL_FILHO_SOLVER_PCS",
        "NEC_DIRETA_PCS": "NEC_DIRETA_FILHO_PCS"
    })[["MES_REF", "UNID_PROD_ESTRUTURA", "COD_INSUMO", "QTD_FILHO_SOLVER_TOTAL", "NEC_TOTAL_FILHO_SOLVER_PCS", "NEC_DIRETA_FILHO_PCS"]]
    atend_filho = atend_filho.rename(columns={"UNID_PROD_ESTRUTURA": "UNID_PROD"})
    ajuste = ajuste.merge(atend_filho, on=["MES_REF", "UNID_PROD", "COD_INSUMO"], how="left")
    ajuste = ajuste.merge(roteiro_filho, on=["MES_REF", "UNID_PROD", "COD_INSUMO"], how="left")
    for c in ["QTD_FILHO_SOLVER_TOTAL", "NEC_TOTAL_FILHO_SOLVER_PCS", "NEC_DIRETA_FILHO_PCS"]:
        ajuste[c] = ajuste[c].fillna(0)
    ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"] = ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"].fillna(False).astype(bool)

    # Atendimento total do filho no solver e parcela da produção do filho atribuída à necessidade estrutural.
    ajuste["FATOR_ATEND_FILHO_TOTAL"] = np.where(
        ajuste["NEC_TOTAL_FILHO_SOLVER_PCS"] > TOL,
        (ajuste["QTD_FILHO_SOLVER_TOTAL"] / ajuste["NEC_TOTAL_FILHO_SOLVER_PCS"]).clip(0, 1),
        1.0
    )
    ajuste["PESO_ESTRUTURA_NO_FILHO"] = np.where(
        ajuste["NEC_TOTAL_FILHO_SOLVER_PCS"] > TOL,
        (ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] / ajuste["NEC_TOTAL_FILHO_SOLVER_PCS"]).clip(0, 1),
        0.0
    )
    ajuste["QTD_FILHO_SOLVER_ESTRUTURA_PCS"] = ajuste["QTD_FILHO_SOLVER_TOTAL"] * ajuste["PESO_ESTRUTURA_NO_FILHO"]

    # Regra: componente sustenta o pai com estoque já consumido na explosão + produção do filho alocada à estrutura.
    ajuste["QTD_FILHO_DISPONIVEL_ESTRUTURA_PCS"] = ajuste["ESTOQUE_CONSUMIDO_PCS"].fillna(0) + ajuste["QTD_FILHO_SOLVER_ESTRUTURA_PCS"].fillna(0)
    ajuste["NEC_FILHO_ATENDIDA_ESTRUTURA_PCS"] = np.minimum(ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"], ajuste["QTD_FILHO_DISPONIVEL_ESTRUTURA_PCS"])

    ajuste["QTD_PAI_SUSTENTADA_PELO_FILHO"] = np.where(
        ajuste["QTD_UTIL_PCS"] > TOL,
        ajuste["QTD_FILHO_DISPONIVEL_ESTRUTURA_PCS"] / ajuste["QTD_UTIL_PCS"],
        ajuste["QTD_PAI_SOLVER"]
    )
    ajuste["QTD_PAI_FINAL_LINHA"] = ajuste["QTD_PAI_SOLVER"]
    mask_limita = (ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] > TOL) & (ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"])
    ajuste.loc[mask_limita, "QTD_PAI_FINAL_LINHA"] = np.minimum(
        ajuste.loc[mask_limita, "QTD_PAI_SOLVER"],
        ajuste.loc[mask_limita, "QTD_PAI_SUSTENTADA_PELO_FILHO"]
    )
    ajuste["FATOR_FILHO_USADO_PARA_LIMITAR"] = np.where(
        ajuste["QTD_PAI_SOLVER"] > TOL,
        (ajuste["QTD_PAI_FINAL_LINHA"] / ajuste["QTD_PAI_SOLVER"]).clip(0, 1),
        1.0
    )
    ajuste.loc[ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] <= TOL, "FATOR_FILHO_USADO_PARA_LIMITAR"] = 1.0
    ajuste.loc[~ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"], "FATOR_FILHO_USADO_PARA_LIMITAR"] = 1.0

    ch_origem = ["MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM"]
    resumo = ajuste.groupby(ch_origem, as_index=False).agg(
        NEC_RAIZ_PCS=("NEC_RAIZ_PCS", "max"),
        QTD_PAI_SOLVER=("QTD_PAI_SOLVER", "max"),
        FATOR_ATEND_PAI_SOLVER=("FATOR_ATEND_PAI_SOLVER", "max"),
        QTD_PAI_FINAL_VIAVEL_EST=("QTD_PAI_FINAL_LINHA", "min"),
        QTD_COMPONENTES_ANALISADOS=("COD_INSUMO", "nunique")
    )
    resumo["QTD_PAI_FINAL_VIAVEL_EST"] = np.minimum(resumo["QTD_PAI_FINAL_VIAVEL_EST"], resumo["QTD_PAI_SOLVER"])
    resumo["QTD_REDUCAO_POS_PAI_FILHO"] = (resumo["QTD_PAI_SOLVER"] - resumo["QTD_PAI_FINAL_VIAVEL_EST"]).clip(lower=0)
    resumo["FATOR_FINAL_VIAVEL_ORIGEM"] = np.where(resumo["NEC_RAIZ_PCS"] > TOL, (resumo["QTD_PAI_FINAL_VIAVEL_EST"] / resumo["NEC_RAIZ_PCS"]).clip(0, 1), 0)
    resumo["FATOR_AJUSTE_SOBRE_SOLVER"] = np.where(resumo["QTD_PAI_SOLVER"] > TOL, (resumo["QTD_PAI_FINAL_VIAVEL_EST"] / resumo["QTD_PAI_SOLVER"]).clip(0, 1), 0)

    limitantes = ajuste[(ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] > TOL) & (ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"]) & (ajuste["QTD_PAI_FINAL_LINHA"] < ajuste["QTD_PAI_SOLVER"] - TOL)]
    qtd_limitantes = limitantes.groupby(ch_origem)["COD_INSUMO"].nunique().reset_index(name="QTD_COMPONENTES_LIMITANTES") if not limitantes.empty else pd.DataFrame(columns=ch_origem + ["QTD_COMPONENTES_LIMITANTES"])
    sem_roteiro = ajuste[(ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] > TOL) & (~ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"])]
    qtd_sem_roteiro = sem_roteiro.groupby(ch_origem)["COD_INSUMO"].nunique().reset_index(name="QTD_COMPONENTES_SEM_ROTEIRO") if not sem_roteiro.empty else pd.DataFrame(columns=ch_origem + ["QTD_COMPONENTES_SEM_ROTEIRO"])
    resumo = resumo.merge(qtd_limitantes, on=ch_origem, how="left").merge(qtd_sem_roteiro, on=ch_origem, how="left")
    resumo["QTD_COMPONENTES_LIMITANTES"] = resumo["QTD_COMPONENTES_LIMITANTES"].fillna(0).astype(int)
    resumo["QTD_COMPONENTES_SEM_ROTEIRO"] = pd.to_numeric(resumo["QTD_COMPONENTES_SEM_ROTEIRO"], errors="coerce").fillna(0).astype("int64")
    resumo["MOTIVO_AJUSTE_PAI_FILHO"] = np.select(
        [resumo["QTD_PAI_SOLVER"] <= TOL, resumo["FATOR_ATEND_PAI_SOLVER"] < 1 - TOL, resumo["QTD_COMPONENTES_LIMITANTES"] > 0, resumo["QTD_COMPONENTES_SEM_ROTEIRO"] > 0],
        ["PAI_NAO_PRODUZIDO_PELO_SOLVER", "PAI_CORTADO_PELO_SOLVER", "FILHO_COM_ROTEIRO_LIMITOU_PAI", "FILHO_SEM_ROTEIRO_EXPOSTO_NAO_LIMITA"],
        default="SEM_AJUSTE_POS_SOLVER"
    )

    ajuste = ajuste.merge(resumo[ch_origem + ["QTD_PAI_FINAL_VIAVEL_EST", "FATOR_FINAL_VIAVEL_ORIGEM", "MOTIVO_AJUSTE_PAI_FILHO"]], on=ch_origem, how="left")
    ajuste.loc[ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] <= TOL, "MOTIVO_AJUSTE_PAI_FILHO"] = "SEM_NECESSIDADE_FILHO_POS_CORTE_PAI"

    ajuste = ajuste[[c for c in cols_ajuste if c in ajuste.columns]].copy()
    resumo = resumo[[c for c in cols_resumo if c in resumo.columns]].copy()

    fatores = resumo.rename(columns={"ID_PROD_UNID_FAT_ORIGEM": "ID_PROD_UNID_FAT"})[["MES_REF", "ID_PROD_UNID_FAT", "FATOR_AJUSTE_SOBRE_SOLVER", "FATOR_FINAL_VIAVEL_ORIGEM"]]
    if plano_producao is not None and not plano_producao.empty:
        cap_lib = plano_producao.merge(fatores, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
        cap_lib["FATOR_AJUSTE_SOBRE_SOLVER"] = cap_lib["FATOR_AJUSTE_SOBRE_SOLVER"].fillna(1.0)
        cap_lib["FATOR_FINAL_VIAVEL_ORIGEM"] = cap_lib["FATOR_FINAL_VIAVEL_ORIGEM"].fillna(1.0)
        for c in ["QTD_PRODUZIR_SOLVER", "PCS_HORA", "HR_PRODUZIR_SOLVER"]:
            cap_lib[c] = to_num(cap_lib[c]) if c in cap_lib.columns else 0.0
        cap_lib["QTD_PRODUZIR_FINAL_VIAVEL_EST"] = cap_lib["QTD_PRODUZIR_SOLVER"] * cap_lib["FATOR_AJUSTE_SOBRE_SOLVER"]
        cap_lib["QTD_REDUCAO_POS_PAI_FILHO"] = (cap_lib["QTD_PRODUZIR_SOLVER"] - cap_lib["QTD_PRODUZIR_FINAL_VIAVEL_EST"]).clip(lower=0)
        cap_lib["HR_LIBERADA_EST"] = np.where(cap_lib["PCS_HORA"] > TOL, cap_lib["QTD_REDUCAO_POS_PAI_FILHO"] / cap_lib["PCS_HORA"], 0)
        cap_lib = cap_lib[cap_lib["HR_LIBERADA_EST"] > TOL].copy()
        cap_lib = cap_lib[[c for c in ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "QTD_PRODUZIR_SOLVER", "QTD_PRODUZIR_FINAL_VIAVEL_EST", "QTD_REDUCAO_POS_PAI_FILHO", "HR_PRODUZIR_SOLVER", "HR_LIBERADA_EST", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER"] if c in cap_lib.columns]].copy()
    else:
        cap_lib = pd.DataFrame(columns=["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "ALOC_REC", "COD_FER_UNID", "HR_LIBERADA_EST"])

    return ajuste, resumo, cap_lib

_finalizar_etapa_v58("PREPARACAO_BASES")
print("       OK - Bases preparadas")


# 03. Solver, regras operacionais e encadeamento


In [ ]:
print("[02/04] Processando Solver, regras operacionais e encadeamento...")
_iniciar_etapa_v58("SOLVER_ENCADEAMENTO")

# ============================================================
# 02-06E. MOTOR MENSAL REUTILIZAVEL
# ============================================================

def _executar_motor_periodo_v58(bd_periodo_entrada, aplicar_regras_industriais=True):

    # ============================================================
    # 02. LEITURA DA BASE + EXPLOSÃO INICIAL LÍQUIDA DA ESTRUTURA
    # ============================================================

    bd = bd_periodo_entrada.copy()

    bd["COD_PROD"] = normalizar_codigo_serie(bd["COD_PROD"])
    bd["UNID_PROD"] = bd["UNID_PROD"].astype("string").fillna("").str.strip()
    if "UNID_FAT" in bd.columns:
        bd["UNID_FAT"] = bd["UNID_FAT"].astype("string").fillna("").str.strip()

    bd = calcular_nec_pcs_se_necessario(bd, aplicar_regras_industriais=aplicar_regras_industriais)
    bd_mes = preparar_base(bd)

    estrutura_produto = ler_estrutura_produto(ESTRUTURA_FILE)
    estoque_componentes = calcular_estoque_componentes_pos_demanda_direta(bd_mes)

    # rotina: a estrutura é localizada pela unidade produtiva do pai.
    # A UNID_FAT da origem é preservada separadamente para devolver a necessidade do componente ao plano.
    bd_mes["UNID_ESTRUTURA"] = bd_mes["UNID_PROD"].astype("string").fillna("").str.strip()
    bd_mes["UNID_FAT_ORIGEM"] = bd_mes["UNID_FAT"].where(
        bd_mes["UNID_FAT"].astype("string").fillna("").str.strip().ne(""),
        bd_mes["UNID_PROD"]
    ).astype("string").fillna("").str.strip()

    # rotina: a raiz da explosão também deve respeitar estoque/cobertura antes de gerar filhos.
    # Regra vigente: a explosão usa somente NEC_ESTRUTURA_PAI_PCS líquida.
    # Se o pai raiz tem demanda bruta, mas estoque/trânsito cobre 100%, ele fica no plano com produção zero
    # e MOTIVO_STATUS = DEMANDA_DIRETA_COBERTA_POR_ESTOQUE, mas não gera necessidade produtiva para componentes.
    mask_pai_coberto_sem_explodir = (
        (to_num(bd_mes["NEC_ESTRUTURA_PAI_PCS"]) <= TOL)
        & (to_num(bd_mes.get("NEC_DIRETA_BRUTA_PCS", 0)) > TOL)
    )

    demanda_raiz_estrutura = (
        bd_mes.sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
        .groupby(["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD"], as_index=False)
        .agg(
            UNID_PROD=("UNID_ESTRUTURA", "first"),
            UNID_FAT_ORIGEM=("UNID_FAT_ORIGEM", "first"),
            NEC_PCS=("NEC_ESTRUTURA_PAI_PCS", "max")
        )
    )
    demanda_raiz_estrutura = demanda_raiz_estrutura[demanda_raiz_estrutura["NEC_PCS"] > TOL].copy()

    mapa_componentes_retorno = (
        bd_mes.sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
        .groupby(["COD_PROD", "UNID_FAT"], as_index=False)
        .agg(UNID_PROD=("UNID_PROD", "first"))
    )

    estrutura_explodida, problemas_estrutura = explodir_estrutura_multinivel_com_estoque(
        estrutura_produto,
        demanda_raiz_estrutura,
        estoque_componentes,
        mapa_componentes_retorno,
        max_nivel=30
    )

    bd_mes, nec_consolidada_solver = consolidar_necessidade_com_estrutura(bd_mes, estrutura_explodida)

    # Validação direcionada rotina operacional do caso crítico 0544 -> 0544NE.
    try:
        _retorno_0544 = estrutura_explodida[(estrutura_explodida["COD_PROD_ORIGEM"].astype(str) == "0544") & (estrutura_explodida["COD_INSUMO"].astype(str) == "0544NE")].copy()
        _retorno_0544ne = nec_consolidada_solver[nec_consolidada_solver["COD_PROD"].astype(str).eq("0544NE")].copy()
    except Exception as e:
        print("VALIDACAO_AUDITORIA_0544NE_FALHOU:", e)

    nec_explodida_componentes = (
        estrutura_explodida.groupby(["MES_REF", "UNID_PROD", "COD_PROD_ORIGEM", "COD_INSUMO"], as_index=False)
        .agg(
            NEC_EXPLODIDA_BRUTA_PCS=("NEC_GERADA_BRUTA_PCS", "sum"),
            ESTOQUE_CONSUMIDO_EXPLOSAO_PCS=("ESTOQUE_CONSUMIDO_PCS", "sum"),
            NEC_EXPLODIDA_PCS=("NEC_GERADA_LIQ_PCS", "sum"),
            MENOR_NIVEL=("NIVEL", "min")
        )
        if not estrutura_explodida.empty else pd.DataFrame(columns=["MES_REF", "UNID_PROD", "COD_PROD_ORIGEM", "COD_INSUMO", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "MENOR_NIVEL"])
    )

    validacao_explosao = pd.DataFrame({
        "METRICA": [
            "QTD_LINHAS_ESTRUTURA_ORIGINAL",
            "QTD_PRODUTOS_RAIZ_COM_DEMANDA",
            "QTD_LINHAS_ESTRUTURA_EXPLODIDA",
            "QTD_COMPONENTES_COM_NEC_EXPLODIDA_LIQUIDA",
            "MAIOR_NIVEL_EXPLODIDO",
            "QTD_PROBLEMAS_ESTRUTURA",
            "QTD_CICLOS_ESTRUTURA",
            "NEC_DIRETA_TOTAL_PCS",
            "NEC_EXPLODIDA_BRUTA_TOTAL_PCS",
            "ESTOQUE_CONSUMIDO_EXPLOSAO_TOTAL_PCS",
            "NEC_EXPLODIDA_LIQUIDA_TOTAL_PCS",
            "NEC_CONSOLIDADA_TOTAL_PCS",
            "QTD_ITENS_COM_DEMANDA_EXPLODIDA_LIQUIDA",
            "QTD_ITENS_EXPLODIDOS_SEM_ROTEIRO_NO_SOLVER"
        ],
        "VALOR": [
            len(estrutura_produto),
            len(demanda_raiz_estrutura),
            len(estrutura_explodida),
            nec_explodida_componentes.loc[nec_explodida_componentes["NEC_EXPLODIDA_PCS"] > TOL, "COD_INSUMO"].nunique() if not nec_explodida_componentes.empty else 0,
            estrutura_explodida["NIVEL"].max() if not estrutura_explodida.empty else 0,
            len(problemas_estrutura),
            (problemas_estrutura["MOTIVO"].eq("CICLO_ESTRUTURA").sum() if not problemas_estrutura.empty else 0),
            nec_consolidada_solver["NEC_DIRETA_PCS"].sum() if not nec_consolidada_solver.empty else 0,
            nec_consolidada_solver["NEC_EXPLODIDA_BRUTA_PCS"].sum() if not nec_consolidada_solver.empty else 0,
            nec_consolidada_solver["ESTOQUE_CONSUMIDO_EXPLOSAO_PCS"].sum() if not nec_consolidada_solver.empty else 0,
            nec_consolidada_solver["NEC_EXPLODIDA_PCS"].sum() if not nec_consolidada_solver.empty else 0,
            nec_consolidada_solver["NEC_PCS_CONSOLIDADA"].sum() if not nec_consolidada_solver.empty else 0,
            (nec_consolidada_solver["NEC_EXPLODIDA_PCS"] > TOL).sum() if not nec_consolidada_solver.empty else 0,
            ((nec_consolidada_solver["NEC_EXPLODIDA_PCS"] > TOL) & (~nec_consolidada_solver["TEM_ROTEIRO_NO_SOLVER"])).sum() if not nec_consolidada_solver.empty else 0
        ]
    })



    # ============================================================
    # 03. MONTAGEM DA DEMANDA E DAS ALTERNATIVAS
    # ============================================================

    meta_cols = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "NEC_PCS_ORIGINAL", "NEC_ESTRUTURA_PAI_PCS", "NEC_DIRETA_BRUTA_PCS", "NEC_DIRETA_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_COMP_PCS", "NEC_PCS_CONSOLIDADA", "LOTE_MIN", "QTD_EMB", "CURVA_ABC", "CURVA_123"]

    bd_demanda_solver = (
        bd_mes.sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
        .groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False)
        .agg({**{c: "first" for c in meta_cols if c not in ["MES_REF", "ID_PROD_UNID_FAT"] and c in bd_mes.columns}, "NEC_PCS": "max"})
    )
    bd_demanda_solver = bd_demanda_solver[bd_demanda_solver["NEC_PCS"] > TOL].copy()

    alt_cols = ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "ID_RECURSO", "ID_FERRAMENTA", "LOTE_MIN", "QTD_EMB", "CURVA_ABC", "CURVA_123"]
    bd_alternativas_item = bd_mes[[c for c in alt_cols if c in bd_mes.columns]].copy()
    bd_alternativas_item = bd_alternativas_item[bd_alternativas_item["ID_PROD_UNID_FAT"].isin(bd_demanda_solver["ID_PROD_UNID_FAT"])].copy()
    bd_alternativas_item = bd_alternativas_item.drop_duplicates([c for c in ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"] if c in bd_alternativas_item.columns]).reset_index(drop=True)

    bd_alternativas_item["ALTERNATIVA_VALIDA"] = (
        bd_alternativas_item["ALOC_REC"].notna() & bd_alternativas_item["COD_FER_UNID"].notna()
        & (bd_alternativas_item["PCS_HORA"] > TOL) & (bd_alternativas_item["HOR_REC"] > TOL)
        & (bd_alternativas_item["HOR_FER"] > TOL) & (bd_alternativas_item["HOR_CAP"] > TOL)
    )
    bd_alternativas_item["MOTIVO_INVALIDA"] = bd_alternativas_item.apply(classificar_motivo_invalida, axis=1)
    bd_alternativas_validas = bd_alternativas_item[bd_alternativas_item["ALTERNATIVA_VALIDA"]].copy()

    bd_cap_recursos = bd_alternativas_item.dropna(subset=["ALOC_REC"]).groupby("ALOC_REC", as_index=False).agg(HOR_REC=("HOR_REC", "max"))
    bd_cap_ferramentas = bd_alternativas_item.dropna(subset=["COD_FER_UNID"]).groupby("COD_FER_UNID", as_index=False).agg(HOR_FER=("HOR_FER", "max"))



    # ============================================================
    # 04. SOLVER - RATEIO PROPORCIONAL COM DUPLA RESTRIÇÃO
    # ============================================================

    def executar_rateio_proporcional_dupla_restricao(bd_demanda, bd_alternativas_validas, bd_cap_recursos, bd_cap_ferramentas):
        alternativas = bd_alternativas_validas.copy()
        cap_restante_rec = bd_cap_recursos.set_index("ALOC_REC")["HOR_REC"].to_dict()
        cap_restante_fer = bd_cap_ferramentas.set_index("COD_FER_UNID")["HOR_FER"].to_dict()
        pendente = bd_demanda.set_index("ID_PROD_UNID_FAT")["NEC_PCS"].to_dict()
        producao, rastro = [], []

        niveis_prioridade = list(alternativas[["PRIOR_MATPAR", "PRIOR_ROT"]].drop_duplicates().sort_values(["PRIOR_MATPAR", "PRIOR_ROT"]).itertuples(index=False, name=None))
        for rodada, (prior_matpar, prior_rot) in enumerate(niveis_prioridade, start=1):
            nivel = alternativas[alternativas["PRIOR_MATPAR"].eq(prior_matpar) & alternativas["PRIOR_ROT"].eq(prior_rot)].copy()
            nivel["RODADA"] = rodada
            nivel["QTD_PENDENTE_ANTES"] = nivel["ID_PROD_UNID_FAT"].map(pendente).fillna(0)
            nivel = nivel[nivel["QTD_PENDENTE_ANTES"] > TOL].copy()
            if nivel.empty:
                continue

            qtd_alt_mesmo_nivel = nivel.groupby("ID_PROD_UNID_FAT")["ALOC_REC"].transform("count")
            nivel["QTD_ALTERNATIVAS_MESMO_NIVEL"] = qtd_alt_mesmo_nivel
            nivel["QTD_SOLICITADA"] = nivel["QTD_PENDENTE_ANTES"] / qtd_alt_mesmo_nivel
            nivel["HR_SOLICITADA"] = nivel["QTD_SOLICITADA"] / nivel["PCS_HORA"]

            nivel["CAP_RESTANTE_ALOC_REC_ANTES"] = nivel["ALOC_REC"].map(cap_restante_rec).fillna(0)
            nivel["CAP_RESTANTE_COD_FER_ANTES"] = nivel["COD_FER_UNID"].map(cap_restante_fer).fillna(0)

            demanda_rec = nivel.groupby("ALOC_REC")["HR_SOLICITADA"].transform("sum")
            demanda_fer = nivel.groupby("COD_FER_UNID")["HR_SOLICITADA"].transform("sum")
            nivel["HR_DEMANDA_TOTAL_ALOC_REC"] = demanda_rec
            nivel["HR_DEMANDA_TOTAL_COD_FER"] = demanda_fer
            nivel["FATOR_RATEIO_ALOC_REC"] = np.where(demanda_rec > TOL, np.minimum(1.0, nivel["CAP_RESTANTE_ALOC_REC_ANTES"] / demanda_rec), 0.0)
            nivel["FATOR_RATEIO_COD_FER"] = np.where(demanda_fer > TOL, np.minimum(1.0, nivel["CAP_RESTANTE_COD_FER_ANTES"] / demanda_fer), 0.0)
            nivel["FATOR_RATEIO"] = nivel[["FATOR_RATEIO_ALOC_REC", "FATOR_RATEIO_COD_FER"]].min(axis=1).clip(lower=0, upper=1)
            nivel["CAP_DISPONIVEL_LINHA"] = nivel[["CAP_RESTANTE_ALOC_REC_ANTES", "CAP_RESTANTE_COD_FER_ANTES"]].min(axis=1)

            cond_rec = nivel["FATOR_RATEIO_ALOC_REC"] < nivel["FATOR_RATEIO_COD_FER"] - TOL
            cond_fer = nivel["FATOR_RATEIO_COD_FER"] < nivel["FATOR_RATEIO_ALOC_REC"] - TOL
            cond_ambos = (~cond_rec) & (~cond_fer) & (nivel["FATOR_RATEIO"] < 1 - TOL)
            nivel["MOTIVO_LIMITANTE"] = np.select([cond_rec, cond_fer, cond_ambos], ["ALOC_REC", "COD_FER_UNID", "ALOC_REC_E_COD_FER"], default="SEM_LIMITACAO")

            # O rateio proporcional permanece contínuo nesta etapa e calcula somente a distribuição factível
            # pela dupla restrição de recurso + ferramenta. LIMIT_PCS/LOTE_MIN/QTD_EMB entram somente
            # no fechamento industrial posterior. Depois desse fechamento, a capacidade nao volta a cortar o plano.
            nivel["HR_PRODUZIR_SOLVER"] = nivel["HR_SOLICITADA"] * nivel["FATOR_RATEIO"]
            nivel["QTD_PRODUZIR_SOLVER"] = (nivel["HR_PRODUZIR_SOLVER"] * nivel["PCS_HORA"]).clip(lower=0.0)
            nivel["QTD_PENDENTE_DEPOIS"] = (nivel["QTD_PENDENTE_ANTES"] - nivel.groupby("ID_PROD_UNID_FAT")["QTD_PRODUZIR_SOLVER"].transform("sum")).clip(lower=0)

            nivel["STATUS_DECISAO"] = np.select(
                [nivel["CAP_RESTANTE_ALOC_REC_ANTES"] <= TOL, nivel["CAP_RESTANTE_COD_FER_ANTES"] <= TOL, nivel["HR_PRODUZIR_SOLVER"] <= TOL, nivel["FATOR_RATEIO"] >= 1 - TOL, nivel["FATOR_RATEIO"] < 1 - TOL],
                ["NAO_ALOCADO_CAP_RECURSO_ZERO", "NAO_ALOCADO_CAP_FERRAMENTA_ZERO", "NAO_ALOCADO_RATEIO_ZERO", "ALOCADO_TOTAL_RODADA", "ALOCADO_PARCIAL_RATEIO"],
                default="VERIFICAR"
            )
            nivel["MOTIVO_CORTE"] = np.where(nivel["FATOR_RATEIO"] >= 1 - TOL, "SEM_CORTE", "CORTE_POR_" + nivel["MOTIVO_LIMITANTE"].astype(str))

            for aloc_rec, consumo in nivel.groupby("ALOC_REC")["HR_PRODUZIR_SOLVER"].sum().items():
                cap_restante_rec[aloc_rec] = max(0.0, cap_restante_rec.get(aloc_rec, 0.0) - float(consumo))
            for cod_fer, consumo in nivel.groupby("COD_FER_UNID")["HR_PRODUZIR_SOLVER"].sum().items():
                cap_restante_fer[cod_fer] = max(0.0, cap_restante_fer.get(cod_fer, 0.0) - float(consumo))
            for item, qtd_prod in nivel.groupby("ID_PROD_UNID_FAT")["QTD_PRODUZIR_SOLVER"].sum().items():
                pendente[item] = max(0.0, pendente.get(item, 0.0) - float(qtd_prod))

            producao.append(nivel[nivel["QTD_PRODUZIR_SOLVER"] > TOL].copy())
            rastro.append(nivel.copy())

        bd_producao = pd.concat(producao, ignore_index=True) if producao else pd.DataFrame()
        bd_rastro = pd.concat(rastro, ignore_index=True) if rastro else pd.DataFrame()
        return bd_producao, bd_rastro, pendente, cap_restante_rec, cap_restante_fer

    bd_plano_producao, rastro_rateio, pendente_final, cap_restante_rec_final, cap_restante_fer_final = executar_rateio_proporcional_dupla_restricao(bd_demanda_solver, bd_alternativas_validas, bd_cap_recursos, bd_cap_ferramentas)


    # ============================================================
    # 04A. PONTE DE DIAGNOSTICO V66 -> MATRIZ DE CORTES
    # ============================================================
    # Esta etapa NAO corta NEC, NAO redistribui capacidade e NAO altera o plano.
    # Ela apenas traduz o estado final do Solver em um diagnostico seguro para a
    # futura Matriz: o que ficou pendente, quais rotas ainda teriam capacidade
    # simultanea de recurso + ferramenta e quais restricoes aparecem como
    # candidatas a corte quando nao existe mais alternativa livre.

    def gerar_diagnostico_matriz_ponte(
        bd_demanda,
        bd_alternativas,
        bd_rastro,
        pendente,
        cap_restante_rec,
        cap_restante_fer,
    ):
        cols_resumo = [
            'MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD',
            'NEC_PCS','QTD_PENDENTE_FINAL','RODADA_FINAL_ITEM','PRIOR_MATPAR_FINAL','PRIOR_ROT_FINAL',
            'QTD_ROTAS_VALIDAS','QTD_ROTAS_ULTIMA_RODADA','QTD_ROTAS_COM_CAPACIDADE_FINAL',
            'QTD_FALTA_ULTIMA_RODADA','DIF_PENDENCIA_VS_ULTIMA_RODADA',
            'STATUS_DIAGNOSTICO','TIPO_RESTRICAO_CANDIDATA','RESTRICAO_CANDIDATA',
            'CORTE_HR_CANDIDATO','QTD_CORTE_PCS_CANDIDATO','RESTRICOES_BLOQUEANTES'
        ]
        cols_rotas = [
            'MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD',
            'PRIOR_MATPAR','PRIOR_ROT','CLASS_ROT','ALOC_REC','COD_FER_UNID','PCS_HORA',
            'QTD_PENDENTE_FINAL','CAP_RESTANTE_ALOC_REC_FINAL','CAP_RESTANTE_COD_FER_FINAL',
            'HR_DISPONIVEL_FINAL','QTD_POSSIVEL_FINAL','ALTERNATIVA_COM_CAPACIDADE_FINAL',
            'RODADA_FINAL_ITEM','FOI_ULTIMA_RODADA','QTD_SOLICITADA_ULTIMA_RODADA',
            'QTD_PRODUZIR_ULTIMA_RODADA','QTD_FALTA_ROTA','HR_FALTA_ROTA',
            'MOTIVO_LIMITANTE_ULTIMA_RODADA','BLOQUEIOS_FINAIS_ROTA'
        ]

        pend_df = pd.DataFrame([
            {'ID_PROD_UNID_FAT': k, 'QTD_PENDENTE_FINAL': float(v)}
            for k, v in pendente.items() if float(v) > TOL
        ])
        if pend_df.empty:
            return pd.DataFrame(columns=cols_resumo), pd.DataFrame(columns=cols_rotas)

        meta_cols = [
            c for c in ['MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD','NEC_PCS']
            if c in bd_demanda.columns
        ]
        meta = bd_demanda[meta_cols].drop_duplicates('ID_PROD_UNID_FAT')
        pend_df = pend_df.merge(meta, on='ID_PROD_UNID_FAT', how='left')

        alts = bd_alternativas[bd_alternativas['ID_PROD_UNID_FAT'].isin(pend_df['ID_PROD_UNID_FAT'])].copy()
        if alts.empty:
            resumo = pend_df.copy()
            resumo['RODADA_FINAL_ITEM'] = np.nan
            resumo['PRIOR_MATPAR_FINAL'] = np.nan
            resumo['PRIOR_ROT_FINAL'] = np.nan
            resumo['QTD_ROTAS_VALIDAS'] = 0
            resumo['QTD_ROTAS_ULTIMA_RODADA'] = 0
            resumo['QTD_ROTAS_COM_CAPACIDADE_FINAL'] = 0
            resumo['QTD_FALTA_ULTIMA_RODADA'] = resumo['QTD_PENDENTE_FINAL']
            resumo['DIF_PENDENCIA_VS_ULTIMA_RODADA'] = 0.0
            resumo['STATUS_DIAGNOSTICO'] = 'SEM_ALTERNATIVA_VALIDA'
            resumo['TIPO_RESTRICAO_CANDIDATA'] = ''
            resumo['RESTRICAO_CANDIDATA'] = ''
            resumo['CORTE_HR_CANDIDATO'] = np.nan
            resumo['QTD_CORTE_PCS_CANDIDATO'] = np.nan
            resumo['RESTRICOES_BLOQUEANTES'] = ''
            return resumo[[c for c in cols_resumo if c in resumo.columns]], pd.DataFrame(columns=cols_rotas)

        for c in ['PCS_HORA','PRIOR_MATPAR','PRIOR_ROT']:
            if c in alts.columns:
                alts[c] = to_num(alts[c])
        alts['QTD_PENDENTE_FINAL'] = alts['ID_PROD_UNID_FAT'].map(
            pend_df.set_index('ID_PROD_UNID_FAT')['QTD_PENDENTE_FINAL']
        ).fillna(0.0)
        alts['CAP_RESTANTE_ALOC_REC_FINAL'] = alts['ALOC_REC'].map(cap_restante_rec).fillna(0.0)
        alts['CAP_RESTANTE_COD_FER_FINAL'] = alts['COD_FER_UNID'].map(cap_restante_fer).fillna(0.0)
        alts['HR_DISPONIVEL_FINAL'] = alts[['CAP_RESTANTE_ALOC_REC_FINAL','CAP_RESTANTE_COD_FER_FINAL']].min(axis=1).clip(lower=0.0)
        alts['QTD_POSSIVEL_FINAL'] = alts['HR_DISPONIVEL_FINAL'] * alts['PCS_HORA']
        alts['ALTERNATIVA_COM_CAPACIDADE_FINAL'] = alts['QTD_POSSIVEL_FINAL'] > TOL

        def _bloqueios_rota(row):
            bloqueios = []
            if float(row['CAP_RESTANTE_ALOC_REC_FINAL']) <= TOL:
                bloqueios.append('REC|' + str(row['ALOC_REC']))
            if float(row['CAP_RESTANTE_COD_FER_FINAL']) <= TOL:
                bloqueios.append('FER|' + str(row['COD_FER_UNID']))
            return bloqueios

        alts['_BLOQUEIOS_SET'] = alts.apply(_bloqueios_rota, axis=1)
        alts['BLOQUEIOS_FINAIS_ROTA'] = alts['_BLOQUEIOS_SET'].apply(lambda x: ';'.join(x))

        rastro = bd_rastro.copy()
        if rastro.empty:
            rodada_final = pd.Series(dtype=float)
            ultimo = pd.DataFrame()
        else:
            rastro = rastro[rastro['ID_PROD_UNID_FAT'].isin(pend_df['ID_PROD_UNID_FAT'])].copy()
            rodada_final = rastro.groupby('ID_PROD_UNID_FAT')['RODADA'].max()
            rastro['_RODADA_FINAL_ITEM'] = rastro['ID_PROD_UNID_FAT'].map(rodada_final)
            ultimo = rastro[rastro['RODADA'].eq(rastro['_RODADA_FINAL_ITEM'])].copy()
            ultimo['QTD_FALTA_ROTA'] = (
                to_num(ultimo['QTD_SOLICITADA']) - to_num(ultimo['QTD_PRODUZIR_SOLVER'])
            ).clip(lower=0.0)
            ultimo['HR_FALTA_ROTA'] = np.divide(
                ultimo['QTD_FALTA_ROTA'],
                to_num(ultimo['PCS_HORA']),
                out=np.zeros(len(ultimo), dtype=float),
                where=to_num(ultimo['PCS_HORA']).to_numpy() > TOL,
            )

        chave_alt = [
            c for c in ['ID_PROD_UNID_FAT','PRIOR_MATPAR','PRIOR_ROT','ALOC_REC','COD_FER_UNID']
            if c in alts.columns
        ]
        if not ultimo.empty:
            ult_cols = chave_alt + [
                'RODADA','QTD_SOLICITADA','QTD_PRODUZIR_SOLVER','QTD_FALTA_ROTA','HR_FALTA_ROTA','MOTIVO_LIMITANTE'
            ]
            ult = ultimo[[c for c in ult_cols if c in ultimo.columns]].copy()
            ren = {
                'RODADA':'RODADA_FINAL_ITEM',
                'QTD_SOLICITADA':'QTD_SOLICITADA_ULTIMA_RODADA',
                'QTD_PRODUZIR_SOLVER':'QTD_PRODUZIR_ULTIMA_RODADA',
                'MOTIVO_LIMITANTE':'MOTIVO_LIMITANTE_ULTIMA_RODADA',
            }
            ult = ult.rename(columns=ren)
            alts = alts.merge(ult, on=chave_alt, how='left')
        else:
            alts['RODADA_FINAL_ITEM'] = np.nan
            alts['QTD_SOLICITADA_ULTIMA_RODADA'] = 0.0
            alts['QTD_PRODUZIR_ULTIMA_RODADA'] = 0.0
            alts['QTD_FALTA_ROTA'] = 0.0
            alts['HR_FALTA_ROTA'] = 0.0
            alts['MOTIVO_LIMITANTE_ULTIMA_RODADA'] = ''

        for c in ['QTD_SOLICITADA_ULTIMA_RODADA','QTD_PRODUZIR_ULTIMA_RODADA','QTD_FALTA_ROTA','HR_FALTA_ROTA']:
            if c not in alts.columns:
                alts[c] = 0.0
            alts[c] = to_num(alts[c])
        alts['FOI_ULTIMA_RODADA'] = alts['RODADA_FINAL_ITEM'].notna()

        # Reaplica metadados do item de forma controlada quando a tabela de alternativas nao os possui.
        meta_map = pend_df.set_index('ID_PROD_UNID_FAT')
        for c in ['MES_REF','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD']:
            if c not in alts.columns and c in meta_map.columns:
                alts[c] = alts['ID_PROD_UNID_FAT'].map(meta_map[c])

        resumos = []
        for item, grp in alts.groupby('ID_PROD_UNID_FAT', sort=False):
            p = pend_df.loc[pend_df['ID_PROD_UNID_FAT'].eq(item)].iloc[0]
            grp_ult = grp[grp['FOI_ULTIMA_RODADA'] & (grp['QTD_FALTA_ROTA'] > TOL)].copy()
            qtd_falta_ult = float(grp_ult['QTD_FALTA_ROTA'].sum()) if not grp_ult.empty else 0.0
            qtd_pend = float(p['QTD_PENDENTE_FINAL'])
            rotas_cap = int(grp['ALTERNATIVA_COM_CAPACIDADE_FINAL'].sum())

            # Uma restricao so pode ser chamada de candidata objetiva se ela bloqueia
            # TODAS as rotas que ainda carregam falta na ultima rodada.
            comum = None
            if not grp_ult.empty:
                sets = [set(x) for x in grp_ult['_BLOQUEIOS_SET']]
                comum = set.intersection(*sets) if sets else set()
            else:
                comum = set()

            todos_bloqueios = sorted(set().union(*[set(x) for x in grp['_BLOQUEIOS_SET']])) if len(grp) else []

            tipo = ''
            restr = ''
            corte_hr = np.nan
            qtd_corte = np.nan

            if rotas_cap > 0:
                status = 'REOTIMIZAR_ANTES_DE_CORTAR'
            elif grp_ult.empty:
                status = 'SEM_RASTRO_FINAL_PARA_PENDENCIA'
            elif len(comum) == 1:
                restr_key = next(iter(comum))
                tipo, restr = restr_key.split('|', 1)
                corte_hr = float(grp_ult['HR_FALTA_ROTA'].sum())
                qtd_corte = qtd_pend
                status = 'CORTE_CANDIDATO_OBJETIVO'
            elif len(comum) == 2 and any(x.startswith('REC|') for x in comum) and any(x.startswith('FER|') for x in comum):
                # Recurso e ferramenta podem estar simultaneamente esgotados na mesma decisao.
                # O volume de demanda deve ser cortado UMA unica vez; esse mesmo corte libera
                # horas nas duas restricoes e nao autoriza dois cortes independentes.
                tipo = 'REC_E_FER'
                restr = ';'.join(sorted(comum))
                corte_hr = float(grp_ult['HR_FALTA_ROTA'].sum())
                qtd_corte = qtd_pend
                status = 'CORTE_CANDIDATO_BLOQUEIO_DUPLO'
            else:
                status = 'AMBIGUO_MULTIPLAS_RESTRICOES'

            rodada_item = grp['RODADA_FINAL_ITEM'].dropna()
            rodada_item = float(rodada_item.iloc[0]) if not rodada_item.empty else np.nan
            pm = grp_ult['PRIOR_MATPAR'].iloc[0] if not grp_ult.empty and 'PRIOR_MATPAR' in grp_ult.columns else np.nan
            pr = grp_ult['PRIOR_ROT'].iloc[0] if not grp_ult.empty and 'PRIOR_ROT' in grp_ult.columns else np.nan

            row = {c: p[c] for c in ['MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD','NEC_PCS'] if c in p.index}
            row.update({
                'QTD_PENDENTE_FINAL': qtd_pend,
                'RODADA_FINAL_ITEM': rodada_item,
                'PRIOR_MATPAR_FINAL': pm,
                'PRIOR_ROT_FINAL': pr,
                'QTD_ROTAS_VALIDAS': int(len(grp)),
                'QTD_ROTAS_ULTIMA_RODADA': int(grp['FOI_ULTIMA_RODADA'].sum()),
                'QTD_ROTAS_COM_CAPACIDADE_FINAL': rotas_cap,
                'QTD_FALTA_ULTIMA_RODADA': qtd_falta_ult,
                'DIF_PENDENCIA_VS_ULTIMA_RODADA': qtd_pend - qtd_falta_ult,
                'STATUS_DIAGNOSTICO': status,
                'TIPO_RESTRICAO_CANDIDATA': tipo,
                'RESTRICAO_CANDIDATA': restr,
                'CORTE_HR_CANDIDATO': corte_hr,
                'QTD_CORTE_PCS_CANDIDATO': qtd_corte,
                'RESTRICOES_BLOQUEANTES': ';'.join(todos_bloqueios),
            })
            resumos.append(row)

        resumo = pd.DataFrame(resumos)
        rotas = alts[[c for c in cols_rotas if c in alts.columns]].copy()
        resumo = resumo[[c for c in cols_resumo if c in resumo.columns]].copy()
        return resumo, rotas

    diagnostico_matriz, diagnostico_matriz_rotas = gerar_diagnostico_matriz_ponte(
        bd_demanda_solver,
        bd_alternativas_validas,
        rastro_rateio,
        pendente_final,
        cap_restante_rec_final,
        cap_restante_fer_final,
    )

    # Validacao interna: a ponte e somente observadora; nao pode alterar o plano.
    if not diagnostico_matriz.empty:
        _dif_diag = diagnostico_matriz['DIF_PENDENCIA_VS_ULTIMA_RODADA'].abs()
        diagnostico_matriz['VALIDA_PENDENCIA_ULTIMA_RODADA'] = _dif_diag <= max(TOL * 10, 1e-6)


    # ============================================================
    # 05. SAÍDAS DE NEGÓCIO E RASTREABILIDADE
    # ============================================================


    cols_plano = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "NEC_PCS_ORIGINAL", "NEC_ESTRUTURA_PAI_PCS", "NEC_DIRETA_BRUTA_PCS", "NEC_DIRETA_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_COMP_PCS", "NEC_PCS_CONSOLIDADA", "LINHA_PROD", "FAMILIA_PROD", "MO", "RODADA", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "QTD_PENDENTE_ANTES", "QTD_SOLICITADA", "HR_SOLICITADA", "CAP_RESTANTE_ALOC_REC_ANTES", "CAP_RESTANTE_COD_FER_ANTES", "CAP_DISPONIVEL_LINHA", "HR_DEMANDA_TOTAL_ALOC_REC", "HR_DEMANDA_TOTAL_COD_FER", "FATOR_RATEIO_ALOC_REC", "FATOR_RATEIO_COD_FER", "FATOR_RATEIO", "MOTIVO_LIMITANTE", "QTD_PRODUZIR_SOLVER", "PESO_PROD_KG", "VOL_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER", "QTD_PENDENTE_DEPOIS", "MOTIVO_CORTE", "STATUS_DECISAO", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER", "QTD_PRODUZIR_FINAL_VIAVEL_EST", "QTD_REDUCAO_POS_PAI_FILHO", "HR_LIBERADA_EST"]
    campos_produto_saida = ["COD_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG"]
    campos_produto_saida = [c for c in campos_produto_saida if c in bd_mes.columns]
    if campos_produto_saida and "COD_PROD" in campos_produto_saida:
        cadastro_produto_saida = bd_mes[campos_produto_saida].drop_duplicates("COD_PROD")
        for c in ["LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG"]:
            if c in bd_plano_producao.columns:
                bd_plano_producao = bd_plano_producao.drop(columns=[c])
        bd_plano_producao = bd_plano_producao.merge(cadastro_produto_saida, on="COD_PROD", how="left")

    bd_plano_producao["PESO_PROD_KG"] = to_num(bd_plano_producao["PESO_PROD_KG"]) if "PESO_PROD_KG" in bd_plano_producao.columns else 0
    bd_plano_producao["VOL_PRODUZIR_SOLVER"] = bd_plano_producao["QTD_PRODUZIR_SOLVER"] * bd_plano_producao["PESO_PROD_KG"] if "QTD_PRODUZIR_SOLVER" in bd_plano_producao.columns else 0
    plano_producao = bd_plano_producao[[c for c in cols_plano if c in bd_plano_producao.columns]].copy()

    atendido_item = plano_producao.groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False).agg(QTD_ATENDIDA_SOLVER=("QTD_PRODUZIR_SOLVER", "sum")) if not plano_producao.empty else pd.DataFrame(columns=["MES_REF", "ID_PROD_UNID_FAT", "QTD_ATENDIDA_SOLVER"])
    nao_atend_item = bd_demanda_solver.merge(atendido_item, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
    nao_atend_item["QTD_ATENDIDA_SOLVER"] = nao_atend_item["QTD_ATENDIDA_SOLVER"].fillna(0)
    nao_atend_item["QTD_NAO_ATEND_SOLVER"] = (nao_atend_item["NEC_PCS"] - nao_atend_item["QTD_ATENDIDA_SOLVER"]).clip(lower=0)
    nao_atend_item["PERC_ATENDIDO_SOLVER"] = np.where(nao_atend_item["NEC_PCS"] > 0, nao_atend_item["QTD_ATENDIDA_SOLVER"] / nao_atend_item["NEC_PCS"], 0)
    nao_atend_item["PERC_NAO_ATENDIDO_SOLVER", "QTD_FINAL_VIAVEL_PAI_FILHO", "QTD_AJUSTE_PAI_FILHO", "FATOR_FINAL_VIAVEL_PAI_FILHO", "MOTIVO_AJUSTE_PAI_FILHO"] = 1 - nao_atend_item["PERC_ATENDIDO_SOLVER"]
    cols_nao_atend = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "NEC_PCS_ORIGINAL", "NEC_ESTRUTURA_PAI_PCS", "NEC_DIRETA_BRUTA_PCS", "NEC_DIRETA_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_COMP_PCS", "NEC_PCS_CONSOLIDADA", "NEC_PCS", "QTD_ATENDIDA_SOLVER", "QTD_NAO_ATEND_SOLVER", "PERC_ATENDIDO_SOLVER", "PERC_NAO_ATENDIDO_SOLVER"]
    nao_atend_item = nao_atend_item[[c for c in cols_nao_atend if c in nao_atend_item.columns]].copy()

    # Inicializa os campos pós-solver. O fechamento estrutural final é executado depois das auditorias base.
    ajuste_pai_filho = pd.DataFrame()
    resumo_pai_filho = pd.DataFrame()
    capacidade_liberada_pf = pd.DataFrame()
    nao_atend_item["QTD_FINAL_VIAVEL_PAI_FILHO"] = nao_atend_item["QTD_ATENDIDA_SOLVER"]
    nao_atend_item["QTD_AJUSTE_PAI_FILHO"] = 0.0
    nao_atend_item["FATOR_FINAL_VIAVEL_PAI_FILHO"] = 1.0
    nao_atend_item["MOTIVO_AJUSTE_PAI_FILHO"] = "SEM_AJUSTE_POS_SOLVER"
    plano_producao["FATOR_FINAL_VIAVEL_ORIGEM"] = 1.0
    plano_producao["FATOR_AJUSTE_SOBRE_SOLVER"] = 1.0
    plano_producao["QTD_PRODUZIR_FINAL_VIAVEL_EST"] = plano_producao["QTD_PRODUZIR_SOLVER"]
    plano_producao["QTD_REDUCAO_POS_PAI_FILHO"] = 0.0
    plano_producao["HR_LIBERADA_EST"] = 0.0

    # Ajuste operacional — blindagem contra duplicidade na saída do plano.
    # A origem do problema era o merge do plano com fatores pai/filho quando o resumo pai/filho tinha mais de uma linha para a mesma chave MES_REF + ID_PROD_UNID_FAT.
    # O solver/rastro não estava estourado; a duplicidade ocorria na montagem/exportação do plano e contaminava os resumos.
    CHAVE_PLANO_UNICA = [c for c in ['MES_REF','ID_PROD_UNID_FAT','COD_PROD','RODADA','PRIOR_MATPAR','PRIOR_ROT','ALOC_REC','COD_FER_UNID','QTD_PRODUZIR_SOLVER','HR_PRODUZIR_SOLVER'] if c in plano_producao.columns]
    QTD_LINHAS_PLANO_ANTES_DEDUP = len(plano_producao)
    plano_producao = plano_producao.drop_duplicates(subset=CHAVE_PLANO_UNICA, keep='first').copy()
    QTD_LINHAS_PLANO_DEPOIS_DEDUP = len(plano_producao)

    prod_recurso = plano_producao.groupby("ALOC_REC", as_index=False).agg(HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"), QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"), QTD_ITENS=("ID_PROD_UNID_FAT", "nunique")) if not plano_producao.empty else pd.DataFrame(columns=["ALOC_REC", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS"])
    resumo_recursos = bd_cap_recursos.merge(prod_recurso, on="ALOC_REC", how="left").fillna({"HR_PRODUZIR_SOLVER": 0, "QTD_PRODUZIR_SOLVER": 0, "QTD_ITENS": 0})
    resumo_recursos["OCUPACAO_RECURSO_PCT"] = np.where(resumo_recursos["HOR_REC"] > 0, resumo_recursos["HR_PRODUZIR_SOLVER"] / resumo_recursos["HOR_REC"], 0)
    resumo_recursos["HR_RECURSO_OCIOSA"] = resumo_recursos["HOR_REC"] - resumo_recursos["HR_PRODUZIR_SOLVER"]
    resumo_recursos["ESTOURO_HR_RECURSO"] = np.where(resumo_recursos["HR_PRODUZIR_SOLVER"] > resumo_recursos["HOR_REC"] + TOL, resumo_recursos["HR_PRODUZIR_SOLVER"] - resumo_recursos["HOR_REC"], 0)
    resumo_recursos = resumo_recursos[["ALOC_REC", "HOR_REC", "HR_PRODUZIR_SOLVER", "OCUPACAO_RECURSO_PCT", "HR_RECURSO_OCIOSA", "ESTOURO_HR_RECURSO", "QTD_PRODUZIR_SOLVER", "QTD_ITENS"]].sort_values(["OCUPACAO_RECURSO_PCT", "HR_PRODUZIR_SOLVER"], ascending=[False, False])

    prod_ferramenta = plano_producao.groupby("COD_FER_UNID", as_index=False).agg(HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"), QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"), QTD_ITENS=("ID_PROD_UNID_FAT", "nunique"), QTD_RECURSOS=("ALOC_REC", "nunique")) if not plano_producao.empty else pd.DataFrame(columns=["COD_FER_UNID", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS", "QTD_RECURSOS"])
    resumo_ferramentas = bd_cap_ferramentas.merge(prod_ferramenta, on="COD_FER_UNID", how="left").fillna({"HR_PRODUZIR_SOLVER": 0, "QTD_PRODUZIR_SOLVER": 0, "QTD_ITENS": 0, "QTD_RECURSOS": 0})
    resumo_ferramentas["OCUPACAO_FER_PCT"] = np.where(resumo_ferramentas["HOR_FER"] > 0, resumo_ferramentas["HR_PRODUZIR_SOLVER"] / resumo_ferramentas["HOR_FER"], 0)
    resumo_ferramentas["HR_FER_OCIOSA"] = resumo_ferramentas["HOR_FER"] - resumo_ferramentas["HR_PRODUZIR_SOLVER"]
    resumo_ferramentas["ESTOURO_HR_FER"] = np.where(resumo_ferramentas["HR_PRODUZIR_SOLVER"] > resumo_ferramentas["HOR_FER"] + TOL, resumo_ferramentas["HR_PRODUZIR_SOLVER"] - resumo_ferramentas["HOR_FER"], 0)
    resumo_ferramentas = resumo_ferramentas[["COD_FER_UNID", "HOR_FER", "HR_PRODUZIR_SOLVER", "OCUPACAO_FER_PCT", "HR_FER_OCIOSA", "ESTOURO_HR_FER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS", "QTD_RECURSOS"]].sort_values(["OCUPACAO_FER_PCT", "HR_PRODUZIR_SOLVER"], ascending=[False, False])

    cols_auditoria = ["MES_REF", "ALOC_REC", "COD_FER_UNID", "HOR_REC", "HOR_FER", "HOR_CAP"]
    auditoria_capacidade = bd_alternativas_item[[c for c in cols_auditoria if c in bd_alternativas_item.columns]].drop_duplicates().sort_values(["ALOC_REC", "COD_FER_UNID"]).copy()

    cols_alt = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "ALTERNATIVA_VALIDA", "MOTIVO_INVALIDA"]
    alternativas_item = bd_alternativas_item[[c for c in cols_alt if c in bd_alternativas_item.columns]].sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC"]).copy()

    cols_rastro = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "RODADA", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "QTD_PENDENTE_ANTES", "QTD_SOLICITADA", "HR_SOLICITADA", "HR_DEMANDA_TOTAL_ALOC_REC", "HR_DEMANDA_TOTAL_COD_FER", "CAP_RESTANTE_ALOC_REC_ANTES", "CAP_RESTANTE_COD_FER_ANTES", "CAP_DISPONIVEL_LINHA", "FATOR_RATEIO_ALOC_REC", "FATOR_RATEIO_COD_FER", "FATOR_RATEIO", "MOTIVO_LIMITANTE", "QTD_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER", "QTD_PENDENTE_DEPOIS", "MOTIVO_CORTE", "STATUS_DECISAO"]
    rastro_rateio = rastro_rateio[[c for c in cols_rastro if c in rastro_rateio.columns]].copy() if not rastro_rateio.empty else pd.DataFrame(columns=cols_rastro)

    qtd_alt_audit = bd_alternativas_item.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_ALTERNATIVAS_AUDITADAS=("ALOC_REC", "count"))
    qtd_alt_valid = bd_alternativas_validas.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_ALTERNATIVAS_VALIDAS=("ALOC_REC", "count"))
    qtd_alt_usadas = plano_producao.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_ALTERNATIVAS_USADAS=("ALOC_REC", "count"), QTD_ALOC_REC_USADOS=("ALOC_REC", "nunique"), QTD_COD_FER_USADOS=("COD_FER_UNID", "nunique")) if not plano_producao.empty else pd.DataFrame(columns=["ID_PROD_UNID_FAT", "QTD_ALTERNATIVAS_USADAS", "QTD_ALOC_REC_USADOS", "QTD_COD_FER_USADOS"])
    qtd_rodadas = rastro_rateio.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_RODADAS_TENTADAS=("RODADA", "nunique"), QTD_ALOC_REC_TENTADOS=("ALOC_REC", "nunique"), QTD_COD_FER_TENTADOS=("COD_FER_UNID", "nunique")) if not rastro_rateio.empty else pd.DataFrame(columns=["ID_PROD_UNID_FAT", "QTD_RODADAS_TENTADAS", "QTD_ALOC_REC_TENTADOS", "QTD_COD_FER_TENTADOS"])

    diagnostico_item = nao_atend_item.copy()
    for df_merge in [qtd_alt_audit, qtd_alt_valid, qtd_alt_usadas, qtd_rodadas]:
        diagnostico_item = diagnostico_item.merge(df_merge, on="ID_PROD_UNID_FAT", how="left")
    for c in ["QTD_ALTERNATIVAS_AUDITADAS", "QTD_ALTERNATIVAS_VALIDAS", "QTD_ALTERNATIVAS_USADAS", "QTD_ALOC_REC_USADOS", "QTD_COD_FER_USADOS", "QTD_RODADAS_TENTADAS", "QTD_ALOC_REC_TENTADOS", "QTD_COD_FER_TENTADOS"]:
        if c in diagnostico_item.columns:
            diagnostico_item[c] = diagnostico_item[c].fillna(0).astype(int)
    diagnostico_item["MOTIVO_SALDO_FINAL"] = np.select([diagnostico_item["QTD_NAO_ATEND_SOLVER"] <= TOL, diagnostico_item.get("QTD_ALTERNATIVAS_AUDITADAS", 0).eq(0), diagnostico_item.get("QTD_ALTERNATIVAS_VALIDAS", 0).eq(0), diagnostico_item.get("QTD_ALTERNATIVAS_USADAS", 0).eq(0)], ["ATENDIDO_TOTAL", "SEM_ALTERNATIVA_CADASTRADA", "SEM_ALTERNATIVA_VALIDA", "SEM_ALOCACAO_COM_ALTERNATIVA_VALIDA"], default="SALDO_APOS_RATEIO_E_LIMITES_DE_CAPACIDADE")

    uso_alt = plano_producao.groupby(["ID_PROD_UNID_FAT", "ALOC_REC", "COD_FER_UNID", "PRIOR_MATPAR", "PRIOR_ROT"], as_index=False).agg(HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"), QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum")) if not plano_producao.empty else pd.DataFrame(columns=["ID_PROD_UNID_FAT", "ALOC_REC", "COD_FER_UNID", "PRIOR_MATPAR", "PRIOR_ROT", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER"])
    diagnostico_alternativas = alternativas_item.merge(uso_alt, on=["ID_PROD_UNID_FAT", "ALOC_REC", "COD_FER_UNID", "PRIOR_MATPAR", "PRIOR_ROT"], how="left")
    diagnostico_alternativas["HR_PRODUZIR_SOLVER"] = diagnostico_alternativas["HR_PRODUZIR_SOLVER"].fillna(0)
    diagnostico_alternativas["QTD_PRODUZIR_SOLVER"] = diagnostico_alternativas["QTD_PRODUZIR_SOLVER"].fillna(0)
    diagnostico_alternativas["ALTERNATIVA_USADA"] = diagnostico_alternativas["HR_PRODUZIR_SOLVER"] > TOL


    # ============================================================
    # 06. REALOCAÇÃO DA CAPACIDADE LIBERADA - rotina operacional
    # ============================================================

    # Para realocar em outro item, o modelo valida a capacidade da máquina e também a capacidade do molde/ferramenta destino (COD_FER_UNID).
    # A realocação não fica mais presa ao mesmo molde que gerou a liberação.

    # ---------- rotina operacional realocacao ----------
    def calcular_realocacao_realoc(nao_atend_item, alternativas_item, plano_producao, capacidade_liberada_pf):
        cols_log = [
            'RODADA_REALOC','MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD',
            'PRIOR_MATPAR','PRIOR_ROT','CLASS_ROT','ALOC_REC','COD_FER_UNID','PCS_HORA',
            'QTD_GAP_ANTES_REALOC','HR_MAQ_LIBERADA_DISP_ANTES','HR_FER_DESTINO_DISP_ANTES','HR_DISPONIVEL_REALOC',
            'QTD_POSSIVEL_REALOCAR','QTD_REALOCADA','HR_CONSUMIDA_REALOC','HR_MAQ_LIBERADA_SALDO_DEPOIS',
            'HR_FER_DESTINO_SALDO_DEPOIS','QTD_GAP_DEPOIS_REALOC','STATUS_REALOCACAO','MOTIVO_NAO_REALOCACAO','MOTIVO_AJUSTE_PAI_FILHO'
        ]
        if capacidade_liberada_pf.empty:
            return pd.DataFrame(columns=cols_log), pd.DataFrame(), pd.DataFrame()
        na = nao_atend_item.copy(); alts = alternativas_item.copy(); plano = plano_producao.copy(); cap = capacidade_liberada_pf.copy()
        for df in [na, alts, plano, cap]:
            if 'MES_REF' in df.columns:
                df['MES_REF'] = pd.to_datetime(df['MES_REF'], errors='coerce')
        for c in ['NEC_PCS','QTD_FINAL_VIAVEL_PAI_FILHO','QTD_AJUSTE_PAI_FILHO','QTD_ATENDIDA_SOLVER']:
            if c not in na.columns: na[c] = 0.0
            na[c] = to_num(na[c])
        na['QTD_GAP_APOS_PAI_FILHO'] = (na['NEC_PCS'] - na['QTD_FINAL_VIAVEL_PAI_FILHO']).clip(lower=0)
        # Mantem bloqueio: itens cortados por pai/filho nao recebem realocacao nesta rodada, para nao produzir sem componente.
        na['BLOQUEADO_AJUSTE_PAI_FILHO'] = na['QTD_AJUSTE_PAI_FILHO'] > TOL

        if 'ALTERNATIVA_VALIDA' in alts.columns:
            alts['_ALT_VALIDA'] = is_valid_alt(alts['ALTERNATIVA_VALIDA'])
        else:
            alts['_ALT_VALIDA'] = True
        for c in ['PCS_HORA','PRIOR_MATPAR','PRIOR_ROT','HOR_FER','HOR_REC']:
            if c in alts.columns: alts[c] = to_num(alts[c])
        for c in ['HR_PRODUZIR_SOLVER','HOR_FER','HOR_REC']:
            if c in plano.columns: plano[c] = to_num(plano[c])
        for c in ['HR_LIBERADA_EST']:
            cap[c] = to_num(cap[c])

        # Capacidade liberada volta para a maquina, nao fica presa ao molde original.
        pool_maq = cap.groupby(['MES_REF','ALOC_REC'], as_index=False).agg(
            HR_MAQ_LIBERADA_TOTAL=('HR_LIBERADA_EST','sum'),
            QTD_ORIGENS_LIBERARAM=('ID_PROD_UNID_FAT','nunique'),
            QTD_LINHAS_LIBERARAM=('ID_PROD_UNID_FAT','count')
        )

        # Capacidade disponivel por molde destino depois do ajuste pai/filho.
        # HOR_FER total mensal por ferramenta.
        fer_cap_1 = alts.groupby(['MES_REF','COD_FER_UNID'], as_index=False).agg(HOR_FER=('HOR_FER','max')) if 'HOR_FER' in alts.columns else pd.DataFrame(columns=['MES_REF','COD_FER_UNID','HOR_FER'])
        fer_cap_2 = plano.groupby(['MES_REF','COD_FER_UNID'], as_index=False).agg(HOR_FER_PLANO=('HOR_FER','max')) if 'HOR_FER' in plano.columns else pd.DataFrame(columns=['MES_REF','COD_FER_UNID','HOR_FER_PLANO'])
        fer_cap = fer_cap_1.merge(fer_cap_2, on=['MES_REF','COD_FER_UNID'], how='outer')
        fer_cap['HOR_FER'] = fer_cap.get('HOR_FER', 0).fillna(0)
        if 'HOR_FER_PLANO' in fer_cap.columns:
            fer_cap['HOR_FER'] = np.maximum(fer_cap['HOR_FER'], fer_cap['HOR_FER_PLANO'].fillna(0))
        fer_cap = fer_cap[['MES_REF','COD_FER_UNID','HOR_FER']]

        fer_usado_solver = plano.groupby(['MES_REF','COD_FER_UNID'], as_index=False).agg(HR_USADA_SOLVER=('HR_PRODUZIR_SOLVER','sum')) if 'HR_PRODUZIR_SOLVER' in plano.columns else pd.DataFrame(columns=['MES_REF','COD_FER_UNID','HR_USADA_SOLVER'])
        fer_liberada = cap.groupby(['MES_REF','COD_FER_UNID'], as_index=False).agg(HR_LIBERADA_FER=('HR_LIBERADA_EST','sum'))
        fer_pool = fer_cap.merge(fer_usado_solver, on=['MES_REF','COD_FER_UNID'], how='left').merge(fer_liberada, on=['MES_REF','COD_FER_UNID'], how='left').fillna({'HR_USADA_SOLVER':0,'HR_LIBERADA_FER':0})
        fer_pool['HR_FER_USADA_POS_PAI_FILHO'] = (fer_pool['HR_USADA_SOLVER'] - fer_pool['HR_LIBERADA_FER']).clip(lower=0)
        fer_pool['HR_FER_DISPONIVEL_POS_PAI_FILHO'] = (fer_pool['HOR_FER'] - fer_pool['HR_FER_USADA_POS_PAI_FILHO']).clip(lower=0)

        # Candidatos: itens com gap na mesma maquina onde houve liberacao; ferramenta destino tem que ter capacidade.
        item_cols = ['MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD','NEC_PCS','QTD_FINAL_VIAVEL_PAI_FILHO','QTD_GAP_APOS_PAI_FILHO','BLOQUEADO_AJUSTE_PAI_FILHO','MOTIVO_AJUSTE_PAI_FILHO']
        cand = alts.merge(na[item_cols], on=['MES_REF','ID_PROD_UNID_FAT'], how='inner', suffixes=('', '_ITEM'))
        for c in ['COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD']:
            ci = c + '_ITEM'
            if ci in cand.columns:
                cand[c] = cand[c].fillna(cand[ci]) if c in cand.columns else cand[ci]
        cand = cand[(cand['_ALT_VALIDA']) & (cand['PCS_HORA'] > TOL) & (cand['QTD_GAP_APOS_PAI_FILHO'] > TOL) & (~cand['BLOQUEADO_AJUSTE_PAI_FILHO'])].copy()
        cand = cand.merge(pool_maq[['MES_REF','ALOC_REC','HR_MAQ_LIBERADA_TOTAL']], on=['MES_REF','ALOC_REC'], how='inner')
        cand = cand.merge(fer_pool[['MES_REF','COD_FER_UNID','HR_FER_DISPONIVEL_POS_PAI_FILHO']], on=['MES_REF','COD_FER_UNID'], how='left')
        cand['HR_FER_DISPONIVEL_POS_PAI_FILHO'] = cand['HR_FER_DISPONIVEL_POS_PAI_FILHO'].fillna(0)
        cand = cand[cand['HR_FER_DISPONIVEL_POS_PAI_FILHO'] > TOL].copy()

        gap_rem = {tuple(k): float(v) for k, v in na.set_index(['MES_REF','ID_PROD_UNID_FAT'])['QTD_GAP_APOS_PAI_FILHO'].items()}
        bloqueado = {tuple(k): bool(v) for k, v in na.set_index(['MES_REF','ID_PROD_UNID_FAT'])['BLOQUEADO_AJUSTE_PAI_FILHO'].items()}
        hr_maq_rem = {tuple(row[['MES_REF','ALOC_REC']]): float(row['HR_MAQ_LIBERADA_TOTAL']) for _, row in pool_maq.iterrows()}
        hr_fer_rem = {tuple(row[['MES_REF','COD_FER_UNID']]): float(row['HR_FER_DISPONIVEL_POS_PAI_FILHO']) for _, row in fer_pool.iterrows()}

        logs = []
        if not cand.empty:
            cand['_GAP_SORT'] = cand['QTD_GAP_APOS_PAI_FILHO']
            cand = cand.sort_values(['MES_REF','ALOC_REC','PRIOR_MATPAR','PRIOR_ROT','_GAP_SORT','ID_PROD_UNID_FAT'], ascending=[True, True, True, True, False, True])
            rodada = 1
            for _, row in cand.iterrows():
                item_key = (row['MES_REF'], row['ID_PROD_UNID_FAT'])
                maq_key = (row['MES_REF'], row['ALOC_REC'])
                fer_key = (row['MES_REF'], row['COD_FER_UNID'])
                gap_antes = gap_rem.get(item_key, 0.0)
                hr_maq_antes = hr_maq_rem.get(maq_key, 0.0)
                hr_fer_antes = hr_fer_rem.get(fer_key, 0.0)
                pcs_hora = float(row['PCS_HORA'])
                hr_disp = min(hr_maq_antes, hr_fer_antes)
                if gap_antes <= TOL:
                    status='NAO_REALOCADO_SEM_GAP_REMANESCENTE'; qtd_possivel=qtd_realoc=hr_consumida=0.0
                elif hr_maq_antes <= TOL:
                    status='NAO_REALOCADO_SEM_HORA_MAQUINA_LIBERADA'; qtd_possivel=qtd_realoc=hr_consumida=0.0
                elif hr_fer_antes <= TOL:
                    status='NAO_REALOCADO_SEM_HORA_MOLDE_DESTINO'; qtd_possivel=qtd_realoc=hr_consumida=0.0
                elif bloqueado.get(item_key, False):
                    status='NAO_REALOCADO_ITEM_BLOQUEADO_PAI_FILHO'; qtd_possivel=qtd_realoc=hr_consumida=0.0
                else:
                    qtd_possivel = hr_disp * pcs_hora
                    qtd_limite_realoc = min(gap_antes, qtd_possivel)

                    # A capacidade pode ser calculada continuamente, mas a decisao efetiva de realocacao
                    # deve manter a producao final em um bloco operacional valido de QTD_EMB.
                    # Em vez de simplesmente arredondar a parcela adicionada, alinhamos o TOTAL final
                    # do item; isso tambem protege contra pequenos residuos numericos vindos da etapa anterior.
                    qtd_base_item = float(max(0.0, row.get('QTD_FINAL_VIAVEL_PAI_FILHO', 0.0) or 0.0))
                    qtd_emb_item = float(max(0.0, row.get('QTD_EMB', 0.0) or 0.0))
                    if qtd_emb_item > TOL:
                        qtd_final_limite = qtd_base_item + qtd_limite_realoc
                        qtd_final_valida = float(np.floor((qtd_final_limite + TOL) / qtd_emb_item) * qtd_emb_item)
                        qtd_realoc = max(0.0, qtd_final_valida - qtd_base_item)
                        # Protecao contra qualquer efeito de ponto flutuante que pudesse ultrapassar
                        # o gap ou a capacidade efetivamente disponivel.
                        if qtd_realoc > qtd_limite_realoc + TOL:
                            qtd_realoc = max(0.0, float(np.floor((qtd_limite_realoc + TOL) / qtd_emb_item) * qtd_emb_item))
                    else:
                        qtd_realoc = qtd_limite_realoc

                    hr_consumida = qtd_realoc / pcs_hora if pcs_hora > TOL else 0.0
                    if qtd_realoc > TOL:
                        status = 'REALOCADO_TOTAL_GAP' if qtd_realoc >= gap_antes - TOL else 'REALOCADO_PARCIAL_POR_HORA_DISPONIVEL'
                        gap_rem[item_key] = max(0.0, gap_antes - qtd_realoc)
                        hr_maq_rem[maq_key] = max(0.0, hr_maq_antes - hr_consumida)
                        hr_fer_rem[fer_key] = max(0.0, hr_fer_antes - hr_consumida)
                    else:
                        status='NAO_REALOCADO_QTD_ZERO'
                logs.append({
                    'RODADA_REALOC': rodada,
                    'MES_REF': row['MES_REF'],
                    'ID_PROD_UNID_FAT': row['ID_PROD_UNID_FAT'],
                    'COD_PROD': row.get('COD_PROD'),
                    'DESC_PROD': row.get('DESC_PROD'),
                    'UNID_PROD': row.get('UNID_PROD'),
                    'UNID_FAT': row.get('UNID_FAT'),
                    'TIPO_PROD': row.get('TIPO_PROD'),
                    'PRIOR_MATPAR': row.get('PRIOR_MATPAR'),
                    'PRIOR_ROT': row.get('PRIOR_ROT'),
                    'CLASS_ROT': row.get('CLASS_ROT'),
                    'ALOC_REC': row['ALOC_REC'],
                    'COD_FER_UNID': row['COD_FER_UNID'],
                    'PCS_HORA': pcs_hora,
                    'QTD_GAP_ANTES_REALOC': gap_antes,
                    'HR_MAQ_LIBERADA_DISP_ANTES': hr_maq_antes,
                    'HR_FER_DESTINO_DISP_ANTES': hr_fer_antes,
                    'HR_DISPONIVEL_REALOC': hr_disp,
                    'QTD_POSSIVEL_REALOCAR': qtd_possivel,
                    'QTD_REALOCADA': qtd_realoc,
                    'HR_CONSUMIDA_REALOC': hr_consumida,
                    'HR_MAQ_LIBERADA_SALDO_DEPOIS': hr_maq_rem.get(maq_key, 0.0),
                    'HR_FER_DESTINO_SALDO_DEPOIS': hr_fer_rem.get(fer_key, 0.0),
                    'QTD_GAP_DEPOIS_REALOC': gap_rem.get(item_key, 0.0),
                    'STATUS_REALOCACAO': status,
                    'MOTIVO_NAO_REALOCACAO': '' if str(status).startswith('REALOCADO') else status,
                    'MOTIVO_AJUSTE_PAI_FILHO': row.get('MOTIVO_AJUSTE_PAI_FILHO')
                })
                rodada += 1
        # rotina: registrar sobras de horas liberadas que passaram pela realocação e não encontraram uso total.
        for (mes_ref, aloc_rec), hr_saldo in list(hr_maq_rem.items()):
            if hr_saldo > TOL:
                logs.append({
                    'RODADA_REALOC': rodada if 'rodada' in locals() else 1,
                    'MES_REF': mes_ref,
                    'ID_PROD_UNID_FAT': '',
                    'COD_PROD': '',
                    'DESC_PROD': '',
                    'UNID_PROD': '',
                    'UNID_FAT': '',
                    'TIPO_PROD': '',
                    'PRIOR_MATPAR': np.nan,
                    'PRIOR_ROT': np.nan,
                    'CLASS_ROT': '',
                    'ALOC_REC': aloc_rec,
                    'COD_FER_UNID': '',
                    'PCS_HORA': 0.0,
                    'QTD_GAP_ANTES_REALOC': 0.0,
                    'HR_MAQ_LIBERADA_DISP_ANTES': hr_saldo,
                    'HR_FER_DESTINO_DISP_ANTES': 0.0,
                    'HR_DISPONIVEL_REALOC': hr_saldo,
                    'QTD_POSSIVEL_REALOCAR': 0.0,
                    'QTD_REALOCADA': 0.0,
                    'HR_CONSUMIDA_REALOC': 0.0,
                    'HR_MAQ_LIBERADA_SALDO_DEPOIS': hr_saldo,
                    'HR_FER_DESTINO_SALDO_DEPOIS': 0.0,
                    'QTD_GAP_DEPOIS_REALOC': 0.0,
                    'STATUS_REALOCACAO': 'NAO_REALOCADO_HORA_LIBERADA_REMANESCENTE',
                    'MOTIVO_NAO_REALOCACAO': 'SEM_ITEM_COM_GAP_E_ALTERNATIVA_VALIDA_NA_MAQUINA_OU_SEM_FERRAMENTA_DESTINO',
                    'MOTIVO_AJUSTE_PAI_FILHO': ''
                })
                rodada = (rodada + 1) if 'rodada' in locals() else 2
        realoc = pd.DataFrame(logs, columns=cols_log)
        realoc_pos = realoc[realoc['QTD_REALOCADA'] > TOL].copy() if not realoc.empty else realoc.copy()

        realoc_item = realoc_pos.groupby(['MES_REF','ID_PROD_UNID_FAT'], as_index=False).agg(
            QTD_REALOCADA_CAP_LIBERADA=('QTD_REALOCADA','sum'),
            HR_CONSUMIDA_REALOC=('HR_CONSUMIDA_REALOC','sum')
        ) if not realoc_pos.empty else pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','QTD_REALOCADA_CAP_LIBERADA','HR_CONSUMIDA_REALOC'])
        resumo_item = na[['MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD','NEC_PCS','QTD_FINAL_VIAVEL_PAI_FILHO','QTD_GAP_APOS_PAI_FILHO','QTD_AJUSTE_PAI_FILHO','MOTIVO_AJUSTE_PAI_FILHO']].copy()
        resumo_item = resumo_item.merge(realoc_item, on=['MES_REF','ID_PROD_UNID_FAT'], how='left').fillna({'QTD_REALOCADA_CAP_LIBERADA':0,'HR_CONSUMIDA_REALOC':0})
        resumo_item['QTD_FINAL_APOS_REALOCACAO'] = resumo_item['QTD_FINAL_VIAVEL_PAI_FILHO'] + resumo_item['QTD_REALOCADA_CAP_LIBERADA']
        resumo_item['QTD_GAP_FINAL_APOS_REALOCACAO'] = (resumo_item['NEC_PCS'] - resumo_item['QTD_FINAL_APOS_REALOCACAO']).clip(lower=0)
        resumo_item['PERC_GAP_RECUPERADO'] = np.where(resumo_item['QTD_GAP_APOS_PAI_FILHO'] > TOL, resumo_item['QTD_REALOCADA_CAP_LIBERADA'] / resumo_item['QTD_GAP_APOS_PAI_FILHO'], 0)
        resumo_item = resumo_item[resumo_item['QTD_GAP_APOS_PAI_FILHO'] > TOL].sort_values(['QTD_REALOCADA_CAP_LIBERADA','QTD_GAP_APOS_PAI_FILHO'], ascending=[False, False])

        # Resumo interno para validacao (nao vai para Excel final como guia 17)
        resumo_maq = pool_maq.copy()
        uso_maq = realoc_pos.groupby(['MES_REF','ALOC_REC'], as_index=False).agg(
            HR_REAPROVEITADA=('HR_CONSUMIDA_REALOC','sum'),
            QTD_REALOCADA=('QTD_REALOCADA','sum'),
            QTD_ITENS_REALOCADOS=('ID_PROD_UNID_FAT','nunique')
        ) if not realoc_pos.empty else pd.DataFrame(columns=['MES_REF','ALOC_REC','HR_REAPROVEITADA','QTD_REALOCADA','QTD_ITENS_REALOCADOS'])
        resumo_maq = resumo_maq.merge(uso_maq, on=['MES_REF','ALOC_REC'], how='left').fillna({'HR_REAPROVEITADA':0,'QTD_REALOCADA':0,'QTD_ITENS_REALOCADOS':0})
        resumo_maq['HR_MAQ_NAO_REAPROVEITADA'] = (resumo_maq['HR_MAQ_LIBERADA_TOTAL'] - resumo_maq['HR_REAPROVEITADA']).clip(lower=0)
        resumo_maq['PERC_REAPROVEITAMENTO_HR'] = np.where(resumo_maq['HR_MAQ_LIBERADA_TOTAL'] > TOL, resumo_maq['HR_REAPROVEITADA'] / resumo_maq['HR_MAQ_LIBERADA_TOTAL'], 0)
        return realoc, resumo_maq, resumo_item

    realocacao_cap_liberada, resumo_realocacao_maquina, resumo_item_realocacao = calcular_realocacao_realoc(
        nao_atend_item,
        alternativas_item,
        plano_producao,
        capacidade_liberada_pf
    )



    # ============================================================
    # 06. VALIDAÇÕES
    # ============================================================

    nec_total = nao_atend_item["NEC_PCS"].sum()
    atend_total = nao_atend_item["QTD_ATENDIDA_SOLVER"].sum()
    nao_atend_total = nao_atend_item["QTD_NAO_ATEND_SOLVER"].sum()
    dif_fechamento = nec_total - atend_total - nao_atend_total
    qtd_rec_estouro = (resumo_recursos["ESTOURO_HR_RECURSO"] > TOL).sum()
    qtd_fer_estouro = (resumo_ferramentas["ESTOURO_HR_FER"] > TOL).sum()

    validacao = pd.DataFrame({
        "METRICA": ["NEC_PCS_TOTAL", "QTD_ATENDIDA_SOLVER_TOTAL", "QTD_NAO_ATEND_SOLVER_TOTAL", "DIF_FECHAMENTO_DEMANDA", "QTD_ITENS_DEMANDA", "QTD_ITENS_COM_PRODUCAO", "QTD_ITENS_SEM_PRODUCAO", "QTD_ITENS_COM_NAO_ATENDIMENTO", "QTD_RECURSOS", "QTD_RECURSOS_ESTOURO", "ESTOURO_HR_RECURSO_TOTAL", "HOR_REC_TOTAL", "HR_PRODUZIR_SOLVER_RECURSO_TOTAL", "OCUPACAO_GLOBAL_RECURSOS", "QTD_FERRAMENTAS", "QTD_FERRAMENTAS_ESTOURO", "ESTOURO_HR_FER_TOTAL", "HOR_FER_TOTAL", "HR_PRODUZIR_SOLVER_FER_TOTAL", "OCUPACAO_GLOBAL_FERRAMENTAS", "QTD_ALTERNATIVAS_AUDITADAS", "QTD_ALTERNATIVAS_VALIDAS_PRODUCAO", "QTD_ALTERNATIVAS_HOR_CAP_ZERO", "QTD_LINHAS_RASTRO_RATEIO", "QTD_ITENS_DIAGNOSTICO"],
        "VALOR": [nec_total, atend_total, nao_atend_total, dif_fechamento, len(nao_atend_item), (nao_atend_item["QTD_ATENDIDA_SOLVER"] > TOL).sum(), (nao_atend_item["QTD_ATENDIDA_SOLVER"] <= TOL).sum(), (nao_atend_item["QTD_NAO_ATEND_SOLVER"] > TOL).sum(), len(resumo_recursos), qtd_rec_estouro, resumo_recursos["ESTOURO_HR_RECURSO"].sum(), resumo_recursos["HOR_REC"].sum(), resumo_recursos["HR_PRODUZIR_SOLVER"].sum(), resumo_recursos["HR_PRODUZIR_SOLVER"].sum() / resumo_recursos["HOR_REC"].sum() if resumo_recursos["HOR_REC"].sum() > 0 else 0, len(resumo_ferramentas), qtd_fer_estouro, resumo_ferramentas["ESTOURO_HR_FER"].sum(), resumo_ferramentas["HOR_FER"].sum(), resumo_ferramentas["HR_PRODUZIR_SOLVER"].sum(), resumo_ferramentas["HR_PRODUZIR_SOLVER"].sum() / resumo_ferramentas["HOR_FER"].sum() if resumo_ferramentas["HOR_FER"].sum() > 0 else 0, len(bd_alternativas_item), len(bd_alternativas_validas), (bd_alternativas_item["HOR_CAP"] <= TOL).sum(), len(rastro_rateio), len(diagnostico_item)]
    })
    validacao_checks = pd.DataFrame({"METRICA": ["FECHAMENTO_DEMANDA_OK", "CAPACIDADE_RECURSOS_OK", "CAPACIDADE_FERRAMENTAS_OK"], "VALOR": [abs(dif_fechamento) <= 1e-4, qtd_rec_estouro == 0, qtd_fer_estouro == 0]})
    validacao = pd.concat([validacao, pd.DataFrame({"METRICA": ["---"], "VALOR": [""]}), validacao_checks], ignore_index=True)


    # Adiciona bloco de validação da explosão de estrutura dentro da validação geral
    validacao = pd.concat([
        validacao,
        pd.DataFrame({"METRICA": ["--- EXPLOSAO_ESTRUTURA ---"], "VALOR": [""]}),
        validacao_explosao
    ], ignore_index=True)


    # Validação específica rotina operacional - componente 0544NE recebendo demanda estrutural do pai 0544
    try:
        validacao_0544ne = bd_mes.loc[bd_mes["COD_PROD"].astype(str).eq("0544NE"), [c for c in ["MES_REF","ID_PROD_UNID_FAT","COD_PROD","UNID_PROD","UNID_FAT","NEC_DIRETA_PCS","NEC_COMP_PCS","NEC_PCS"] if c in bd_mes.columns]].drop_duplicates().copy()
    except Exception as e:
        print("Aviso validação 0544NE não executada:", e)



    # ============================================================
    # 06B. AUDITORIA COMPLETA DO PLANO - rotina operacional
    # ============================================================
    # AUDITORIA_PLANO_COMPLETO_STATUS_ZERO_PLANO
    # Regra: toda combinação MES_REF + ID_PROD_UNID_FAT existente no bd_LTP_NEC_SOLVER
    # deve aparecer no 03_PLANO_PRODUCAO.
    # Se QTD_PRODUZIR_SOLVER = 0, MOTIVO_STATUS deve explicar o motivo correto:
    # - DEMANDA_DIRETA_COBERTA_POR_ESTOQUE
    # - DEMANDA_ESTRUTURAL_COBERTA_POR_ESTOQUE
    # - ITEM_SEM_DEMANDA_NO_MES
    # - NAO_ALOCADO_FERRAMENTA_SEM_CAPACIDADE
    # - NAO_ALOCADO_RECURSO_SEM_CAPACIDADE
    # - NAO_ALOCADO_PCS_HORA_INVALIDO
    # - NAO_ALOCADO_SEM_ALTERNATIVA_VALIDA
    # - FALHA_RETORNO_DEMANDA_COMPONENTE_LTP

    try:
        def _key_mes_id(df):
            return pd.to_datetime(df["MES_REF"], errors="coerce").dt.strftime("%Y-%m-%d") + "|" + df["ID_PROD_UNID_FAT"].astype(str)

        def _calc_demanda_bruta_direta_ref(df):
            cols = ["LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS", "LTP_EST_SEG_PCS"]
            bruta = sum([to_num(df[c]) if c in df.columns else 0 for c in cols])
            if "MESMA_REG" in df.columns and "LTP_SALDO_PREV_PROX_MES_PCS" in df.columns:
                bruta = bruta + np.where(df["MESMA_REG"].astype(str).str.upper().eq("NAO"), to_num(df["LTP_SALDO_PREV_PROX_MES_PCS"]), 0)
            return pd.Series(bruta, index=df.index)

        def _calc_cobertura_ref(df):
            cols = ["LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS"]
            return sum([to_num(df[c]) if c in df.columns else 0 for c in cols])

        def _map_status_decisao_auditoria(motivo):
            if motivo in ["DEMANDA_DIRETA_COBERTA_POR_ESTOQUE", "DEMANDA_ESTRUTURAL_COBERTA_POR_ESTOQUE"]:
                return "NAO_PLANEJADO_DEMANDA_COBERTA_ESTOQUE"
            if motivo == "ITEM_SEM_DEMANDA_NO_MES":
                return "NAO_PLANEJADO_SEM_DEMANDA"
            if motivo == "NAO_ALOCADO_FERRAMENTA_SEM_CAPACIDADE":
                return "NAO_ALOCADO_CAP_FERRAMENTA_ZERO"
            if motivo == "NAO_ALOCADO_RECURSO_SEM_CAPACIDADE":
                return "NAO_ALOCADO_CAP_RECURSO_ZERO"
            if motivo == "NAO_ALOCADO_PCS_HORA_INVALIDO":
                return "NAO_ALOCADO_ALTERNATIVA_INVALIDA"
            if motivo == "NAO_ALOCADO_SEM_ALTERNATIVA_VALIDA":
                return "NAO_ALOCADO_SEM_ALTERNATIVA_VALIDA"
            if motivo == "FALHA_RETORNO_DEMANDA_COMPONENTE_LTP":
                return "NAO_PLANEJADO_ALERTA_COMPONENTE"
            return "NAO_PLANEJADO_SEM_DEMANDA"

        def _motivo_limitante_auditoria(motivo):
            if motivo == "NAO_ALOCADO_FERRAMENTA_SEM_CAPACIDADE":
                return "COD_FER_UNID"
            if motivo == "NAO_ALOCADO_RECURSO_SEM_CAPACIDADE":
                return "ALOC_REC"
            if motivo == "NAO_ALOCADO_PCS_HORA_INVALIDO":
                return "PCS_HORA_INVALIDO"
            if motivo in ["DEMANDA_DIRETA_COBERTA_POR_ESTOQUE", "DEMANDA_ESTRUTURAL_COBERTA_POR_ESTOQUE", "ITEM_SEM_DEMANDA_NO_MES"]:
                return "NAO_APLICAVEL_SEM_DEMANDA_LIQUIDA"
            return "NAO_APLICAVEL"

        def _motivo_corte_auditoria(motivo):
            if motivo == "NAO_ALOCADO_FERRAMENTA_SEM_CAPACIDADE":
                return "CORTE_POR_COD_FER_UNID"
            if motivo == "NAO_ALOCADO_RECURSO_SEM_CAPACIDADE":
                return "CORTE_POR_ALOC_REC"
            if motivo == "NAO_ALOCADO_PCS_HORA_INVALIDO":
                return "CORTE_POR_ALTERNATIVA_INVALIDA"
            if motivo in ["DEMANDA_DIRETA_COBERTA_POR_ESTOQUE", "DEMANDA_ESTRUTURAL_COBERTA_POR_ESTOQUE", "ITEM_SEM_DEMANDA_NO_MES"]:
                return "SEM_CORTE_SEM_DEMANDA_LIQUIDA"
            return "SEM_CORTE"

        base_ltp_auditoria = bd_mes.copy()
        base_ltp_auditoria["MES_REF"] = pd.to_datetime(base_ltp_auditoria["MES_REF"], errors="coerce")
        base_ltp_auditoria["DEMANDA_BRUTA_DIRETA_REF_PCS"] = _calc_demanda_bruta_direta_ref(base_ltp_auditoria)
        base_ltp_auditoria["COBERTURA_ESTOQUE_REF_PCS"] = _calc_cobertura_ref(base_ltp_auditoria)
        base_ltp_auditoria["NEC_DIRETA_CALC_REF_PCS"] = (base_ltp_auditoria["DEMANDA_BRUTA_DIRETA_REF_PCS"] - base_ltp_auditoria["COBERTURA_ESTOQUE_REF_PCS"]).clip(lower=0)

        base_cols = [c for c in ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "DEMANDA_BRUTA_DIRETA_REF_PCS", "COBERTURA_ESTOQUE_REF_PCS", "NEC_DIRETA_CALC_REF_PCS"] if c in base_ltp_auditoria.columns]
        base_status = base_ltp_auditoria.sort_values([c for c in ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"] if c in base_ltp_auditoria.columns])[base_cols].drop_duplicates(["MES_REF", "ID_PROD_UNID_FAT"], keep="first").copy()

        nec_cols = [c for c in ["MES_REF", "ID_PROD_UNID_FAT", "NEC_DIRETA_PCS", "NEC_COMP_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_PCS_CONSOLIDADA"] if c in nec_consolidada_solver.columns]
        if nec_cols:
            base_status = base_status.merge(nec_consolidada_solver[nec_cols].drop_duplicates(["MES_REF", "ID_PROD_UNID_FAT"]), on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")

        # Motivo de alternativa inválida para itens com necessidade líquida, mas sem produção.
        motivos_alt = pd.DataFrame(columns=["MES_REF", "ID_PROD_UNID_FAT", "MOTIVOS_INVALIDA"])
        if "MOTIVO_INVALIDA" in bd_alternativas_item.columns:
            motivos_alt = bd_alternativas_item.groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False).agg(MOTIVOS_INVALIDA=("MOTIVO_INVALIDA", lambda x: " | ".join(sorted(set(map(str, x))))))
            base_status = base_status.merge(motivos_alt, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
        else:
            base_status["MOTIVOS_INVALIDA"] = ""

        def _classificar_status_auditoria(row):
            nec_total = float(row.get("NEC_PCS_CONSOLIDADA", 0) or 0)
            direta = float(row.get("NEC_DIRETA_PCS", 0) or 0)
            comp = float(row.get("NEC_COMP_PCS", 0) or 0)
            exp_bruta = float(row.get("NEC_EXPLODIDA_BRUTA_PCS", 0) or 0)
            exp_liq = float(row.get("NEC_EXPLODIDA_PCS", 0) or 0)
            bruta_dir = float(row.get("DEMANDA_BRUTA_DIRETA_REF_PCS", 0) or 0)
            cobertura = float(row.get("COBERTURA_ESTOQUE_REF_PCS", 0) or 0)
            motivos = str(row.get("MOTIVOS_INVALIDA", ""))
            if nec_total <= TOL:
                if bruta_dir > TOL and cobertura + TOL >= bruta_dir:
                    return "DEMANDA_DIRETA_COBERTA_POR_ESTOQUE"
                if exp_bruta > TOL and exp_liq <= TOL:
                    return "DEMANDA_ESTRUTURAL_COBERTA_POR_ESTOQUE"
                if exp_bruta > TOL and exp_liq > TOL and comp <= TOL:
                    return "FALHA_RETORNO_DEMANDA_COMPONENTE_LTP"
                return "ITEM_SEM_DEMANDA_NO_MES"
            if "HOR_FER_ZERO_OU_NEGATIVO" in motivos:
                return "NAO_ALOCADO_FERRAMENTA_SEM_CAPACIDADE"
            if "HOR_REC_ZERO_OU_NEGATIVO" in motivos:
                return "NAO_ALOCADO_RECURSO_SEM_CAPACIDADE"
            if "PCS_HORA_ZERO_OU_INVALIDO" in motivos:
                return "NAO_ALOCADO_PCS_HORA_INVALIDO"
            return "NAO_ALOCADO_SEM_ALTERNATIVA_VALIDA"

        base_status["STATUS_SUGERIDO_PLANO"] = base_status.apply(_classificar_status_auditoria, axis=1)
        base_status["_KEY_PLANO"] = _key_mes_id(base_status)
        plano_producao["_KEY_PLANO"] = _key_mes_id(plano_producao)

        diag_cols_auditoria = ["DEMANDA_BRUTA_DIRETA_REF_PCS", "COBERTURA_ESTOQUE_REF_PCS", "NEC_DIRETA_CALC_REF_PCS", "NEC_DIRETA_PCS", "NEC_COMP_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_PCS_CONSOLIDADA"]
        for c in diag_cols_auditoria:
            if c not in plano_producao.columns:
                plano_producao[c] = np.nan

        base_merge_cols = ["_KEY_PLANO", "STATUS_SUGERIDO_PLANO"] + [c for c in diag_cols_auditoria if c in base_status.columns]
        plano_producao = plano_producao.merge(base_status[base_merge_cols].drop_duplicates("_KEY_PLANO"), on="_KEY_PLANO", how="left", suffixes=("", "_AUD"))
        for c in diag_cols_auditoria:
            ca = c + "_AUD"
            if ca in plano_producao.columns:
                plano_producao[c] = plano_producao[c].where(plano_producao[c].notna(), plano_producao[ca])
                plano_producao = plano_producao.drop(columns=[ca])

        if "MOTIVO_STATUS" not in plano_producao.columns:
            plano_producao["MOTIVO_STATUS"] = ""
        qtd_plan = to_num(plano_producao["QTD_PRODUZIR_SOLVER"])
        plano_producao.loc[qtd_plan > TOL, "MOTIVO_STATUS"] = "PRODUCAO_ALOCADA_PELO_SOLVER"
        mask_zero = qtd_plan <= TOL
        plano_producao.loc[mask_zero & plano_producao["STATUS_SUGERIDO_PLANO"].notna(), "MOTIVO_STATUS"] = plano_producao.loc[mask_zero & plano_producao["STATUS_SUGERIDO_PLANO"].notna(), "STATUS_SUGERIDO_PLANO"]
        for idx in plano_producao.index[mask_zero]:
            motivo = str(plano_producao.at[idx, "MOTIVO_STATUS"])
            plano_producao.at[idx, "STATUS_DECISAO"] = _map_status_decisao_auditoria(motivo)
            if "MOTIVO_LIMITANTE" in plano_producao.columns:
                plano_producao.at[idx, "MOTIVO_LIMITANTE"] = _motivo_limitante_auditoria(motivo)
            if "MOTIVO_CORTE" in plano_producao.columns:
                plano_producao.at[idx, "MOTIVO_CORTE"] = _motivo_corte_auditoria(motivo)

        keys_plano = set(plano_producao["_KEY_PLANO"].astype(str))
        faltantes_auditoria = base_status[~base_status["_KEY_PLANO"].astype(str).isin(keys_plano)].copy()
        linhas_zero = []
        for _, r in faltantes_auditoria.iterrows():
            row = {c: 0 for c in plano_producao.columns if c != "STATUS_SUGERIDO_PLANO"}
            for c in ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP"] + diag_cols_auditoria:
                if c in row and c in r.index:
                    row[c] = r[c]
            motivo = str(r.get("STATUS_SUGERIDO_PLANO", "ITEM_SEM_DEMANDA_NO_MES"))
            row["STATUS_DECISAO"] = _map_status_decisao_auditoria(motivo)
            row["MOTIVO_STATUS"] = motivo
            if "MOTIVO_LIMITANTE" in row:
                row["MOTIVO_LIMITANTE"] = _motivo_limitante_auditoria(motivo)
            if "MOTIVO_CORTE" in row:
                row["MOTIVO_CORTE"] = _motivo_corte_auditoria(motivo)
            if "RODADA" in row:
                row["RODADA"] = 0
            row["_KEY_PLANO"] = r["_KEY_PLANO"]
            linhas_zero.append(row)

        if linhas_zero:
            plano_producao = pd.concat([plano_producao, pd.DataFrame(linhas_zero).reindex(columns=plano_producao.columns)], ignore_index=True)

        plano_producao = plano_producao.drop(columns=[c for c in ["STATUS_SUGERIDO_PLANO", "_KEY_PLANO"] if c in plano_producao.columns])
    except Exception as e:
        print("Aviso auditoria operacional plano completo:", e)



    # ============================================================
    # 06C. CORREÇÃO rotina operacional - PLANO FINAL SEM ALTERAR LAYOUT
    # ============================================================
    # Regra:
    # - não criar coluna nova para o cliente;
    # - não renomear coluna;
    # - não remover coluna;
    # - manter QTD_PRODUZIR_SOLVER como coluna oficial da 03_PLANO_PRODUCAO;
    # - atualizar apenas o valor de QTD_PRODUZIR_SOLVER para a produção final válida,
    #   usando as colunas já existentes desde a rotina operacional.

    pass

    _keys_roteamento = ['MES_REF', 'ID_PROD_UNID_FAT']
    if 'MES_REF' in plano_producao.columns:
        plano_producao['MES_REF'] = pd.to_datetime(plano_producao['MES_REF'], errors='coerce')
    if 'MES_REF' in nao_atend_item.columns:
        nao_atend_item['MES_REF'] = pd.to_datetime(nao_atend_item['MES_REF'], errors='coerce')

    # Restaura colunas já validadas na rotina operacional quando a execução local ainda não as tiver montado.
    for _c in ['MOTIVO_AJUSTE_PAI_FILHO', 'QTD_REALOCADA_CAP_LIBERADA', 'QTD_FINAL_APOS_REALOCACAO', 'QTD_GAP_FINAL_APOS_REALOCACAO']:
        if _c not in plano_producao.columns:
            plano_producao[_c] = '' if _c == 'MOTIVO_AJUSTE_PAI_FILHO' else 0.0

    if 'MOTIVO_AJUSTE_PAI_FILHO' in nao_atend_item.columns:
        _mot_pf_roteamento = nao_atend_item[_keys_roteamento + ['MOTIVO_AJUSTE_PAI_FILHO']].drop_duplicates(_keys_roteamento)
        plano_producao = plano_producao.merge(_mot_pf_roteamento, on=_keys_roteamento, how='left', suffixes=('', '_NA_ROTEAMENTO'))
        if 'MOTIVO_AJUSTE_PAI_FILHO_NA_ROTEAMENTO' in plano_producao.columns:
            plano_producao['MOTIVO_AJUSTE_PAI_FILHO'] = np.where(
                plano_producao['MOTIVO_AJUSTE_PAI_FILHO'].astype(str).str.len() > 0,
                plano_producao['MOTIVO_AJUSTE_PAI_FILHO'],
                plano_producao['MOTIVO_AJUSTE_PAI_FILHO_NA_ROTEAMENTO'].fillna('')
            )
            plano_producao = plano_producao.drop(columns=['MOTIVO_AJUSTE_PAI_FILHO_NA_ROTEAMENTO'])

    # Restaura colunas da guia 04 que já existiam na rotina operacional.
    if 'FATOR_AJUSTE_SOBRE_SOLVER' not in nao_atend_item.columns:
        nao_atend_item['FATOR_AJUSTE_SOBRE_SOLVER'] = nao_atend_item['FATOR_FINAL_VIAVEL_PAI_FILHO'] if 'FATOR_FINAL_VIAVEL_PAI_FILHO' in nao_atend_item.columns else 1.0
    if 'QTD_GAP_APOS_PAI_FILHO' not in nao_atend_item.columns:
        _nec_na_roteamento = to_num(nao_atend_item['NEC_PCS']) if 'NEC_PCS' in nao_atend_item.columns else 0.0
        _final_na_roteamento = to_num(nao_atend_item['QTD_FINAL_VIAVEL_PAI_FILHO']) if 'QTD_FINAL_VIAVEL_PAI_FILHO' in nao_atend_item.columns else to_num(nao_atend_item['QTD_ATENDIDA_SOLVER'])
        nao_atend_item['QTD_GAP_APOS_PAI_FILHO'] = np.maximum(_nec_na_roteamento - _final_na_roteamento, 0)
    if 'BLOQUEADO_AJUSTE_PAI_FILHO' not in nao_atend_item.columns:
        _mot_na_roteamento = nao_atend_item['MOTIVO_AJUSTE_PAI_FILHO'].astype(str) if 'MOTIVO_AJUSTE_PAI_FILHO' in nao_atend_item.columns else ''
        nao_atend_item['BLOQUEADO_AJUSTE_PAI_FILHO'] = _mot_na_roteamento.eq('FILHO_COM_ROTEIRO_LIMITOU_PAI')

    # Base de realocação por linha produtiva.
    # rotina: a quantidade realocada deve ser somada na alternativa em que a realocação ocorreu.
    # Não distribuir a realocação proporcionalmente por todas as linhas do item, para não criar estouro artificial de capacidade.
    _realoc_linha_roteamento = pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID','PRIOR_MATPAR','PRIOR_ROT','QTD_REALOCADA_CAP_LIBERADA'])
    try:
        if 'realocacao_cap_liberada' in locals() and isinstance(realocacao_cap_liberada, pd.DataFrame) and not realocacao_cap_liberada.empty:
            _keys_realoc_linha_roteamento = ['MES_REF','ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID','PRIOR_MATPAR','PRIOR_ROT']
            _cols_realoc_linha_roteamento = [c for c in _keys_realoc_linha_roteamento + ['QTD_REALOCADA'] if c in realocacao_cap_liberada.columns]
            if set(_keys_realoc_linha_roteamento).issubset(_cols_realoc_linha_roteamento) and 'QTD_REALOCADA' in _cols_realoc_linha_roteamento:
                _realoc_linha_roteamento = realocacao_cap_liberada[_cols_realoc_linha_roteamento].copy()
                _realoc_linha_roteamento['MES_REF'] = pd.to_datetime(_realoc_linha_roteamento['MES_REF'], errors='coerce')
                _realoc_linha_roteamento['QTD_REALOCADA'] = to_num(_realoc_linha_roteamento['QTD_REALOCADA'])
                _realoc_linha_roteamento = _realoc_linha_roteamento[_realoc_linha_roteamento['QTD_REALOCADA'] > TOL].copy()
                _realoc_linha_roteamento = _realoc_linha_roteamento.groupby(_keys_realoc_linha_roteamento, as_index=False).agg(QTD_REALOCADA_CAP_LIBERADA=('QTD_REALOCADA', 'sum'))
    except Exception as e:
        print('Aviso rotina: não foi possível usar realocacao_cap_liberada por linha:', e)

    plano_producao['_QTD_BASE_FINAL_FINAL'] = to_num(plano_producao['QTD_PRODUZIR_FINAL_VIAVEL_EST']) if 'QTD_PRODUZIR_FINAL_VIAVEL_EST' in plano_producao.columns else to_num(plano_producao['QTD_PRODUZIR_SOLVER'])
    if not _realoc_linha_roteamento.empty:
        _keys_merge_realoc_roteamento = ['MES_REF','ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID','PRIOR_MATPAR','PRIOR_ROT']
        plano_producao = plano_producao.merge(_realoc_linha_roteamento, on=_keys_merge_realoc_roteamento, how='left', suffixes=('', '_LINHA_ROTEAMENTO'))
        if 'QTD_REALOCADA_CAP_LIBERADA_LINHA_ROTEAMENTO' in plano_producao.columns:
            plano_producao['QTD_REALOCADA_CAP_LIBERADA'] = to_num(plano_producao['QTD_REALOCADA_CAP_LIBERADA_LINHA_ROTEAMENTO'])
            plano_producao = plano_producao.drop(columns=['QTD_REALOCADA_CAP_LIBERADA_LINHA_ROTEAMENTO'])
        else:
            plano_producao['QTD_REALOCADA_CAP_LIBERADA'] = 0.0
    else:
        plano_producao['QTD_REALOCADA_CAP_LIBERADA'] = 0.0

    plano_producao['QTD_FINAL_APOS_REALOCACAO'] = plano_producao['_QTD_BASE_FINAL_FINAL'] + to_num(plano_producao['QTD_REALOCADA_CAP_LIBERADA'])

    if 'NEC_PCS_CONSOLIDADA' in plano_producao.columns:
        _nec_ref_final = to_num(plano_producao['NEC_PCS_CONSOLIDADA'])
    elif 'NEC_PCS' in plano_producao.columns:
        _nec_ref_final = to_num(plano_producao['NEC_PCS'])
    else:
        _nec_ref_final = 0.0
    plano_producao['QTD_GAP_FINAL_APOS_REALOCACAO'] = np.maximum(_nec_ref_final - to_num(plano_producao['QTD_FINAL_APOS_REALOCACAO']), 0)

    # Atualiza a coluna oficial sem mudar o nome.
    plano_producao['QTD_PRODUZIR_SOLVER'] = to_num(plano_producao['QTD_FINAL_APOS_REALOCACAO'])
    plano_producao['HR_PRODUZIR_SOLVER'] = np.where(to_num(plano_producao['PCS_HORA']) > TOL, to_num(plano_producao['QTD_PRODUZIR_SOLVER']) / to_num(plano_producao['PCS_HORA']), 0)
    if 'PESO_PROD_KG' in plano_producao.columns and 'VOL_PRODUZIR_SOLVER' in plano_producao.columns:
        plano_producao['VOL_PRODUZIR_SOLVER'] = to_num(plano_producao['QTD_PRODUZIR_SOLVER']) * to_num(plano_producao['PESO_PROD_KG'])
    plano_producao = plano_producao.drop(columns=['_QTD_BASE_FINAL_FINAL'])

    # Recalcula resumos que dependem diretamente da coluna oficial, sem mudar layout.
    prod_recurso = plano_producao.groupby('ALOC_REC', as_index=False).agg(
        HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER', 'sum'),
        QTD_PRODUZIR_SOLVER=('QTD_PRODUZIR_SOLVER', 'sum'),
        QTD_ITENS=('ID_PROD_UNID_FAT', 'nunique')
    ) if not plano_producao.empty else pd.DataFrame(columns=['ALOC_REC', 'HR_PRODUZIR_SOLVER', 'QTD_PRODUZIR_SOLVER', 'QTD_ITENS'])
    resumo_recursos = bd_cap_recursos.merge(prod_recurso, on='ALOC_REC', how='left').fillna({'HR_PRODUZIR_SOLVER': 0, 'QTD_PRODUZIR_SOLVER': 0, 'QTD_ITENS': 0})
    resumo_recursos['OCUPACAO_RECURSO_PCT'] = np.where(resumo_recursos['HOR_REC'] > 0, resumo_recursos['HR_PRODUZIR_SOLVER'] / resumo_recursos['HOR_REC'], 0)
    resumo_recursos['HR_RECURSO_OCIOSA'] = resumo_recursos['HOR_REC'] - resumo_recursos['HR_PRODUZIR_SOLVER']
    resumo_recursos['ESTOURO_HR_RECURSO'] = np.where(resumo_recursos['HR_PRODUZIR_SOLVER'] > resumo_recursos['HOR_REC'] + TOL, resumo_recursos['HR_PRODUZIR_SOLVER'] - resumo_recursos['HOR_REC'], 0)
    resumo_recursos = resumo_recursos[['ALOC_REC', 'HOR_REC', 'HR_PRODUZIR_SOLVER', 'OCUPACAO_RECURSO_PCT', 'HR_RECURSO_OCIOSA', 'ESTOURO_HR_RECURSO', 'QTD_PRODUZIR_SOLVER', 'QTD_ITENS']].sort_values(['OCUPACAO_RECURSO_PCT', 'HR_PRODUZIR_SOLVER'], ascending=[False, False])

    prod_ferramenta = plano_producao.groupby('COD_FER_UNID', as_index=False).agg(
        HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER', 'sum'),
        QTD_PRODUZIR_SOLVER=('QTD_PRODUZIR_SOLVER', 'sum'),
        QTD_ITENS=('ID_PROD_UNID_FAT', 'nunique'),
        QTD_RECURSOS=('ALOC_REC', 'nunique')
    ) if not plano_producao.empty else pd.DataFrame(columns=['COD_FER_UNID', 'HR_PRODUZIR_SOLVER', 'QTD_PRODUZIR_SOLVER', 'QTD_ITENS', 'QTD_RECURSOS'])
    resumo_ferramentas = bd_cap_ferramentas.merge(prod_ferramenta, on='COD_FER_UNID', how='left').fillna({'HR_PRODUZIR_SOLVER': 0, 'QTD_PRODUZIR_SOLVER': 0, 'QTD_ITENS': 0, 'QTD_RECURSOS': 0})
    resumo_ferramentas['OCUPACAO_FER_PCT'] = np.where(resumo_ferramentas['HOR_FER'] > 0, resumo_ferramentas['HR_PRODUZIR_SOLVER'] / resumo_ferramentas['HOR_FER'], 0)
    resumo_ferramentas['HR_FER_OCIOSA'] = resumo_ferramentas['HOR_FER'] - resumo_ferramentas['HR_PRODUZIR_SOLVER']
    resumo_ferramentas['ESTOURO_HR_FER'] = np.where(resumo_ferramentas['HR_PRODUZIR_SOLVER'] > resumo_ferramentas['HOR_FER'] + TOL, resumo_ferramentas['HR_PRODUZIR_SOLVER'] - resumo_ferramentas['HOR_FER'], 0)
    resumo_ferramentas = resumo_ferramentas[['COD_FER_UNID', 'HOR_FER', 'HR_PRODUZIR_SOLVER', 'OCUPACAO_FER_PCT', 'HR_FER_OCIOSA', 'ESTOURO_HR_FER', 'QTD_PRODUZIR_SOLVER', 'QTD_ITENS', 'QTD_RECURSOS']].sort_values(['OCUPACAO_FER_PCT', 'HR_PRODUZIR_SOLVER'], ascending=[False, False])

    # Recalcula diagnóstico de alternativas para manter consistência com a coluna oficial.
    try:
        uso_alt = plano_producao.groupby(['ID_PROD_UNID_FAT', 'ALOC_REC', 'COD_FER_UNID', 'PRIOR_MATPAR', 'PRIOR_ROT'], as_index=False).agg(
            HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER', 'sum'),
            QTD_PRODUZIR_SOLVER=('QTD_PRODUZIR_SOLVER', 'sum')
        ) if not plano_producao.empty else pd.DataFrame(columns=['ID_PROD_UNID_FAT', 'ALOC_REC', 'COD_FER_UNID', 'PRIOR_MATPAR', 'PRIOR_ROT', 'HR_PRODUZIR_SOLVER', 'QTD_PRODUZIR_SOLVER'])
        diagnostico_alternativas = alternativas_item.merge(uso_alt, on=['ID_PROD_UNID_FAT', 'ALOC_REC', 'COD_FER_UNID', 'PRIOR_MATPAR', 'PRIOR_ROT'], how='left')
        diagnostico_alternativas['HR_PRODUZIR_SOLVER'] = diagnostico_alternativas['HR_PRODUZIR_SOLVER'].fillna(0)
        diagnostico_alternativas['QTD_PRODUZIR_SOLVER'] = diagnostico_alternativas['QTD_PRODUZIR_SOLVER'].fillna(0)
        diagnostico_alternativas['ALTERNATIVA_USADA'] = diagnostico_alternativas['HR_PRODUZIR_SOLVER'] > TOL
    except Exception as e:
        print('Aviso rotina: diagnóstico de alternativas não recalculado:', e)


    # Preserva exatamente a ordem de colunas já validada na rotina operacional para as guias afetadas.
    _ORDEM_03_PLANO_FINAL = ['MES_REF', 'ID_PROD_UNID_FAT', 'COD_PROD', 'DESC_PROD', 'UNID_PROD', 'UNID_FAT', 'TIPO_PROD', 'LINHA_PROD', 'FAMILIA_PROD', 'MO', 'RODADA', 'PRIOR_MATPAR', 'PRIOR_ROT', 'CLASS_ROT', 'ALOC_REC', 'COD_FER_UNID', 'PCS_HORA', 'HOR_REC', 'HOR_FER', 'HOR_CAP', 'QTD_PENDENTE_ANTES', 'QTD_SOLICITADA', 'HR_SOLICITADA', 'CAP_RESTANTE_ALOC_REC_ANTES', 'CAP_RESTANTE_COD_FER_ANTES', 'CAP_DISPONIVEL_LINHA', 'HR_DEMANDA_TOTAL_ALOC_REC', 'HR_DEMANDA_TOTAL_COD_FER', 'FATOR_RATEIO_ALOC_REC', 'FATOR_RATEIO_COD_FER', 'FATOR_RATEIO', 'MOTIVO_LIMITANTE', 'QTD_PRODUZIR_SOLVER', 'PESO_PROD_KG', 'VOL_PRODUZIR_SOLVER', 'HR_PRODUZIR_SOLVER', 'QTD_PENDENTE_DEPOIS', 'MOTIVO_CORTE', 'STATUS_DECISAO', 'DEMANDA_BRUTA_DIRETA_REF_PCS', 'COBERTURA_ESTOQUE_REF_PCS', 'NEC_DIRETA_CALC_REF_PCS', 'NEC_DIRETA_PCS', 'NEC_COMP_PCS', 'NEC_EXPLODIDA_BRUTA_PCS', 'ESTOQUE_CONSUMIDO_EXPLOSAO_PCS', 'NEC_EXPLODIDA_PCS', 'NEC_PCS_CONSOLIDADA', 'MOTIVO_STATUS', 'QTD_REDUCAO_POS_PAI_FILHO', 'FATOR_FINAL_VIAVEL_ORIGEM', 'FATOR_AJUSTE_SOBRE_SOLVER', 'MOTIVO_AJUSTE_PAI_FILHO', 'QTD_PRODUZIR_FINAL_VIAVEL_EST', 'HR_LIBERADA_EST', 'QTD_REALOCADA_CAP_LIBERADA', 'QTD_FINAL_APOS_REALOCACAO', 'QTD_GAP_FINAL_APOS_REALOCACAO']
    _ORDEM_04_NAO_ATEND_FINAL = ['MES_REF', 'ID_PROD_UNID_FAT', 'COD_PROD', 'DESC_PROD', 'UNID_PROD', 'UNID_FAT', 'TIPO_PROD', 'NEC_PCS_ORIGINAL', 'NEC_ESTRUTURA_PAI_PCS', 'NEC_DIRETA_BRUTA_PCS', 'NEC_DIRETA_PCS', 'NEC_EXPLODIDA_BRUTA_PCS', 'ESTOQUE_CONSUMIDO_EXPLOSAO_PCS', 'NEC_EXPLODIDA_PCS', 'NEC_COMP_PCS', 'NEC_PCS_CONSOLIDADA', 'NEC_PCS', 'QTD_ATENDIDA_SOLVER', 'QTD_NAO_ATEND_SOLVER', 'PERC_ATENDIDO_SOLVER', 'QTD_FINAL_VIAVEL_PAI_FILHO', 'QTD_AJUSTE_PAI_FILHO', 'FATOR_FINAL_VIAVEL_PAI_FILHO', 'FATOR_AJUSTE_SOBRE_SOLVER', 'MOTIVO_AJUSTE_PAI_FILHO', 'QTD_GAP_APOS_PAI_FILHO', 'BLOQUEADO_AJUSTE_PAI_FILHO']
    plano_producao = plano_producao[[c for c in _ORDEM_03_PLANO_FINAL if c in plano_producao.columns] + [c for c in plano_producao.columns if c not in _ORDEM_03_PLANO_FINAL]].copy()
    nao_atend_item = nao_atend_item[[c for c in _ORDEM_04_NAO_ATEND_FINAL if c in nao_atend_item.columns] + [c for c in nao_atend_item.columns if c not in _ORDEM_04_NAO_ATEND_FINAL]].copy()

    pass



    # ============================================================
    # 06D. FECHAMENTO FINAL LEVE - V66
    # ============================================================
    # Nesta versao de validacao da Matriz, a otimizacao principal opera em modo continuo.
    # LIMIT_PCS/LOTE_MIN/QTD_EMB nao sao aplicados aqui; o fechamento industrial final
    # sera validado em etapa posterior, depois que a convergencia REC/FER estiver fechada.
    # Mantem-se apenas o arredondamento legado da materializacao do plano, sem usar lote/multiplo.
    # Como as quantidades sao nao negativas, floor(x + 0.5) implementa arredondamento convencional
    # com .5 sempre para cima, evitando o arredondamento bancario do np.round.
    if not plano_producao.empty:
        _qtd_final_v66 = pd.to_numeric(plano_producao['QTD_PRODUZIR_SOLVER'], errors='coerce').fillna(0.0).clip(lower=0.0)
        plano_producao['QTD_PRODUZIR_SOLVER'] = np.floor(_qtd_final_v66 + 0.5)

        _pcs_hora_v66 = pd.to_numeric(plano_producao['PCS_HORA'], errors='coerce').fillna(0.0)
        plano_producao['HR_PRODUZIR_SOLVER'] = np.divide(
            plano_producao['QTD_PRODUZIR_SOLVER'].to_numpy(dtype=float),
            _pcs_hora_v66.to_numpy(dtype=float),
            out=np.zeros(len(plano_producao), dtype=float),
            where=_pcs_hora_v66.to_numpy(dtype=float) > TOL
        )

        if 'PESO_PROD_KG' in plano_producao.columns and 'VOL_PRODUZIR_SOLVER' in plano_producao.columns:
            plano_producao['VOL_PRODUZIR_SOLVER'] = plano_producao['QTD_PRODUZIR_SOLVER'] * pd.to_numeric(plano_producao['PESO_PROD_KG'], errors='coerce').fillna(0.0)

        # Mantem campos historicos da saida coerentes, sem executar fechamento/propagacao pos-Solver.
        if 'QTD_PRODUZIR_FINAL_VIAVEL_EST' in plano_producao.columns:
            plano_producao['QTD_PRODUZIR_FINAL_VIAVEL_EST'] = plano_producao['QTD_PRODUZIR_SOLVER']
        if 'QTD_FINAL_APOS_REALOCACAO' in plano_producao.columns:
            plano_producao['QTD_FINAL_APOS_REALOCACAO'] = plano_producao['QTD_PRODUZIR_SOLVER']
        if 'QTD_REDUCAO_POS_PAI_FILHO' in plano_producao.columns:
            plano_producao['QTD_REDUCAO_POS_PAI_FILHO'] = 0.0
        if 'HR_LIBERADA_EST' in plano_producao.columns:
            plano_producao['HR_LIBERADA_EST'] = 0.0
        if 'QTD_REALOCADA_CAP_LIBERADA' in plano_producao.columns:
            plano_producao['QTD_REALOCADA_CAP_LIBERADA'] = 0.0
        if 'MOTIVO_AJUSTE_PAI_FILHO' in plano_producao.columns:
            plano_producao['MOTIVO_AJUSTE_PAI_FILHO'] = 'SEM_AJUSTE_POS_SOLVER'

    # Nao existe mais fechamento pai/filho ou propagacao pos-Solver nesta versao.
    ajuste_pai_filho = pd.DataFrame()
    resumo_pai_filho = pd.DataFrame()
    capacidade_liberada_pf = pd.DataFrame()
    realocacao_cap_liberada = pd.DataFrame()
    resumo_item_realocacao = pd.DataFrame()

    # Mantem somente a demanda estrutural ja calculada antes do Solver para o encadeamento mensal.
    _cols_demanda_final_v66 = [c for c in ['MES_REF','ID_PROD_UNID_FAT','COD_PROD','UNID_PROD','UNID_FAT','NEC_EXPLODIDA_BRUTA_PCS'] if c in nec_consolidada_solver.columns]
    demanda_final_m1 = nec_consolidada_solver[_cols_demanda_final_v66].copy() if _cols_demanda_final_v66 else pd.DataFrame()
    if not demanda_final_m1.empty:
        if 'NEC_EXPLODIDA_BRUTA_PCS' in demanda_final_m1.columns:
            demanda_final_m1 = demanda_final_m1.rename(columns={'NEC_EXPLODIDA_BRUTA_PCS':'NEC_BRUTA_FINAL'})
        elif 'NEC_BRUTA_FINAL' not in demanda_final_m1.columns:
            demanda_final_m1['NEC_BRUTA_FINAL'] = 0.0

    # Recalcula os resumos diretamente da quantidade final inteira.
    prod_recurso = plano_producao.groupby('ALOC_REC', as_index=False).agg(
        HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER', 'sum'),
        QTD_PRODUZIR_SOLVER=('QTD_PRODUZIR_SOLVER', 'sum'),
        QTD_ITENS=('ID_PROD_UNID_FAT', 'nunique')
    ) if not plano_producao.empty else pd.DataFrame(columns=['ALOC_REC','HR_PRODUZIR_SOLVER','QTD_PRODUZIR_SOLVER','QTD_ITENS'])
    resumo_recursos = bd_cap_recursos.merge(prod_recurso, on='ALOC_REC', how='left').fillna({'HR_PRODUZIR_SOLVER':0,'QTD_PRODUZIR_SOLVER':0,'QTD_ITENS':0})
    resumo_recursos['OCUPACAO_RECURSO_PCT'] = np.where(resumo_recursos['HOR_REC'] > 0, resumo_recursos['HR_PRODUZIR_SOLVER'] / resumo_recursos['HOR_REC'], 0)
    resumo_recursos['HR_RECURSO_OCIOSA'] = resumo_recursos['HOR_REC'] - resumo_recursos['HR_PRODUZIR_SOLVER']
    resumo_recursos['ESTOURO_HR_RECURSO'] = np.where(resumo_recursos['HR_PRODUZIR_SOLVER'] > resumo_recursos['HOR_REC'] + TOL, resumo_recursos['HR_PRODUZIR_SOLVER'] - resumo_recursos['HOR_REC'], 0)
    resumo_recursos = resumo_recursos[['ALOC_REC','HOR_REC','HR_PRODUZIR_SOLVER','OCUPACAO_RECURSO_PCT','HR_RECURSO_OCIOSA','ESTOURO_HR_RECURSO','QTD_PRODUZIR_SOLVER','QTD_ITENS']].sort_values(['OCUPACAO_RECURSO_PCT','HR_PRODUZIR_SOLVER'], ascending=[False,False])

    prod_ferramenta = plano_producao.groupby('COD_FER_UNID', as_index=False).agg(
        HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER', 'sum'),
        QTD_PRODUZIR_SOLVER=('QTD_PRODUZIR_SOLVER', 'sum'),
        QTD_ITENS=('ID_PROD_UNID_FAT', 'nunique'),
        QTD_RECURSOS=('ALOC_REC', 'nunique')
    ) if not plano_producao.empty else pd.DataFrame(columns=['COD_FER_UNID','HR_PRODUZIR_SOLVER','QTD_PRODUZIR_SOLVER','QTD_ITENS','QTD_RECURSOS'])
    resumo_ferramentas = bd_cap_ferramentas.merge(prod_ferramenta, on='COD_FER_UNID', how='left').fillna({'HR_PRODUZIR_SOLVER':0,'QTD_PRODUZIR_SOLVER':0,'QTD_ITENS':0,'QTD_RECURSOS':0})
    resumo_ferramentas['OCUPACAO_FER_PCT'] = np.where(resumo_ferramentas['HOR_FER'] > 0, resumo_ferramentas['HR_PRODUZIR_SOLVER'] / resumo_ferramentas['HOR_FER'], 0)
    resumo_ferramentas['HR_FER_OCIOSA'] = resumo_ferramentas['HOR_FER'] - resumo_ferramentas['HR_PRODUZIR_SOLVER']
    resumo_ferramentas['ESTOURO_HR_FER'] = np.where(resumo_ferramentas['HR_PRODUZIR_SOLVER'] > resumo_ferramentas['HOR_FER'] + TOL, resumo_ferramentas['HR_PRODUZIR_SOLVER'] - resumo_ferramentas['HOR_FER'], 0)
    resumo_ferramentas = resumo_ferramentas[['COD_FER_UNID','HOR_FER','HR_PRODUZIR_SOLVER','OCUPACAO_FER_PCT','HR_FER_OCIOSA','ESTOURO_HR_FER','QTD_PRODUZIR_SOLVER','QTD_ITENS','QTD_RECURSOS']].sort_values(['OCUPACAO_FER_PCT','HR_PRODUZIR_SOLVER'], ascending=[False,False])

    # Atualiza atendimento/gap apos o arredondamento final, sem outra regra industrial.
    if not nao_atend_item.empty:
        _atendido_final_v66 = plano_producao.groupby(['MES_REF','ID_PROD_UNID_FAT'], as_index=False).agg(QTD_ATENDIDA_SOLVER=('QTD_PRODUZIR_SOLVER','sum')) if not plano_producao.empty else pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','QTD_ATENDIDA_SOLVER'])
        nao_atend_item = nao_atend_item.drop(columns=['QTD_ATENDIDA_SOLVER'], errors='ignore').merge(_atendido_final_v66, on=['MES_REF','ID_PROD_UNID_FAT'], how='left')
        nao_atend_item['QTD_ATENDIDA_SOLVER'] = pd.to_numeric(nao_atend_item['QTD_ATENDIDA_SOLVER'], errors='coerce').fillna(0.0)
        nao_atend_item['QTD_NAO_ATEND_SOLVER'] = (pd.to_numeric(nao_atend_item['NEC_PCS'], errors='coerce').fillna(0.0) - nao_atend_item['QTD_ATENDIDA_SOLVER']).clip(lower=0.0)
        nao_atend_item['PERC_ATENDIDO_SOLVER'] = np.where(pd.to_numeric(nao_atend_item['NEC_PCS'], errors='coerce').fillna(0.0) > 0, nao_atend_item['QTD_ATENDIDA_SOLVER'] / pd.to_numeric(nao_atend_item['NEC_PCS'], errors='coerce').fillna(0.0), 0.0)
        nao_atend_item['PERC_NAO_ATENDIDO_SOLVER'] = 1.0 - nao_atend_item['PERC_ATENDIDO_SOLVER']

    try:
        _uso_alt_final_v66 = plano_producao.groupby(['ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID','PRIOR_MATPAR','PRIOR_ROT'], as_index=False).agg(HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER','sum'), QTD_PRODUZIR_SOLVER=('QTD_PRODUZIR_SOLVER','sum'))
        diagnostico_alternativas = alternativas_item.merge(_uso_alt_final_v66, on=['ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID','PRIOR_MATPAR','PRIOR_ROT'], how='left')
        diagnostico_alternativas['HR_PRODUZIR_SOLVER'] = diagnostico_alternativas['HR_PRODUZIR_SOLVER'].fillna(0.0)
        diagnostico_alternativas['QTD_PRODUZIR_SOLVER'] = diagnostico_alternativas['QTD_PRODUZIR_SOLVER'].fillna(0.0)
        diagnostico_alternativas['ALTERNATIVA_USADA'] = diagnostico_alternativas['HR_PRODUZIR_SOLVER'] > TOL
    except Exception as exc:
        print('Diagnóstico de alternativas não recalculado:', exc)


    # ============================================================
    # 06E.1. RASTREABILIDADE DA FORMACAO DA NEC NO PLANO
    # ============================================================
    # Exibe no 03_PLANO_PRODUCAO os mesmos campos de necessidade usados pelo motor.
    # M1: os valores permanecem os dados originais do LTP.
    # M2 em diante: LTP_EST_INI_PCS e LTP_CART_ARR_MES_ANT ja chegam em bd_mes
    # atualizados pelo encadeamento mensal; portanto nao sao criadas colunas paralelas.
    _cols_formacao_nec_v58 = [
        'LTP_CART_ARR_MES_ANT',
        'LTP_CART_MES_ATUAL',
        'LTP_SALDO_PREV_PCS',
        'LTP_EST_SEG_PCS',
        'LTP_EST_INI_PCS',
        'LTP_EST_TRANS_PCS',
        'ORI_TOT_PCS',
        'TRIANG_TOT_PCS',
        'NEC_PCS',
    ]
    _cols_formacao_nec_v58 = [c for c in _cols_formacao_nec_v58 if c in bd_mes.columns]

    if not plano_producao.empty and _cols_formacao_nec_v58:
        _base_formacao_nec_v58 = bd_mes.copy()
        _base_formacao_nec_v58['MES_REF'] = pd.to_datetime(_base_formacao_nec_v58['MES_REF'], errors='coerce')
        _base_formacao_nec_v58['ID_PROD_UNID_FAT'] = _base_formacao_nec_v58['ID_PROD_UNID_FAT'].astype(str)
        for _c_v58 in _cols_formacao_nec_v58:
            _base_formacao_nec_v58[_c_v58] = to_num(_base_formacao_nec_v58[_c_v58])

        _base_formacao_nec_v58 = (
            _base_formacao_nec_v58
            .groupby(['MES_REF', 'ID_PROD_UNID_FAT'], as_index=False)
            .agg({_c_v58: 'max' for _c_v58 in _cols_formacao_nec_v58})
        )

        plano_producao['MES_REF'] = pd.to_datetime(plano_producao['MES_REF'], errors='coerce')
        plano_producao['ID_PROD_UNID_FAT'] = plano_producao['ID_PROD_UNID_FAT'].astype(str)
        plano_producao = plano_producao.drop(columns=[c for c in _cols_formacao_nec_v58 if c in plano_producao.columns], errors='ignore')
        plano_producao = plano_producao.merge(
            _base_formacao_nec_v58,
            on=['MES_REF', 'ID_PROD_UNID_FAT'],
            how='left',
            validate='many_to_one',
        )
        for _c_v58 in _cols_formacao_nec_v58:
            plano_producao[_c_v58] = to_num(plano_producao[_c_v58])

        # Traz regras operacionais e classificacoes para a mesma tabela de validação do plano.
        _cols_validacao_v58 = ['QTD_EMB','LOTE_MIN','CURVA_ABC','CURVA_123']
        _base_validacao_v58 = bd_mes[['MES_REF','ID_PROD_UNID_FAT'] + [c for c in _cols_validacao_v58 if c in bd_mes.columns]].copy()
        for _c_v58 in ['QTD_EMB','LOTE_MIN']:
            if _c_v58 in _base_validacao_v58.columns:
                _base_validacao_v58[_c_v58] = to_num(_base_validacao_v58[_c_v58])
        _agg_validacao_v58 = {c:'first' for c in _base_validacao_v58.columns if c not in ['MES_REF','ID_PROD_UNID_FAT']}
        _base_validacao_v58 = _base_validacao_v58.groupby(['MES_REF','ID_PROD_UNID_FAT'],as_index=False).agg(_agg_validacao_v58)
        plano_producao = plano_producao.drop(columns=[c for c in _cols_validacao_v58 if c in plano_producao.columns], errors='ignore')
        plano_producao = plano_producao.merge(_base_validacao_v58,on=['MES_REF','ID_PROD_UNID_FAT'],how='left',validate='many_to_one')
        for _c_v58 in _cols_validacao_v58:
            if _c_v58 not in plano_producao.columns:
                plano_producao[_c_v58] = pd.NA

        # Mantem os campos de formacao da necessidade juntos e no inicio do bloco de negocio.
        _prefixo_plano_v58 = [
            'MES_REF', 'ID_PROD_UNID_FAT', 'COD_PROD', 'DESC_PROD', 'UNID_PROD', 'UNID_FAT', 'TIPO_PROD',
            'QTD_EMB', 'LOTE_MIN', 'CURVA_ABC', 'CURVA_123',
            'LTP_CART_ARR_MES_ANT', 'LTP_CART_MES_ATUAL', 'LTP_SALDO_PREV_PCS', 'LTP_EST_SEG_PCS',
            'LTP_EST_INI_PCS', 'LTP_EST_TRANS_PCS', 'ORI_TOT_PCS', 'TRIANG_TOT_PCS', 'NEC_PCS',
        ]
        plano_producao = plano_producao[
            [c for c in _prefixo_plano_v58 if c in plano_producao.columns]
            + [c for c in plano_producao.columns if c not in _prefixo_plano_v58]
        ].copy()

    return {
        'resumo_recursos': resumo_recursos,
        'resumo_ferramentas': resumo_ferramentas,
        'plano_producao': plano_producao,
        'nao_atend_item': nao_atend_item,
        'auditoria_capacidade': auditoria_capacidade,
        'alternativas_item': alternativas_item,
        'rastro_rateio': rastro_rateio,
        'diagnostico_matriz': diagnostico_matriz,
        'diagnostico_matriz_rotas': diagnostico_matriz_rotas,
        'diagnostico_item': diagnostico_item,
        'diagnostico_alternativas': diagnostico_alternativas,
        'estrutura_explodida': estrutura_explodida,
        'nec_explodida_componentes': nec_explodida_componentes,
        'nec_consolidada_solver': nec_consolidada_solver,
        'ajuste_pai_filho': ajuste_pai_filho,
        'resumo_pai_filho': resumo_pai_filho,
        'capacidade_liberada_pf': capacidade_liberada_pf,
        'realocacao_cap_liberada': realocacao_cap_liberada,
        'resumo_item_realocacao': resumo_item_realocacao,
        'demanda_final_m1': demanda_final_m1,
    }

# ============================================================
# 06F. PROCESSAMENTO SEQUENCIAL DE TODOS OS PERIODOS
# ============================================================

def _chave_estado_v58(cod_prod, unid_fat):
    # Chave oficial alinhada com o cliente: estoque/carteira fecham e reabrem por COD_PROD + UNID_FAT.
    return (str(cod_prod).strip(), str(unid_fat).strip())


def _base_item_periodo_v58(bd_periodo):
    cols_valor = [
        'LTP_EST_INI_PCS','LTP_EST_TRANS_PCS','ORI_TOT_PCS','TRIANG_TOT_PCS',
        'LTP_SALDO_PREV_PCS','LTP_CART_MES_ATUAL','LTP_CART_ARR_MES_ANT'
    ]
    base = bd_periodo.copy()
    base['MES_REF'] = pd.to_datetime(base['MES_REF'], errors='coerce')
    base['COD_PROD'] = normalizar_codigo_serie(base['COD_PROD'])
    base['UNID_FAT'] = base['UNID_FAT'].astype('string').fillna('').str.strip()
    for c in cols_valor:
        if c not in base.columns:
            base[c] = 0.0
        base[c] = to_num(base[c])
    chave = ['MES_REF','ID_PROD_UNID_FAT']
    agg = {c:'max' for c in cols_valor}
    agg.update({c:'first' for c in ['COD_PROD','UNID_PROD','UNID_FAT'] if c in base.columns})
    return base.groupby(chave, as_index=False).agg(agg)


def _calcular_estado_proximo_periodo_v58(bd_periodo, resultado_periodo):
    base = _base_item_periodo_v58(bd_periodo)
    plano = resultado_periodo['plano_producao'].copy()
    if plano.empty:
        prod = pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','PRODUCAO_MES'])
    else:
        plano['MES_REF'] = pd.to_datetime(plano['MES_REF'], errors='coerce')
        plano['QTD_PRODUZIR_SOLVER'] = to_num(plano['QTD_PRODUZIR_SOLVER'])
        prod = plano.groupby(['MES_REF','ID_PROD_UNID_FAT'], as_index=False).agg(PRODUCAO_MES=('QTD_PRODUZIR_SOLVER','sum'))

    dem = resultado_periodo.get('demanda_final_m1', pd.DataFrame()).copy()
    if dem.empty or 'NEC_BRUTA_FINAL' not in dem.columns:
        estrut = pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','DEMANDA_ESTRUTURAL_MES'])
    else:
        dem['MES_REF'] = pd.to_datetime(dem['MES_REF'], errors='coerce')
        dem['NEC_BRUTA_FINAL'] = to_num(dem['NEC_BRUTA_FINAL'])
        estrut = dem.groupby(['MES_REF','ID_PROD_UNID_FAT'], as_index=False).agg(DEMANDA_ESTRUTURAL_MES=('NEC_BRUTA_FINAL','max'))

    saldo = base.merge(prod, on=['MES_REF','ID_PROD_UNID_FAT'], how='left').merge(estrut, on=['MES_REF','ID_PROD_UNID_FAT'], how='left')
    saldo[['PRODUCAO_MES','DEMANDA_ESTRUTURAL_MES']] = saldo[['PRODUCAO_MES','DEMANDA_ESTRUTURAL_MES']].fillna(0.0)
    saldo['SALDO_PROX_PERIODO'] = (
        saldo['LTP_EST_INI_PCS'] + saldo['LTP_EST_TRANS_PCS'] + saldo['ORI_TOT_PCS'] + saldo['TRIANG_TOT_PCS']
        + saldo['PRODUCAO_MES'] - saldo['LTP_SALDO_PREV_PCS'] - saldo['LTP_CART_MES_ATUAL']
        - saldo['LTP_CART_ARR_MES_ANT'] - saldo['DEMANDA_ESTRUTURAL_MES']
    )
    saldo['ESTOQUE_INICIAL_PROX_PERIODO'] = saldo['SALDO_PROX_PERIODO'].clip(lower=0.0)
    saldo['CARTEIRA_ARRASTE_PROX_PERIODO'] = (-saldo['SALDO_PROX_PERIODO']).clip(lower=0.0)
    saldo['CHAVE_ESTADO'] = [_chave_estado_v58(c, u) for c, u in zip(saldo['COD_PROD'], saldo['UNID_FAT'])]
    return saldo


def _aplicar_estado_anterior_v58(bd_periodo, estado_itens):
    out = bd_periodo.copy()
    if 'LTP_EST_INI_PCS' not in out.columns:
        out['LTP_EST_INI_PCS'] = 0.0
    if 'LTP_CART_ARR_MES_ANT' not in out.columns:
        out['LTP_CART_ARR_MES_ANT'] = 0.0
    out['COD_PROD'] = normalizar_codigo_serie(out['COD_PROD'])
    out['UNID_FAT'] = out['UNID_FAT'].astype('string').fillna('').str.strip()
    chaves = [_chave_estado_v58(c, u) for c, u in zip(out['COD_PROD'], out['UNID_FAT'])]
    # Itens novos no horizonte, sem estado anterior, entram com zero projetado; itens existentes usam o fechamento anterior.
    out['LTP_EST_INI_PCS'] = [float(estado_itens[k]['ESTOQUE']) if k in estado_itens else 0.0 for k in chaves]
    out['LTP_CART_ARR_MES_ANT'] = [float(estado_itens[k]['ARRASTE']) if k in estado_itens else 0.0 for k in chaves]
    out['_CHAVE_ESTADO_V58'] = chaves
    # NEC e campos derivados precisam ser recalculados com o novo estoque inicial e novo arraste.
    out = out.drop(columns=[c for c in ['NEC_PCS','NEC_ESTRUTURA_PAI_PCS','NEC_DIRETA_BRUTA_PCS'] if c in out.columns], errors='ignore')
    return out


def _auditar_transporte_estado_v58(bd_periodo, estado_anterior, mes_ref):
    # Compara, antes do Solver do mês, o fechamento anterior com o valor efetivamente injetado no mês corrente.
    if not estado_anterior:
        return pd.DataFrame()
    atual = bd_periodo.copy()
    atual['COD_PROD'] = normalizar_codigo_serie(atual['COD_PROD'])
    atual['UNID_FAT'] = atual['UNID_FAT'].astype('string').fillna('').str.strip()
    atual['CHAVE_ESTADO_V58'] = [_chave_estado_v58(c, u) for c, u in zip(atual['COD_PROD'], atual['UNID_FAT'])]
    atual = atual.groupby('CHAVE_ESTADO_V58', as_index=False).agg(
        COD_PROD=('COD_PROD','first'), UNID_FAT=('UNID_FAT','first'),
        ESTOQUE_INICIAL_ATUAL=('LTP_EST_INI_PCS','max'), CARTEIRA_ARR_ATUAL=('LTP_CART_ARR_MES_ANT','max')
    )
    registros=[]
    for r in atual.itertuples(index=False):
        k=r.CHAVE_ESTADO_V58
        if k not in estado_anterior:
            continue
        esp=estado_anterior[k]
        dif_est=float(r.ESTOQUE_INICIAL_ATUAL)-float(esp['ESTOQUE'])
        dif_arr=float(r.CARTEIRA_ARR_ATUAL)-float(esp['ARRASTE'])
        registros.append({
            'MES_REF':pd.Timestamp(mes_ref),'COD_PROD':r.COD_PROD,'UNID_FAT':r.UNID_FAT,
            'ESTOQUE_FINAL_MES_ANT':float(esp['ESTOQUE']),'ESTOQUE_INICIAL_MES_ATUAL':float(r.ESTOQUE_INICIAL_ATUAL),
            'DIF_ESTOQUE':dif_est,'CARTEIRA_FINAL_MES_ANT':float(esp['ARRASTE']),
            'CARTEIRA_ARR_MES_ATUAL':float(r.CARTEIRA_ARR_ATUAL),'DIF_CARTEIRA':dif_arr,
            'STATUS':'OK' if abs(dif_est)<=TOL and abs(dif_arr)<=TOL else 'DIVERGENTE'
        })
    audit=pd.DataFrame(registros)
    if not audit.empty and audit['STATUS'].ne('OK').any():
        qtd=int(audit['STATUS'].ne('OK').sum())
        raise RuntimeError(f'ENCADEAMENTO_MENSAL_DIVERGENTE: {qtd} chaves COD_PROD + UNID_FAT nao reconciliaram em {pd.Timestamp(mes_ref):%Y-%m}.')
    return audit


bd_entrada_completa = pd.read_excel(INPUT_FILE, dtype={'COD_PROD': str, 'ID_PROD_UNID_FAT': str, 'UNID_PROD': str, 'UNID_FAT': str})
bd_entrada_completa['MES_REF'] = pd.to_datetime(bd_entrada_completa['MES_REF'], errors='coerce').dt.normalize()
periodos_v58 = [pd.Timestamp(x) for x in sorted(bd_entrada_completa['MES_REF'].dropna().unique().tolist())]
if not periodos_v58:
    raise RuntimeError('PROCESSAMENTO_MENSAL_SEM_PERIODOS_VALIDOS')

_saidas_v58 = {k: [] for k in [
    'resumo_recursos','resumo_ferramentas','plano_producao','nao_atend_item','auditoria_capacidade',
    'alternativas_item','rastro_rateio','diagnostico_matriz','diagnostico_matriz_rotas','diagnostico_item','diagnostico_alternativas','estrutura_explodida',
    'nec_explodida_componentes','nec_consolidada_solver','ajuste_pai_filho','resumo_pai_filho',
    'capacidade_liberada_pf','realocacao_cap_liberada','resumo_item_realocacao'
]}
_estado_itens_v58 = {}
_saldos_periodos_v58 = []
_auditoria_encadeamento_v58 = []
_tempos_periodos_v58 = []

for _idx_periodo_v58, _mes_ref_v58 in enumerate(periodos_v58):
    _bd_periodo_v58 = bd_entrada_completa.loc[bd_entrada_completa['MES_REF'].eq(_mes_ref_v58)].copy().reset_index(drop=True)
    _inicio_periodo_v58 = time.perf_counter()
    if _idx_periodo_v58 > 0:
        _bd_periodo_v58 = _aplicar_estado_anterior_v58(_bd_periodo_v58, _estado_itens_v58)
        _audit_v58 = _auditar_transporte_estado_v58(_bd_periodo_v58, _estado_itens_v58, _mes_ref_v58)
        if not _audit_v58.empty:
            _auditoria_encadeamento_v58.append(_audit_v58)

    _resultado_v58 = _executar_motor_periodo_v58(_bd_periodo_v58, aplicar_regras_industriais=False)

    # Os resumos 01/02 passam a identificar o MES_REF porque agora existem varios periodos no mesmo arquivo.
    for _nome_resumo_v58 in ['resumo_recursos','resumo_ferramentas']:
        _df_resumo_v58 = _resultado_v58[_nome_resumo_v58].copy()
        if 'MES_REF' not in _df_resumo_v58.columns:
            _df_resumo_v58.insert(0, 'MES_REF', _mes_ref_v58)
        _resultado_v58[_nome_resumo_v58] = _df_resumo_v58

    for _nome_saida_v58 in _saidas_v58:
        _df_saida_v58 = _resultado_v58.get(_nome_saida_v58)
        if isinstance(_df_saida_v58, pd.DataFrame):
            _saidas_v58[_nome_saida_v58].append(_df_saida_v58.copy())

    _saldo_v58 = _calcular_estado_proximo_periodo_v58(_bd_periodo_v58, _resultado_v58)
    _saldos_periodos_v58.append(_saldo_v58.copy())
    for _r_v58 in _saldo_v58.itertuples(index=False):
        _chave_v58 = _chave_estado_v58(_r_v58.COD_PROD, _r_v58.UNID_FAT)
        _estado_itens_v58[_chave_v58] = {
            'ESTOQUE': float(_r_v58.ESTOQUE_INICIAL_PROX_PERIODO),
            'ARRASTE': float(_r_v58.CARTEIRA_ARRASTE_PROX_PERIODO),
        }
    _tempos_periodos_v58.append({'MES_REF':_mes_ref_v58,'SEGUNDOS':time.perf_counter()-_inicio_periodo_v58})

# Itens que nao aparecem em um periodo preservam o estado ate voltarem a aparecer em um periodo posterior.
for _nome_saida_v58, _partes_v58 in _saidas_v58.items():
    globals()[_nome_saida_v58] = pd.concat(_partes_v58, ignore_index=True, sort=False) if _partes_v58 else pd.DataFrame()

# Recorte auxiliar do primeiro mes para validacao da ponte, sem exportacao para o arquivo do cliente.
if isinstance(diagnostico_matriz, pd.DataFrame) and not diagnostico_matriz.empty and 'MES_REF' in diagnostico_matriz.columns:
    _mes_diag_m1 = pd.to_datetime(diagnostico_matriz['MES_REF'], errors='coerce').min()
    diagnostico_matriz_m1 = diagnostico_matriz[pd.to_datetime(diagnostico_matriz['MES_REF'], errors='coerce').eq(_mes_diag_m1)].copy()
    diagnostico_matriz_rotas_m1 = diagnostico_matriz_rotas[pd.to_datetime(diagnostico_matriz_rotas['MES_REF'], errors='coerce').eq(_mes_diag_m1)].copy() if isinstance(diagnostico_matriz_rotas, pd.DataFrame) and not diagnostico_matriz_rotas.empty else pd.DataFrame()
    _resumo_status_diag = diagnostico_matriz_m1['STATUS_DIAGNOSTICO'].value_counts(dropna=False).to_dict()
    print('       DIAGNOSTICO MATRIZ M1 - itens pendentes:', len(diagnostico_matriz_m1), '| status:', _resumo_status_diag)
else:
    diagnostico_matriz_m1 = pd.DataFrame()
    diagnostico_matriz_rotas_m1 = pd.DataFrame()

saldo_encadeamento_v58 = pd.concat(_saldos_periodos_v58, ignore_index=True, sort=False) if _saldos_periodos_v58 else pd.DataFrame()
auditoria_encadeamento_v58 = pd.concat(_auditoria_encadeamento_v58, ignore_index=True, sort=False) if _auditoria_encadeamento_v58 else pd.DataFrame()
print('       OK - Encadeamento mensal reconciliado por COD_PROD + UNID_FAT')

_finalizar_etapa_v58("SOLVER_ENCADEAMENTO")
print("       OK - Solver processado e regras operacionais validadas")


# 03A. Validação do CORTE_HR por restrição — fotografia bruta do V66


In [ ]:
# ============================================================
# 04B. VALIDACAO DO CORTE_HR POR RESTRICAO - M1
# ============================================================
# Etapa somente diagnostica.
# Nao altera NEC, nao aplica a Matriz e nao modifica nenhuma guia exportada.
#
# IMPORTANTE:
# Esta validacao usa SOMENTE a fotografia bruta do Solver V66:
# - rastro_rateio;
# - capacidade residual final registrada pela ponte.
#
# Nao usa resumo_recursos, pois esse objeto e recalculado depois de ajustes
# operacionais/finais do plano e representa outra etapa do processo.


def validar_corte_hr_por_restricao_m1(
    diagnostico_m1,
    rotas_m1,
    rastro,
):
    cols_saida = [
        'MES_REF','TIPO_RESTRICAO_CANDIDATA','RESTRICAO_CANDIDATA',
        'QTD_ITENS','QTD_ITENS_RASTRO','QTD_ROTAS_FINAIS_RASTRO',
        'HOR_REC_RAW_RASTRO','HR_PRODUZIR_RAW_RASTRO',
        'CAP_REC_FINAL_RECONSTRUIDA','MAX_CAP_REC_FINAL',
        'DIF_CAP_FINAL_RECON_VS_SOLVER',
        'HR_FALTA_RASTRO','HR_DEMANDA_EFETIVA_RECONSTRUIDA',
        'EXCESSO_HR_RECONSTRUIDO','CORTE_HR_CANDIDATO',
        'DIF_CORTE_VS_RASTRO','DIF_CORTE_VS_EXCESSO',
        'MIN_CAP_FER_FINAL',
        'VALIDA_QTD_ITENS','VALIDA_ROTA_UNICA_FINAL',
        'VALIDA_CAP_FINAL_RECONSTRUIDA','VALIDA_RECURSO_ESGOTADO',
        'VALIDA_FERRAMENTA_NAO_LIMITANTE',
        'VALIDA_CORTE_VS_RASTRO','VALIDA_CORTE_VS_EXCESSO',
        'STATUS_VALIDACAO'
    ]

    if not isinstance(diagnostico_m1, pd.DataFrame) or diagnostico_m1.empty:
        return pd.DataFrame(columns=cols_saida)

    diag = diagnostico_m1.copy()
    diag = diag[
        diag['STATUS_DIAGNOSTICO'].eq('CORTE_CANDIDATO_OBJETIVO')
        & diag['TIPO_RESTRICAO_CANDIDATA'].eq('REC')
    ].copy()
    if diag.empty:
        return pd.DataFrame(columns=cols_saida)

    mes_ref = pd.to_datetime(diag['MES_REF'], errors='coerce').min()
    tol_hr = max(TOL * 100, 1e-6)

    # 1) Consolidacao candidata produzida pela ponte.
    cand = (
        diag.groupby(['TIPO_RESTRICAO_CANDIDATA','RESTRICAO_CANDIDATA'], as_index=False)
        .agg(
            QTD_ITENS=('ID_PROD_UNID_FAT','nunique'),
            CORTE_HR_CANDIDATO=('CORTE_HR_CANDIDATO','sum'),
        )
    )
    cand.insert(0, 'MES_REF', mes_ref)

    ids = set(diag['ID_PROD_UNID_FAT'].astype(str))

    # 2) Falta final dos itens candidatos, diretamente da ultima rodada do rastro bruto.
    ras = rastro.copy()
    ras['MES_REF'] = pd.to_datetime(ras['MES_REF'], errors='coerce')
    ras_m1 = ras[ras['MES_REF'].eq(mes_ref)].copy()
    if ras_m1.empty:
        raise RuntimeError('VALIDACAO_CORTE_M1_SEM_RASTRO')

    ras_cand = ras_m1[ras_m1['ID_PROD_UNID_FAT'].astype(str).isin(ids)].copy()
    ras_cand['_RODADA_FINAL_ITEM'] = ras_cand.groupby('ID_PROD_UNID_FAT')['RODADA'].transform('max')
    ras_cand = ras_cand[ras_cand['RODADA'].eq(ras_cand['_RODADA_FINAL_ITEM'])].copy()
    ras_cand['QTD_FALTA_RASTRO'] = (
        to_num(ras_cand['QTD_SOLICITADA']) - to_num(ras_cand['QTD_PRODUZIR_SOLVER'])
    ).clip(lower=0.0)
    ras_cand['HR_FALTA_RASTRO'] = np.divide(
        ras_cand['QTD_FALTA_RASTRO'],
        to_num(ras_cand['PCS_HORA']),
        out=np.zeros(len(ras_cand), dtype=float),
        where=to_num(ras_cand['PCS_HORA']).to_numpy() > TOL,
    )
    ras_cand = ras_cand[ras_cand['QTD_FALTA_RASTRO'] > TOL].copy()

    ras_rec = (
        ras_cand.groupby('ALOC_REC', as_index=False)
        .agg(
            QTD_ITENS_RASTRO=('ID_PROD_UNID_FAT','nunique'),
            QTD_ROTAS_FINAIS_RASTRO=('ID_PROD_UNID_FAT','size'),
            HR_FALTA_RASTRO=('HR_FALTA_RASTRO','sum'),
        )
        .rename(columns={'ALOC_REC':'RESTRICAO_CANDIDATA'})
    )

    # 3) Reconstrucao da capacidade bruta do recurso usando TODO o rastro do M1.
    # CAP_RESTANTE_ALOC_REC_ANTES e monotonicamente decrescente por recurso.
    # Logo, o maior valor observado e a capacidade inicial disponivel daquele recurso.
    # A soma de HR_PRODUZIR_SOLVER no rastro e o consumo bruto real feito pelo V66.
    ras_m1['CAP_RESTANTE_ALOC_REC_ANTES'] = to_num(ras_m1['CAP_RESTANTE_ALOC_REC_ANTES'])
    ras_m1['HR_PRODUZIR_SOLVER'] = to_num(ras_m1['HR_PRODUZIR_SOLVER'])
    rec_raw = (
        ras_m1.groupby('ALOC_REC', as_index=False)
        .agg(
            HOR_REC_RAW_RASTRO=('CAP_RESTANTE_ALOC_REC_ANTES','max'),
            HR_PRODUZIR_RAW_RASTRO=('HR_PRODUZIR_SOLVER','sum'),
        )
        .rename(columns={'ALOC_REC':'RESTRICAO_CANDIDATA'})
    )
    rec_raw['CAP_REC_FINAL_RECONSTRUIDA'] = (
        rec_raw['HOR_REC_RAW_RASTRO'] - rec_raw['HR_PRODUZIR_RAW_RASTRO']
    ).clip(lower=0.0)

    # 4) Estado residual final gravado pela propria ponte no momento em que o V66 termina.
    rot = rotas_m1.copy()
    rot['MES_REF'] = pd.to_datetime(rot['MES_REF'], errors='coerce')
    rot = rot[
        rot['MES_REF'].eq(mes_ref)
        & rot['ID_PROD_UNID_FAT'].astype(str).isin(ids)
        & rot['FOI_ULTIMA_RODADA'].fillna(False)
        & (to_num(rot['QTD_FALTA_ROTA']) > TOL)
    ].copy()
    rot_estado = (
        rot.groupby('ALOC_REC', as_index=False)
        .agg(
            MIN_CAP_FER_FINAL=('CAP_RESTANTE_COD_FER_FINAL','min'),
            MAX_CAP_REC_FINAL=('CAP_RESTANTE_ALOC_REC_FINAL','max'),
        )
        .rename(columns={'ALOC_REC':'RESTRICAO_CANDIDATA'})
    )

    val = (
        cand
        .merge(ras_rec, on='RESTRICAO_CANDIDATA', how='left', validate='one_to_one')
        .merge(rec_raw, on='RESTRICAO_CANDIDATA', how='left', validate='one_to_one')
        .merge(rot_estado, on='RESTRICAO_CANDIDATA', how='left', validate='one_to_one')
    )

    for c in [
        'QTD_ITENS_RASTRO','QTD_ROTAS_FINAIS_RASTRO','HR_FALTA_RASTRO',
        'HOR_REC_RAW_RASTRO','HR_PRODUZIR_RAW_RASTRO','CAP_REC_FINAL_RECONSTRUIDA',
        'MIN_CAP_FER_FINAL','MAX_CAP_REC_FINAL'
    ]:
        val[c] = to_num(val[c])

    val['DIF_CAP_FINAL_RECON_VS_SOLVER'] = (
        val['CAP_REC_FINAL_RECONSTRUIDA'] - val['MAX_CAP_REC_FINAL']
    )
    val['HR_DEMANDA_EFETIVA_RECONSTRUIDA'] = (
        val['HR_PRODUZIR_RAW_RASTRO'] + val['HR_FALTA_RASTRO']
    )
    val['EXCESSO_HR_RECONSTRUIDO'] = (
        val['HR_DEMANDA_EFETIVA_RECONSTRUIDA'] - val['HOR_REC_RAW_RASTRO']
    ).clip(lower=0.0)
    val['DIF_CORTE_VS_RASTRO'] = val['CORTE_HR_CANDIDATO'] - val['HR_FALTA_RASTRO']
    val['DIF_CORTE_VS_EXCESSO'] = val['CORTE_HR_CANDIDATO'] - val['EXCESSO_HR_RECONSTRUIDO']

    val['VALIDA_QTD_ITENS'] = val['QTD_ITENS'].eq(val['QTD_ITENS_RASTRO'])
    val['VALIDA_ROTA_UNICA_FINAL'] = val['QTD_ROTAS_FINAIS_RASTRO'].eq(val['QTD_ITENS_RASTRO'])
    val['VALIDA_CAP_FINAL_RECONSTRUIDA'] = val['DIF_CAP_FINAL_RECON_VS_SOLVER'].abs() <= tol_hr
    val['VALIDA_RECURSO_ESGOTADO'] = val['MAX_CAP_REC_FINAL'].abs() <= tol_hr
    val['VALIDA_FERRAMENTA_NAO_LIMITANTE'] = val['MIN_CAP_FER_FINAL'] > TOL
    val['VALIDA_CORTE_VS_RASTRO'] = val['DIF_CORTE_VS_RASTRO'].abs() <= tol_hr
    val['VALIDA_CORTE_VS_EXCESSO'] = val['DIF_CORTE_VS_EXCESSO'].abs() <= tol_hr

    checks = [
        'VALIDA_QTD_ITENS','VALIDA_ROTA_UNICA_FINAL',
        'VALIDA_CAP_FINAL_RECONSTRUIDA','VALIDA_RECURSO_ESGOTADO',
        'VALIDA_FERRAMENTA_NAO_LIMITANTE',
        'VALIDA_CORTE_VS_RASTRO','VALIDA_CORTE_VS_EXCESSO'
    ]
    val['STATUS_VALIDACAO'] = np.where(val[checks].all(axis=1), 'PASSOU', 'FALHOU')

    val = val.sort_values('CORTE_HR_CANDIDATO', ascending=False).reset_index(drop=True)
    return val[[c for c in cols_saida if c in val.columns]]


validacao_corte_restricao_m1 = validar_corte_hr_por_restricao_m1(
    diagnostico_matriz_m1,
    diagnostico_matriz_rotas_m1,
    rastro_rateio,
)

if validacao_corte_restricao_m1.empty:
    print('       VALIDACAO CORTE M1 RAW - sem candidatos REC objetivos para validar')
else:
    _qtd_restr = len(validacao_corte_restricao_m1)
    _qtd_passou = int(validacao_corte_restricao_m1['STATUS_VALIDACAO'].eq('PASSOU').sum())
    _qtd_falhou = int(validacao_corte_restricao_m1['STATUS_VALIDACAO'].eq('FALHOU').sum())
    _total_corte = float(validacao_corte_restricao_m1['CORTE_HR_CANDIDATO'].sum())
    _total_excesso = float(validacao_corte_restricao_m1['EXCESSO_HR_RECONSTRUIDO'].sum())
    _dif_max = float(validacao_corte_restricao_m1['DIF_CORTE_VS_EXCESSO'].abs().max())
    _dif_cap_max = float(validacao_corte_restricao_m1['DIF_CAP_FINAL_RECON_VS_SOLVER'].abs().max())

    print(
        '       VALIDACAO CORTE M1 RAW - restricoes:', _qtd_restr,
        '| passou:', _qtd_passou,
        '| falhou:', _qtd_falhou,
        '| corte_hr:', round(_total_corte, 6),
        '| excesso_reconstruido:', round(_total_excesso, 6),
        '| dif_max:', round(_dif_max, 9),
        '| dif_cap_max:', round(_dif_cap_max, 9),
    )

    if _qtd_falhou > 0:
        print('       ATENCAO - existem restricoes que ainda nao podem seguir para a Matriz de Cortes')
        display(validacao_corte_restricao_m1.loc[
            validacao_corte_restricao_m1['STATUS_VALIDACAO'].eq('FALHOU')
        ])


# 03B. Diagnóstico DE ONDE CORTAR — Matriz dos 9 critérios (v66_12: pool completo da REC)


In [ ]:
# ============================================================
# 04C. DIAGNOSTICO DE ONDE CORTAR - M1
# ============================================================
# Etapa somente diagnostica.
# Usa o CORTE_HR ja validado e identifica quais componentes da demanda
# responderiam pelo corte segundo a ordem fixa da Matriz.
# Nao altera NEC, nao altera a base de entrada e nao modifica o Excel final.

CRITERIOS_MATRIZ_CORTE = [
    (1, 'SIM', 'EST_SEG', 'LTP_EST_SEG_PCS'),
    (2, 'NAO', 'PREV_PROX', 'LTP_SALDO_PREV_PROX_MES_PCS'),
    (3, 'NAO', 'EST_SEG', 'LTP_EST_SEG_PCS'),
    (4, 'SIM', 'PREV_VEN', 'LTP_SALDO_PREV_PCS'),
    (5, 'NAO', 'PREV_VEN', 'LTP_SALDO_PREV_PCS'),
    (6, 'SIM', 'C_AT', 'LTP_CART_MES_ATUAL'),
    (7, 'NAO', 'C_AT', 'LTP_CART_MES_ATUAL'),
    (8, 'SIM', 'C_ARR', 'LTP_CART_ARR_MES_ANT'),
    (9, 'NAO', 'C_ARR', 'LTP_CART_ARR_MES_ANT'),
]

_MAPA_CRITERIO_MATRIZ = {
    (mesma_reg, componente): (ordem, coluna_ltp)
    for ordem, mesma_reg, componente, coluna_ltp in CRITERIOS_MATRIZ_CORTE
}


def _decompor_componentes_cortaveis_m1(bd_m1, nec_consolidada_m1):
    """Decompoe a demanda direta liquida nos cinco componentes cortaveis."""
    cols_num = [
        'LTP_CART_ARR_MES_ANT', 'LTP_CART_MES_ATUAL', 'LTP_SALDO_PREV_PCS',
        'LTP_SALDO_PREV_PROX_MES_PCS', 'LTP_EST_SEG_PCS',
        'LTP_EST_INI_PCS', 'LTP_EST_TRANS_PCS', 'TRIANG_TOT_PCS', 'ORI_TOT_PCS',
    ]
    base = bd_m1.copy()
    base['MES_REF'] = pd.to_datetime(base['MES_REF'], errors='coerce').dt.normalize()
    base['ID_PROD_UNID_FAT'] = base['ID_PROD_UNID_FAT'].astype(str)
    base['COD_PROD'] = normalizar_codigo_serie(base['COD_PROD'])
    for c in ['UNID_PROD', 'UNID_FAT']:
        if c not in base.columns:
            base[c] = ''
        base[c] = base[c].astype('string').fillna('').str.strip()
    if 'MESMA_REG' not in base.columns:
        base['MESMA_REG'] = 'SIM'
    base['MESMA_REG'] = base['MESMA_REG'].astype('string').fillna('SIM').str.strip().str.upper()
    base['MESMA_REG'] = base['MESMA_REG'].where(base['MESMA_REG'].isin(['SIM', 'NAO']), 'SIM')
    for c in cols_num:
        if c not in base.columns:
            base[c] = 0.0
        base[c] = to_num(base[c])

    agg = {c: 'max' for c in cols_num}
    agg.update({c: 'first' for c in ['COD_PROD', 'DESC_PROD', 'UNID_PROD', 'UNID_FAT', 'TIPO_PROD', 'MESMA_REG'] if c in base.columns})
    item = base.groupby(['MES_REF', 'ID_PROD_UNID_FAT'], as_index=False).agg(agg)

    nec = nec_consolidada_m1.copy() if isinstance(nec_consolidada_m1, pd.DataFrame) else pd.DataFrame()
    if not nec.empty:
        nec['MES_REF'] = pd.to_datetime(nec['MES_REF'], errors='coerce').dt.normalize()
        nec['ID_PROD_UNID_FAT'] = nec['ID_PROD_UNID_FAT'].astype(str)
        if 'NEC_DIRETA_PCS' not in nec.columns:
            nec['NEC_DIRETA_PCS'] = 0.0
        nec['NEC_DIRETA_PCS'] = to_num(nec['NEC_DIRETA_PCS'])
        nec_dir = nec.groupby(['MES_REF', 'ID_PROD_UNID_FAT'], as_index=False).agg(NEC_DIRETA_PCS=('NEC_DIRETA_PCS', 'max'))
        item = item.merge(nec_dir, on=['MES_REF', 'ID_PROD_UNID_FAT'], how='left')
    else:
        item['NEC_DIRETA_PCS'] = 0.0
    item['NEC_DIRETA_PCS'] = to_num(item['NEC_DIRETA_PCS'])

    et = (
        item['LTP_EST_INI_PCS'] + item['LTP_EST_TRANS_PCS']
        + item['TRIANG_TOT_PCS'] + item['ORI_TOT_PCS']
    )
    ativo = item['NEC_DIRETA_PCS'] > TOL

    item['C_ARR_PCS'] = np.where(
        ativo,
        (item['LTP_CART_ARR_MES_ANT'] - et).clip(lower=0.0),
        0.0,
    )
    item['C_AT_PCS'] = np.where(
        ativo,
        (
            (item['LTP_CART_ARR_MES_ANT'] + item['LTP_CART_MES_ATUAL'] - et).clip(lower=0.0)
            - item['C_ARR_PCS']
        ).clip(lower=0.0),
        0.0,
    )
    item['PREV_VEN_PCS'] = np.where(
        ativo,
        (
            (
                item['LTP_CART_ARR_MES_ANT'] + item['LTP_CART_MES_ATUAL']
                + item['LTP_SALDO_PREV_PCS'] - et
            ).clip(lower=0.0)
            - item['C_ARR_PCS'] - item['C_AT_PCS']
        ).clip(lower=0.0),
        0.0,
    )
    item['PREV_PROX_PCS'] = np.where(
        ativo & item['MESMA_REG'].eq('NAO'),
        (
            (
                item['LTP_CART_ARR_MES_ANT'] + item['LTP_CART_MES_ATUAL']
                + item['LTP_SALDO_PREV_PCS'] + item['LTP_SALDO_PREV_PROX_MES_PCS'] - et
            ).clip(lower=0.0)
            - item['C_ARR_PCS'] - item['C_AT_PCS'] - item['PREV_VEN_PCS']
        ).clip(lower=0.0),
        0.0,
    )
    demanda_com_es = (
        item['LTP_CART_ARR_MES_ANT'] + item['LTP_CART_MES_ATUAL'] + item['LTP_SALDO_PREV_PCS']
        + np.where(item['MESMA_REG'].eq('NAO'), item['LTP_SALDO_PREV_PROX_MES_PCS'], 0.0)
        + item['LTP_EST_SEG_PCS'] - et
    )
    item['EST_SEG_PCS'] = np.where(
        ativo,
        (
            pd.Series(demanda_com_es, index=item.index).clip(lower=0.0)
            - item['C_ARR_PCS'] - item['C_AT_PCS'] - item['PREV_VEN_PCS'] - item['PREV_PROX_PCS']
        ).clip(lower=0.0),
        0.0,
    )

    componentes = {
        'C_ARR': 'C_ARR_PCS',
        'C_AT': 'C_AT_PCS',
        'PREV_VEN': 'PREV_VEN_PCS',
        'PREV_PROX': 'PREV_PROX_PCS',
        'EST_SEG': 'EST_SEG_PCS',
    }
    registros = []
    meta = [c for c in ['MES_REF', 'ID_PROD_UNID_FAT', 'COD_PROD', 'DESC_PROD', 'UNID_PROD', 'UNID_FAT', 'TIPO_PROD', 'MESMA_REG'] if c in item.columns]
    for componente, col_pcs in componentes.items():
        tmp = item[meta].copy()
        tmp['COMPONENTE'] = componente
        tmp['PCS_DISP_ORIGEM'] = to_num(item[col_pcs])
        registros.append(tmp)
    out = pd.concat(registros, ignore_index=True) if registros else pd.DataFrame()
    if out.empty:
        return out

    crit = out.apply(
        lambda r: _MAPA_CRITERIO_MATRIZ.get((str(r['MESMA_REG']), str(r['COMPONENTE'])), (np.nan, '')),
        axis=1,
        result_type='expand',
    )
    out['CRITERIO_ORDEM'] = pd.to_numeric(crit[0], errors='coerce')
    out['COLUNA_LTP_CORTE'] = crit[1].fillna('')
    out['CRITERIO'] = out['MESMA_REG'].astype(str) + '|' + out['COMPONENTE'].astype(str)
    out = out[out['CRITERIO_ORDEM'].notna() & (out['PCS_DISP_ORIGEM'] > TOL)].copy()
    out['CRITERIO_ORDEM'] = out['CRITERIO_ORDEM'].astype(int)
    out = out.rename(columns={'ID_PROD_UNID_FAT': 'ID_PROD_UNID_FAT_ORIGEM'})
    return out


def _calcular_fator_estrutural_ativo_m1(estrutura_m1):
    """Calcula o fator acumulado ao longo das trilhas realmente ativas da explosao do M1."""
    cols_saida = [
        'MES_REF', 'ID_PROD_UNID_FAT_ORIGEM', 'COD_PROD_ORIGEM',
        'COD_INSUMO', 'UNID_PROD_RETORNO', 'FATOR_ESTRUTURAL'
    ]
    if not isinstance(estrutura_m1, pd.DataFrame) or estrutura_m1.empty:
        return pd.DataFrame(columns=cols_saida)

    est = estrutura_m1.copy()
    est['MES_REF'] = pd.to_datetime(est['MES_REF'], errors='coerce').dt.normalize()
    est['ID_PROD_UNID_FAT_ORIGEM'] = est['ID_PROD_UNID_FAT_ORIGEM'].astype(str)
    est['COD_PROD_ORIGEM'] = normalizar_codigo_serie(est['COD_PROD_ORIGEM'])
    est['COD_INSUMO'] = normalizar_codigo_serie(est['COD_INSUMO'])
    est['UNID_PROD_RETORNO'] = est['UNID_PROD'].astype('string').fillna('').str.strip()
    est['QTD_UTIL_PCS'] = to_num(est['QTD_UTIL_PCS'])
    est['NIVEL'] = pd.to_numeric(est['NIVEL'], errors='coerce').fillna(0).astype(int)
    est['TRILHA'] = est['TRILHA'].astype('string').fillna('').str.strip()

    fatores = np.zeros(len(est), dtype=float)
    for _, idxs in est.groupby(['MES_REF', 'ID_PROD_UNID_FAT_ORIGEM'], sort=False).groups.items():
        idxs = list(idxs)
        idxs.sort(key=lambda i: (int(est.at[i, 'NIVEL']), str(est.at[i, 'TRILHA'])))
        por_trilha = {}
        for i in idxs:
            trilha = str(est.at[i, 'TRILHA'])
            qtd = float(est.at[i, 'QTD_UTIL_PCS'])
            nivel = int(est.at[i, 'NIVEL'])
            if qtd <= TOL:
                fator = 0.0
            elif nivel <= 1 or ' -> ' not in trilha:
                fator = qtd
            else:
                trilha_pai = trilha.rsplit(' -> ', 1)[0]
                fator_pai = float(por_trilha.get(trilha_pai, np.nan))
                fator = fator_pai * qtd if np.isfinite(fator_pai) else np.nan
            fatores[est.index.get_loc(i) if not isinstance(est.index, pd.RangeIndex) else i] = fator
            por_trilha[trilha] = fator

    # Atribuicao robusta mesmo se o indice original nao for RangeIndex.
    est = est.reset_index(drop=True)
    if len(fatores) != len(est):
        fatores = np.resize(fatores, len(est))
    est['FATOR_ESTRUTURAL'] = fatores
    est = est[np.isfinite(est['FATOR_ESTRUTURAL']) & (est['FATOR_ESTRUTURAL'] > TOL)].copy()

    out = (
        est.groupby(
            ['MES_REF', 'ID_PROD_UNID_FAT_ORIGEM', 'COD_PROD_ORIGEM', 'COD_INSUMO', 'UNID_PROD_RETORNO'],
            as_index=False,
        )
        .agg(FATOR_ESTRUTURAL=('FATOR_ESTRUTURAL', 'sum'))
    )
    return out[cols_saida]


def gerar_de_onde_cortar_m1(
    validacao_corte,
    diagnostico_m1,
    rastro_m1,
    bd_entrada,
    nec_consolidada,
    estrutura_exp,
):
    """Monta o pool de impacto e distribui o CORTE_HR pelos 9 criterios da Matriz.

    CORRECAO v66_12:
    - QUANTO CORTAR continua vindo da pendencia final ja validada da V66;
    - DE ONDE CORTAR usa todos os itens que efetivamente disputaram as RECs alvo no M1,
      independentemente de terem terminado atendidos ou pendentes na ultima rodada.
    """
    cols_corte = [
        'MES_REF', 'ALOC_REC', 'CRITERIO_ORDEM', 'CRITERIO', 'MESMA_REG', 'COMPONENTE',
        'ID_PROD_UNID_FAT_ORIGEM', 'COD_PROD_ORIGEM', 'DESC_PROD_ORIGEM',
        'UNID_PROD_ORIGEM', 'UNID_FAT_ORIGEM', 'TIPO_PROD_ORIGEM',
        'TIPO_IMPACTO', 'ITENS_CAPACIDADE_AFETADOS', 'QTD_ITENS_CAPACIDADE_AFETADOS',
        'PCS_DISP_ORIGEM', 'HR_POR_PCA_ORIGEM_NO_REC', 'HR_DISP_ORIGEM',
        'CORTE_HR_ALVO', 'SALDO_CORTE_HR_ANTES', 'HR_CORTE',
        'PERC_CORTE_NO_COMPONENTE', 'PCS_CORTE', 'COLUNA_LTP_CORTE',
        'SALDO_CORTE_HR_DEPOIS', 'TIPO_APLICACAO_CRITERIO'
    ]
    cols_valid = [
        'MES_REF', 'ALOC_REC', 'CORTE_HR_ALVO', 'HR_POOL_CORTAVEL',
        'HR_CORTE_ATRIBUIDO', 'DIF_HR_CORTE', 'SALDO_FINAL_HR',
        'QTD_LINHAS_CORTE', 'QTD_CRITERIOS_USADOS', 'ULTIMO_CRITERIO_USADO',
        'STATUS_VALIDACAO'
    ]
    if not isinstance(validacao_corte, pd.DataFrame) or validacao_corte.empty:
        return pd.DataFrame(columns=cols_corte), pd.DataFrame(columns=cols_valid), pd.DataFrame()

    val = validacao_corte.copy()
    val = val[
        val['STATUS_VALIDACAO'].eq('PASSOU')
        & val['TIPO_RESTRICAO_CANDIDATA'].eq('REC')
        & (to_num(val['CORTE_HR_CANDIDATO']) > TOL)
    ].copy()
    if val.empty:
        return pd.DataFrame(columns=cols_corte), pd.DataFrame(columns=cols_valid), pd.DataFrame()
    val['MES_REF'] = pd.to_datetime(val['MES_REF'], errors='coerce').dt.normalize()
    mat_cortes = val[['MES_REF', 'RESTRICAO_CANDIDATA', 'CORTE_HR_CANDIDATO']].copy()
    mat_cortes = mat_cortes.rename(columns={'RESTRICAO_CANDIDATA': 'ALOC_REC', 'CORTE_HR_CANDIDATO': 'CORTE_HR'})
    mat_cortes['ALOC_REC'] = mat_cortes['ALOC_REC'].astype(str)
    mat_cortes['CORTE_HR'] = to_num(mat_cortes['CORTE_HR'])

    mes_ref = mat_cortes['MES_REF'].min()

    # ============================================================
    # CORRECAO v66_12 - UNIVERSO DE PARTICIPANTES DA REC
    # ============================================================
    # A pendencia final serve exclusivamente para definir QUANTO CORTAR.
    # Para definir DE ONDE CORTAR, a Matriz precisa enxergar todos os itens
    # que efetivamente disputaram cada ALOC_REC alvo durante o M1.
    #
    # Portanto, NAO filtrar por:
    # - FOI_ULTIMA_RODADA;
    # - QTD_FALTA_ROTA > 0;
    # - itens presentes somente no diagnostico final de pendencia.
    #
    # O rastro bruto da V66 e usado apenas para identificar quem participou
    # de fato da disputa da REC. A regra dos 9 criterios e o CORTE_HR nao mudam.
    ras = rastro_m1.copy() if isinstance(rastro_m1, pd.DataFrame) else pd.DataFrame()
    if ras.empty:
        return pd.DataFrame(columns=cols_corte), pd.DataFrame(columns=cols_valid), pd.DataFrame()

    ras['MES_REF'] = pd.to_datetime(ras['MES_REF'], errors='coerce').dt.normalize()
    ras['ID_PROD_UNID_FAT'] = ras['ID_PROD_UNID_FAT'].astype(str)
    ras['ALOC_REC'] = ras['ALOC_REC'].astype(str)
    ras['PCS_HORA'] = to_num(ras['PCS_HORA'])
    for _c in ['QTD_SOLICITADA', 'HR_SOLICITADA']:
        if _c not in ras.columns:
            ras[_c] = 0.0
        ras[_c] = to_num(ras[_c])

    ras = ras[
        ras['MES_REF'].eq(mes_ref)
        & ras['ALOC_REC'].isin(set(mat_cortes['ALOC_REC']))
        & (ras['PCS_HORA'] > TOL)
        & ((ras['QTD_SOLICITADA'] > TOL) | (ras['HR_SOLICITADA'] > TOL))
    ].copy()

    # Uma mesma rota pode aparecer em varias rodadas. Para o mapa de impacto,
    # cada item participa uma unica vez por REC; preserva-se a rota de maior prioridade.
    ras = ras.sort_values(['ID_PROD_UNID_FAT', 'ALOC_REC', 'PRIOR_MATPAR', 'PRIOR_ROT']).drop_duplicates(
        ['ID_PROD_UNID_FAT', 'ALOC_REC'], keep='first'
    )

    alvos = ras[[
        'MES_REF', 'ID_PROD_UNID_FAT', 'COD_PROD', 'UNID_PROD', 'UNID_FAT',
        'ALOC_REC', 'PCS_HORA'
    ]].copy()
    alvos = alvos.rename(columns={
        'ID_PROD_UNID_FAT': 'ID_ITEM_CAPACIDADE',
        'COD_PROD': 'COD_PROD_CAPACIDADE',
        'UNID_PROD': 'UNID_PROD_CAPACIDADE',
        'UNID_FAT': 'UNID_FAT_CAPACIDADE',
        'PCS_HORA': 'PCS_HORA_CAPACIDADE',
    })
    alvos['COD_PROD_CAPACIDADE'] = normalizar_codigo_serie(alvos['COD_PROD_CAPACIDADE'])
    alvos['UNID_FAT_CAPACIDADE'] = alvos['UNID_FAT_CAPACIDADE'].astype('string').fillna('').str.strip()

    bd_m1 = bd_entrada.copy()
    bd_m1['MES_REF'] = pd.to_datetime(bd_m1['MES_REF'], errors='coerce').dt.normalize()
    bd_m1 = bd_m1[bd_m1['MES_REF'].eq(mes_ref)].copy()
    nec_m1 = nec_consolidada.copy() if isinstance(nec_consolidada, pd.DataFrame) else pd.DataFrame()
    if not nec_m1.empty:
        nec_m1['MES_REF'] = pd.to_datetime(nec_m1['MES_REF'], errors='coerce').dt.normalize()
        nec_m1 = nec_m1[nec_m1['MES_REF'].eq(mes_ref)].copy()
    componentes = _decompor_componentes_cortaveis_m1(bd_m1, nec_m1)

    est_m1 = estrutura_exp.copy() if isinstance(estrutura_exp, pd.DataFrame) else pd.DataFrame()
    if not est_m1.empty:
        est_m1['MES_REF'] = pd.to_datetime(est_m1['MES_REF'], errors='coerce').dt.normalize()
        est_m1 = est_m1[est_m1['MES_REF'].eq(mes_ref)].copy()
    fatores = _calcular_fator_estrutural_ativo_m1(est_m1)

    # Impacto direto: uma peca cortada no proprio item reduz 1 / PCS_HORA da rota limitada.
    impacto_direto = alvos[[
        'MES_REF', 'ALOC_REC', 'ID_ITEM_CAPACIDADE', 'COD_PROD_CAPACIDADE',
        'UNID_FAT_CAPACIDADE', 'PCS_HORA_CAPACIDADE'
    ]].copy()
    impacto_direto['ID_PROD_UNID_FAT_ORIGEM'] = impacto_direto['ID_ITEM_CAPACIDADE']
    impacto_direto['FATOR_ESTRUTURAL'] = 1.0
    impacto_direto['TIPO_IMPACTO_PARCELA'] = 'DIRETO'

    # Impacto estrutural: uma peca do PA gera FATOR_ESTRUTURAL pecas do item limitado.
    impacto_estrut = pd.DataFrame()
    if not fatores.empty and not alvos.empty:
        impacto_estrut = fatores.merge(
            alvos,
            left_on=['MES_REF', 'COD_INSUMO', 'UNID_PROD_RETORNO'],
            right_on=['MES_REF', 'COD_PROD_CAPACIDADE', 'UNID_FAT_CAPACIDADE'],
            how='inner',
        )
        if not impacto_estrut.empty:
            impacto_estrut['TIPO_IMPACTO_PARCELA'] = 'ESTRUTURAL'
            impacto_estrut = impacto_estrut[[
                'MES_REF', 'ALOC_REC', 'ID_ITEM_CAPACIDADE', 'COD_PROD_CAPACIDADE',
                'UNID_FAT_CAPACIDADE', 'PCS_HORA_CAPACIDADE',
                'ID_PROD_UNID_FAT_ORIGEM', 'FATOR_ESTRUTURAL', 'TIPO_IMPACTO_PARCELA'
            ]].copy()

    impactos = pd.concat([impacto_direto, impacto_estrut], ignore_index=True, sort=False)
    impactos['PCS_HORA_CAPACIDADE'] = to_num(impactos['PCS_HORA_CAPACIDADE'])
    impactos['FATOR_ESTRUTURAL'] = to_num(impactos['FATOR_ESTRUTURAL'])
    impactos = impactos[(impactos['PCS_HORA_CAPACIDADE'] > TOL) & (impactos['FATOR_ESTRUTURAL'] > TOL)].copy()
    impactos['HR_POR_PCA_PARCELA'] = impactos['FATOR_ESTRUTURAL'] / impactos['PCS_HORA_CAPACIDADE']

    # Uma origem pode afetar mais de um item do mesmo REC. O corte na origem acontece uma unica vez;
    # por isso os impactos em horas sao somados antes de disponibilizar as pecas do componente.
    def _tipos_impacto(s):
        tipos = set(s.astype(str))
        if {'DIRETO', 'ESTRUTURAL'}.issubset(tipos):
            return 'DIRETO_E_ESTRUTURAL'
        return 'ESTRUTURAL' if 'ESTRUTURAL' in tipos else 'DIRETO'

    impacto_origem = (
        impactos.groupby(['MES_REF', 'ALOC_REC', 'ID_PROD_UNID_FAT_ORIGEM'], as_index=False)
        .agg(
            HR_POR_PCA_ORIGEM_NO_REC=('HR_POR_PCA_PARCELA', 'sum'),
            TIPO_IMPACTO=('TIPO_IMPACTO_PARCELA', _tipos_impacto),
            ITENS_CAPACIDADE_AFETADOS=('ID_ITEM_CAPACIDADE', lambda s: ';'.join(sorted(set(map(str, s))))),
            QTD_ITENS_CAPACIDADE_AFETADOS=('ID_ITEM_CAPACIDADE', 'nunique'),
        )
    )

    pool = componentes.merge(impacto_origem, on=['MES_REF', 'ID_PROD_UNID_FAT_ORIGEM'], how='inner')
    if pool.empty:
        pool = pd.DataFrame(columns=cols_corte)
    else:
        pool = pool.rename(columns={
            'COD_PROD': 'COD_PROD_ORIGEM',
            'DESC_PROD': 'DESC_PROD_ORIGEM',
            'UNID_PROD': 'UNID_PROD_ORIGEM',
            'UNID_FAT': 'UNID_FAT_ORIGEM',
            'TIPO_PROD': 'TIPO_PROD_ORIGEM',
        })
        pool['HR_DISP_ORIGEM'] = pool['PCS_DISP_ORIGEM'] * pool['HR_POR_PCA_ORIGEM_NO_REC']
        pool = pool[(pool['HR_DISP_ORIGEM'] > TOL) & pool['ALOC_REC'].isin(set(mat_cortes['ALOC_REC']))].copy()
        pool = pool.sort_values(['ALOC_REC', 'CRITERIO_ORDEM', 'ID_PROD_UNID_FAT_ORIGEM', 'COMPONENTE']).reset_index(drop=True)

    cortes = []
    validacoes = []
    tol_hr = max(TOL * 100, 1e-6)

    for alvo in mat_cortes.sort_values(['MES_REF', 'ALOC_REC']).itertuples(index=False):
        corte_alvo = float(alvo.CORTE_HR)
        saldo = corte_alvo
        pool_rec = pool[(pool['MES_REF'].eq(alvo.MES_REF)) & pool['ALOC_REC'].eq(str(alvo.ALOC_REC))].copy()
        hr_pool = float(to_num(pool_rec['HR_DISP_ORIGEM']).sum()) if not pool_rec.empty else 0.0
        usados = []

        for ordem, mesma_reg, componente, _ in CRITERIOS_MATRIZ_CORTE:
            if saldo <= tol_hr:
                break
            grp = pool_rec[
                pool_rec['CRITERIO_ORDEM'].eq(ordem)
                & pool_rec['MESMA_REG'].eq(mesma_reg)
                & pool_rec['COMPONENTE'].eq(componente)
            ].copy()
            if grp.empty:
                continue
            hr_criterio = float(to_num(grp['HR_DISP_ORIGEM']).sum())
            if hr_criterio <= TOL:
                continue

            saldo_antes = saldo
            if hr_criterio <= saldo + tol_hr:
                fator_corte = 1.0
                tipo_aplicacao = 'CORTE_TOTAL_CRITERIO'
            else:
                fator_corte = saldo / hr_criterio
                tipo_aplicacao = 'CORTE_PROPORCIONAL_CRITERIO_FINAL'

            grp['CORTE_HR_ALVO'] = corte_alvo
            grp['SALDO_CORTE_HR_ANTES'] = saldo_antes
            grp['HR_CORTE'] = grp['HR_DISP_ORIGEM'] * fator_corte
            grp['PERC_CORTE_NO_COMPONENTE'] = fator_corte
            grp['PCS_CORTE'] = np.divide(
                grp['HR_CORTE'],
                grp['HR_POR_PCA_ORIGEM_NO_REC'],
                out=np.zeros(len(grp), dtype=float),
                where=grp['HR_POR_PCA_ORIGEM_NO_REC'].to_numpy() > TOL,
            )
            grp['PCS_CORTE'] = np.minimum(grp['PCS_CORTE'], grp['PCS_DISP_ORIGEM'])
            hr_aplicado = float(grp['HR_CORTE'].sum())
            saldo = max(saldo - hr_aplicado, 0.0)
            grp['SALDO_CORTE_HR_DEPOIS'] = saldo
            grp['TIPO_APLICACAO_CRITERIO'] = tipo_aplicacao
            cortes.append(grp)
            usados.append(ordem)

        corte_df = pd.concat(cortes, ignore_index=True, sort=False) if cortes else pd.DataFrame()
        if not corte_df.empty:
            atrib_rec = corte_df[(corte_df['MES_REF'].eq(alvo.MES_REF)) & corte_df['ALOC_REC'].eq(str(alvo.ALOC_REC))]
            hr_atribuido = float(to_num(atrib_rec['HR_CORTE']).sum())
            qtd_linhas = int(len(atrib_rec))
        else:
            hr_atribuido = 0.0
            qtd_linhas = 0
        dif = corte_alvo - hr_atribuido
        validacoes.append({
            'MES_REF': alvo.MES_REF,
            'ALOC_REC': str(alvo.ALOC_REC),
            'CORTE_HR_ALVO': corte_alvo,
            'HR_POOL_CORTAVEL': hr_pool,
            'HR_CORTE_ATRIBUIDO': hr_atribuido,
            'DIF_HR_CORTE': dif,
            'SALDO_FINAL_HR': max(dif, 0.0),
            'QTD_LINHAS_CORTE': qtd_linhas,
            'QTD_CRITERIOS_USADOS': len(set(usados)),
            'ULTIMO_CRITERIO_USADO': max(usados) if usados else np.nan,
            'STATUS_VALIDACAO': 'PASSOU' if abs(dif) <= tol_hr else 'FALHOU_POOL_INSUFICIENTE',
        })

    mat_de_onde = pd.concat(cortes, ignore_index=True, sort=False) if cortes else pd.DataFrame(columns=cols_corte)
    if not mat_de_onde.empty:
        for c in cols_corte:
            if c not in mat_de_onde.columns:
                mat_de_onde[c] = np.nan
        mat_de_onde = mat_de_onde[cols_corte].sort_values(
            ['ALOC_REC', 'CRITERIO_ORDEM', 'HR_CORTE'], ascending=[True, True, False]
        ).reset_index(drop=True)

    validacao = pd.DataFrame(validacoes)
    if not validacao.empty:
        validacao = validacao[cols_valid].sort_values('CORTE_HR_ALVO', ascending=False).reset_index(drop=True)
    return mat_de_onde, validacao, pool


MAT_DE_ONDE_CORTAR_M1, VALIDACAO_DE_ONDE_CORTAR_M1, MAT_POOL_CORTAVEL_M1 = gerar_de_onde_cortar_m1(
    validacao_corte_restricao_m1,
    diagnostico_matriz_m1,
    rastro_rateio,
    bd_entrada_completa,
    nec_consolidada_solver,
    estrutura_explodida,
)

if VALIDACAO_DE_ONDE_CORTAR_M1.empty:
    print('       DE ONDE CORTAR M1 - sem restricoes REC validadas para processar')
else:
    _qtd_de_onde = len(VALIDACAO_DE_ONDE_CORTAR_M1)
    _qtd_de_onde_ok = int(VALIDACAO_DE_ONDE_CORTAR_M1['STATUS_VALIDACAO'].eq('PASSOU').sum())
    _qtd_de_onde_falha = _qtd_de_onde - _qtd_de_onde_ok
    _hr_alvo_de_onde = float(VALIDACAO_DE_ONDE_CORTAR_M1['CORTE_HR_ALVO'].sum())
    _hr_atrib_de_onde = float(VALIDACAO_DE_ONDE_CORTAR_M1['HR_CORTE_ATRIBUIDO'].sum())
    _dif_de_onde = float(VALIDACAO_DE_ONDE_CORTAR_M1['DIF_HR_CORTE'].abs().max())
    print(
        '       DE ONDE CORTAR M1 - restricoes:', _qtd_de_onde,
        '| passou:', _qtd_de_onde_ok,
        '| falhou:', _qtd_de_onde_falha,
        '| corte_hr_alvo:', round(_hr_alvo_de_onde, 6),
        '| hr_atribuido:', round(_hr_atrib_de_onde, 6),
        '| dif_max:', round(_dif_de_onde, 9),
    )
    if _qtd_de_onde_falha > 0:
        print('       ATENCAO - existem restricoes sem pool cortavel suficiente nos 9 criterios')
        display(VALIDACAO_DE_ONDE_CORTAR_M1.loc[
            ~VALIDACAO_DE_ONDE_CORTAR_M1['STATUS_VALIDACAO'].eq('PASSOU')
        ])


# 03C. Diagnóstico da restrição sem pool — parcela protegida fora dos 9 critérios


In [ ]:
# ============================================================
# 04D. DIAGNOSTICO DA FALHA DE POOL - M1
# ============================================================
# Etapa somente diagnostica.
# Objetivo: explicar por que uma restricao pode ter CORTE_HR maior que o pool
# disponivel nos 9 criterios, sem alterar regra, NEC ou saida do cliente.
#
# A memoria oficial confirma:
# - somente C_ARR, C_AT, PREV_VEN, PREV_PROX e EST_SEG sao cortaveis;
# - LIMIT_PCS nao e componente de corte;
# - DIF_LM e DIF_EMB nao participam da Matriz vigente.
#
# Portanto, este bloco mede se a diferenca observada vem de parcelas protegidas
# geradas por LIMIT_PCS, lote minimo e multiplo/embalagem.


def diagnosticar_pool_protegido_m1(
    validacao_de_onde,
    diagnostico_m1,
    rotas_m1,
    bd_entrada,
    nec_consolidada,
    estrutura_exp,
):
    cols_resumo = [
        'MES_REF', 'ALOC_REC', 'CORTE_HR_ALVO', 'HR_POOL_9_CRITERIOS',
        'DEFICIT_HR_POOL', 'HR_PROTEG_LIMIT', 'HR_PROTEG_LOTE_MIN',
        'HR_PROTEG_EMB', 'HR_PROTEG_OUTRO', 'HR_PROTEG_TOTAL',
        'DIF_DEFICIT_VS_PROTEGIDO', 'STATUS_DIAGNOSTICO_POOL'
    ]
    cols_detalhe = [
        'MES_REF', 'ALOC_REC', 'ID_PROD_UNID_FAT_ORIGEM', 'COD_PROD_ORIGEM',
        'MESMA_REG', 'TIPO_IMPACTO', 'ITENS_CAPACIDADE_AFETADOS',
        'HR_POR_PCA_ORIGEM_NO_REC', 'NEC_DIRETA_PCS', 'PCS_9_CRITERIOS',
        'RESIDUO_PROTEGIDO_PCS', 'LIMIT_PCS', 'LOTE_MIN', 'QTD_EMB',
        'PROTEG_LIMIT_PCS', 'PROTEG_LOTE_MIN_PCS', 'PROTEG_EMB_PCS',
        'PROTEG_OUTRO_PCS', 'HR_PROTEG_LIMIT', 'HR_PROTEG_LOTE_MIN',
        'HR_PROTEG_EMB', 'HR_PROTEG_OUTRO', 'HR_PROTEG_TOTAL'
    ]

    if not isinstance(validacao_de_onde, pd.DataFrame) or validacao_de_onde.empty:
        return pd.DataFrame(columns=cols_resumo), pd.DataFrame(columns=cols_detalhe)

    falhas = validacao_de_onde[
        ~validacao_de_onde['STATUS_VALIDACAO'].eq('PASSOU')
    ].copy()
    if falhas.empty:
        return pd.DataFrame(columns=cols_resumo), pd.DataFrame(columns=cols_detalhe)

    falhas['MES_REF'] = pd.to_datetime(falhas['MES_REF'], errors='coerce').dt.normalize()
    falhas['ALOC_REC'] = falhas['ALOC_REC'].astype(str)
    recs_falha = set(falhas['ALOC_REC'])
    mes_ref = falhas['MES_REF'].min()
    tol_hr = max(TOL * 100, 1e-6)

    # ------------------------------------------------------------
    # 1. Itens de capacidade que estao efetivamente ligados a REC falha
    # ------------------------------------------------------------
    diag = diagnostico_m1.copy()
    diag['MES_REF'] = pd.to_datetime(diag['MES_REF'], errors='coerce').dt.normalize()
    diag['ID_PROD_UNID_FAT'] = diag['ID_PROD_UNID_FAT'].astype(str)
    diag['RESTRICAO_CANDIDATA'] = diag['RESTRICAO_CANDIDATA'].astype(str)
    diag = diag[
        diag['MES_REF'].eq(mes_ref)
        & diag['STATUS_DIAGNOSTICO'].eq('CORTE_CANDIDATO_OBJETIVO')
        & diag['TIPO_RESTRICAO_CANDIDATA'].eq('REC')
        & diag['RESTRICAO_CANDIDATA'].isin(recs_falha)
    ].copy()

    rot = rotas_m1.copy()
    rot['MES_REF'] = pd.to_datetime(rot['MES_REF'], errors='coerce').dt.normalize()
    rot['ID_PROD_UNID_FAT'] = rot['ID_PROD_UNID_FAT'].astype(str)
    rot['ALOC_REC'] = rot['ALOC_REC'].astype(str)
    rot['PCS_HORA'] = to_num(rot['PCS_HORA'])
    rot['QTD_FALTA_ROTA'] = to_num(rot['QTD_FALTA_ROTA'])
    rot = rot[
        rot['MES_REF'].eq(mes_ref)
        & rot['FOI_ULTIMA_RODADA'].fillna(False)
        & (rot['QTD_FALTA_ROTA'] > TOL)
        & (rot['PCS_HORA'] > TOL)
        & rot['ALOC_REC'].isin(recs_falha)
    ].copy()
    rot = rot.merge(
        diag[['ID_PROD_UNID_FAT', 'RESTRICAO_CANDIDATA']].drop_duplicates(),
        left_on=['ID_PROD_UNID_FAT', 'ALOC_REC'],
        right_on=['ID_PROD_UNID_FAT', 'RESTRICAO_CANDIDATA'],
        how='inner',
    )
    rot = rot.sort_values(['ID_PROD_UNID_FAT', 'PRIOR_MATPAR', 'PRIOR_ROT']).drop_duplicates(
        ['ID_PROD_UNID_FAT', 'ALOC_REC'], keep='first'
    )

    alvos = rot[[
        'MES_REF', 'ID_PROD_UNID_FAT', 'COD_PROD', 'UNID_FAT',
        'ALOC_REC', 'PCS_HORA'
    ]].copy()
    alvos = alvos.rename(columns={
        'ID_PROD_UNID_FAT': 'ID_ITEM_CAPACIDADE',
        'COD_PROD': 'COD_PROD_CAPACIDADE',
        'UNID_FAT': 'UNID_FAT_CAPACIDADE',
        'PCS_HORA': 'PCS_HORA_CAPACIDADE',
    })
    alvos['COD_PROD_CAPACIDADE'] = normalizar_codigo_serie(alvos['COD_PROD_CAPACIDADE'])
    alvos['UNID_FAT_CAPACIDADE'] = alvos['UNID_FAT_CAPACIDADE'].astype('string').fillna('').str.strip()

    # ------------------------------------------------------------
    # 2. Reconstrucao do impacto DIRETO + ESTRUTURAL por origem
    # ------------------------------------------------------------
    est_m1 = estrutura_exp.copy() if isinstance(estrutura_exp, pd.DataFrame) else pd.DataFrame()
    if not est_m1.empty:
        est_m1['MES_REF'] = pd.to_datetime(est_m1['MES_REF'], errors='coerce').dt.normalize()
        est_m1 = est_m1[est_m1['MES_REF'].eq(mes_ref)].copy()
    fatores = _calcular_fator_estrutural_ativo_m1(est_m1)

    impacto_direto = alvos[[
        'MES_REF', 'ALOC_REC', 'ID_ITEM_CAPACIDADE', 'COD_PROD_CAPACIDADE',
        'UNID_FAT_CAPACIDADE', 'PCS_HORA_CAPACIDADE'
    ]].copy()
    impacto_direto['ID_PROD_UNID_FAT_ORIGEM'] = impacto_direto['ID_ITEM_CAPACIDADE']
    impacto_direto['FATOR_ESTRUTURAL'] = 1.0
    impacto_direto['TIPO_IMPACTO_PARCELA'] = 'DIRETO'

    impacto_estrut = pd.DataFrame()
    if not fatores.empty and not alvos.empty:
        impacto_estrut = fatores.merge(
            alvos,
            left_on=['MES_REF', 'COD_INSUMO', 'UNID_PROD_RETORNO'],
            right_on=['MES_REF', 'COD_PROD_CAPACIDADE', 'UNID_FAT_CAPACIDADE'],
            how='inner',
        )
        if not impacto_estrut.empty:
            impacto_estrut['TIPO_IMPACTO_PARCELA'] = 'ESTRUTURAL'
            impacto_estrut = impacto_estrut[[
                'MES_REF', 'ALOC_REC', 'ID_ITEM_CAPACIDADE', 'COD_PROD_CAPACIDADE',
                'UNID_FAT_CAPACIDADE', 'PCS_HORA_CAPACIDADE',
                'ID_PROD_UNID_FAT_ORIGEM', 'FATOR_ESTRUTURAL', 'TIPO_IMPACTO_PARCELA'
            ]].copy()

    impactos = pd.concat([impacto_direto, impacto_estrut], ignore_index=True, sort=False)
    impactos['PCS_HORA_CAPACIDADE'] = to_num(impactos['PCS_HORA_CAPACIDADE'])
    impactos['FATOR_ESTRUTURAL'] = to_num(impactos['FATOR_ESTRUTURAL'])
    impactos = impactos[
        (impactos['PCS_HORA_CAPACIDADE'] > TOL)
        & (impactos['FATOR_ESTRUTURAL'] > TOL)
    ].copy()
    impactos['HR_POR_PCA_PARCELA'] = impactos['FATOR_ESTRUTURAL'] / impactos['PCS_HORA_CAPACIDADE']

    def _tipos_impacto_diag(s):
        tipos = set(s.astype(str))
        if {'DIRETO', 'ESTRUTURAL'}.issubset(tipos):
            return 'DIRETO_E_ESTRUTURAL'
        return 'ESTRUTURAL' if 'ESTRUTURAL' in tipos else 'DIRETO'

    impacto_origem = (
        impactos.groupby(['MES_REF', 'ALOC_REC', 'ID_PROD_UNID_FAT_ORIGEM'], as_index=False)
        .agg(
            HR_POR_PCA_ORIGEM_NO_REC=('HR_POR_PCA_PARCELA', 'sum'),
            TIPO_IMPACTO=('TIPO_IMPACTO_PARCELA', _tipos_impacto_diag),
            ITENS_CAPACIDADE_AFETADOS=('ID_ITEM_CAPACIDADE', lambda s: ';'.join(sorted(set(map(str, s))))),
        )
    )

    # ------------------------------------------------------------
    # 3. Quantidade explicada pelos 9 criterios na origem
    # ------------------------------------------------------------
    bd_m1 = bd_entrada.copy()
    bd_m1['MES_REF'] = pd.to_datetime(bd_m1['MES_REF'], errors='coerce').dt.normalize()
    bd_m1 = bd_m1[bd_m1['MES_REF'].eq(mes_ref)].copy()

    nec_m1 = nec_consolidada.copy() if isinstance(nec_consolidada, pd.DataFrame) else pd.DataFrame()
    if not nec_m1.empty:
        nec_m1['MES_REF'] = pd.to_datetime(nec_m1['MES_REF'], errors='coerce').dt.normalize()
        nec_m1 = nec_m1[nec_m1['MES_REF'].eq(mes_ref)].copy()

    comp = _decompor_componentes_cortaveis_m1(bd_m1, nec_m1)
    if comp.empty:
        comp_sum = pd.DataFrame(columns=['MES_REF', 'ID_PROD_UNID_FAT_ORIGEM', 'PCS_9_CRITERIOS'])
    else:
        comp_sum = (
            comp.groupby(['MES_REF', 'ID_PROD_UNID_FAT_ORIGEM'], as_index=False)
            .agg(PCS_9_CRITERIOS=('PCS_DISP_ORIGEM', 'sum'))
        )

    # ------------------------------------------------------------
    # 4. NEC direta e parametros protegidos da origem
    # ------------------------------------------------------------
    base = bd_m1.copy()
    base['ID_PROD_UNID_FAT'] = base['ID_PROD_UNID_FAT'].astype(str)
    base['COD_PROD'] = normalizar_codigo_serie(base['COD_PROD'])
    if 'MESMA_REG' not in base.columns:
        base['MESMA_REG'] = 'SIM'
    base['MESMA_REG'] = base['MESMA_REG'].astype('string').fillna('SIM').str.strip().str.upper()
    for c in ['LIMIT_PCS', 'LOTE_MIN', 'QTD_EMB']:
        if c not in base.columns:
            base[c] = 0.0
        base[c] = to_num(base[c])

    base_item = (
        base.sort_values(['ID_PROD_UNID_FAT', 'PRIOR_MATPAR', 'PRIOR_ROT'])
        .groupby(['MES_REF', 'ID_PROD_UNID_FAT'], as_index=False)
        .agg(
            COD_PROD_ORIGEM=('COD_PROD', 'first'),
            MESMA_REG=('MESMA_REG', 'first'),
            LIMIT_PCS=('LIMIT_PCS', 'max'),
            LOTE_MIN=('LOTE_MIN', 'max'),
            QTD_EMB=('QTD_EMB', 'max'),
        )
        .rename(columns={'ID_PROD_UNID_FAT': 'ID_PROD_UNID_FAT_ORIGEM'})
    )

    if not nec_m1.empty:
        nec_dir = (
            nec_m1.groupby(['MES_REF', 'ID_PROD_UNID_FAT'], as_index=False)
            .agg(NEC_DIRETA_PCS=('NEC_DIRETA_PCS', 'max'))
            .rename(columns={'ID_PROD_UNID_FAT': 'ID_PROD_UNID_FAT_ORIGEM'})
        )
    else:
        nec_dir = pd.DataFrame(columns=['MES_REF', 'ID_PROD_UNID_FAT_ORIGEM', 'NEC_DIRETA_PCS'])

    det = (
        impacto_origem
        .merge(base_item, on=['MES_REF', 'ID_PROD_UNID_FAT_ORIGEM'], how='left')
        .merge(nec_dir, on=['MES_REF', 'ID_PROD_UNID_FAT_ORIGEM'], how='left')
        .merge(comp_sum, on=['MES_REF', 'ID_PROD_UNID_FAT_ORIGEM'], how='left')
    )

    for c in ['HR_POR_PCA_ORIGEM_NO_REC', 'LIMIT_PCS', 'LOTE_MIN', 'QTD_EMB', 'NEC_DIRETA_PCS', 'PCS_9_CRITERIOS']:
        det[c] = to_num(det[c])

    det['RESIDUO_PROTEGIDO_PCS'] = (
        det['NEC_DIRETA_PCS'] - det['PCS_9_CRITERIOS']
    ).clip(lower=0.0)

    # Decomposicao explicativa do residuo protegido.
    det['PROTEG_LIMIT_PCS'] = np.minimum(
        det['RESIDUO_PROTEGIDO_PCS'],
        (det['LIMIT_PCS'] - det['PCS_9_CRITERIOS']).clip(lower=0.0),
    )
    saldo_resid = (det['RESIDUO_PROTEGIDO_PCS'] - det['PROTEG_LIMIT_PCS']).clip(lower=0.0)
    base_pos_limit = np.maximum(det['PCS_9_CRITERIOS'], det['LIMIT_PCS'])

    delta_lote = np.where(
        base_pos_limit > TOL,
        np.maximum(det['LOTE_MIN'] - base_pos_limit, 0.0),
        0.0,
    )
    det['PROTEG_LOTE_MIN_PCS'] = np.minimum(saldo_resid, delta_lote)
    saldo_resid = (saldo_resid - det['PROTEG_LOTE_MIN_PCS']).clip(lower=0.0)

    base_pos_lote = base_pos_limit + delta_lote
    emb_target = np.where(
        (base_pos_lote > TOL) & (det['QTD_EMB'] > TOL),
        np.ceil((base_pos_lote - 1e-12) / det['QTD_EMB']) * det['QTD_EMB'],
        base_pos_lote,
    )
    delta_emb = np.maximum(emb_target - base_pos_lote, 0.0)
    det['PROTEG_EMB_PCS'] = np.minimum(saldo_resid, delta_emb)
    saldo_resid = (saldo_resid - det['PROTEG_EMB_PCS']).clip(lower=0.0)
    det['PROTEG_OUTRO_PCS'] = saldo_resid

    for nome in ['LIMIT', 'LOTE_MIN', 'EMB', 'OUTRO']:
        det[f'HR_PROTEG_{nome}'] = det[f'PROTEG_{nome}_PCS'] * det['HR_POR_PCA_ORIGEM_NO_REC']
    det['HR_PROTEG_TOTAL'] = (
        det['HR_PROTEG_LIMIT'] + det['HR_PROTEG_LOTE_MIN']
        + det['HR_PROTEG_EMB'] + det['HR_PROTEG_OUTRO']
    )

    # ------------------------------------------------------------
    # 5. Resumo por REC falha e reconciliacao com o deficit observado
    # ------------------------------------------------------------
    prot_rec = (
        det.groupby(['MES_REF', 'ALOC_REC'], as_index=False)
        .agg(
            HR_PROTEG_LIMIT=('HR_PROTEG_LIMIT', 'sum'),
            HR_PROTEG_LOTE_MIN=('HR_PROTEG_LOTE_MIN', 'sum'),
            HR_PROTEG_EMB=('HR_PROTEG_EMB', 'sum'),
            HR_PROTEG_OUTRO=('HR_PROTEG_OUTRO', 'sum'),
            HR_PROTEG_TOTAL=('HR_PROTEG_TOTAL', 'sum'),
        )
    )

    resumo = falhas[[
        'MES_REF', 'ALOC_REC', 'CORTE_HR_ALVO', 'HR_POOL_CORTAVEL', 'DIF_HR_CORTE'
    ]].copy()
    resumo = resumo.rename(columns={
        'HR_POOL_CORTAVEL': 'HR_POOL_9_CRITERIOS',
        'DIF_HR_CORTE': 'DEFICIT_HR_POOL',
    })
    resumo = resumo.merge(prot_rec, on=['MES_REF', 'ALOC_REC'], how='left')
    for c in ['HR_PROTEG_LIMIT', 'HR_PROTEG_LOTE_MIN', 'HR_PROTEG_EMB', 'HR_PROTEG_OUTRO', 'HR_PROTEG_TOTAL']:
        resumo[c] = to_num(resumo[c])
    resumo['DIF_DEFICIT_VS_PROTEGIDO'] = resumo['DEFICIT_HR_POOL'] - resumo['HR_PROTEG_TOTAL']
    resumo['STATUS_DIAGNOSTICO_POOL'] = np.where(
        resumo['DIF_DEFICIT_VS_PROTEGIDO'].abs() <= tol_hr,
        'FECHA_COM_PARCELA_PROTEGIDA',
        'NAO_FECHA_COM_PARCELA_PROTEGIDA',
    )

    det = det[det['HR_PROTEG_TOTAL'] > TOL].copy()
    det = det[[c for c in cols_detalhe if c in det.columns]].sort_values(
        ['ALOC_REC', 'HR_PROTEG_TOTAL'], ascending=[True, False]
    ).reset_index(drop=True)
    resumo = resumo[[c for c in cols_resumo if c in resumo.columns]].reset_index(drop=True)
    return resumo, det


DIAG_POOL_PROTEGIDO_M1, DIAG_POOL_PROTEGIDO_DETALHE_M1 = diagnosticar_pool_protegido_m1(
    VALIDACAO_DE_ONDE_CORTAR_M1,
    diagnostico_matriz_m1,
    diagnostico_matriz_rotas_m1,
    bd_entrada_completa,
    nec_consolidada_solver,
    estrutura_explodida,
)

if DIAG_POOL_PROTEGIDO_M1.empty:
    print('       DIAGNOSTICO POOL PROTEGIDO M1 - nenhuma restricao com falha de pool')
else:
    print('       DIAGNOSTICO POOL PROTEGIDO M1')
    display(DIAG_POOL_PROTEGIDO_M1)
    print('       ORIGENS COM PARCELA PROTEGIDA NA(S) REC(S) COM FALHA')
    display(DIAG_POOL_PROTEGIDO_DETALHE_M1)


# 03D. Aplicação operacional dos cortes — M1 (v66_13)

Aplica em peças os cortes já validados pelo `DE ONDE CORTAR`, recalcula a NEC/estrutura e reroda a V66 uma vez para medir o estado real pós-corte.


In [ ]:
# ============================================================
# 04E. APLICACAO OPERACIONAL DOS CORTES - M1 / CORTE UNICO MULTI-REC
# ============================================================
_iniciar_etapa_v58("MATRIZ_CORTES_M1")
_TEMPO_RODADA1_M1_INICIO = time.perf_counter()

# Esta etapa parte exclusivamente do QUANTO CORTAR e DE ONDE CORTAR
# ja validados na v66_12, com consolidacao fisica global corrigida na v66_14.
#
# Regras preservadas:
# - NEC_PCS nao recebe corte direto;
# - somente C_ARR, C_AT, PREV_VEN, PREV_PROX e EST_SEG podem ser alterados;
# - ORI_* e congelado antes do primeiro corte e nunca sobrescrito;
# - o mesmo volume de origem+componente e cortado UMA unica vez, mesmo quando impacta varias RECs;
# - depois do corte, NEC e estrutura sao recalculadas pelo proprio motor V66;
# - a primeira aplicacao permanece explicita; as rodadas seguintes sao controladas pelo loop REC -> FER.

_MAPA_ORI_COMPONENTE_M1 = {
    'C_ARR': ('LTP_CART_ARR_MES_ANT', 'ORI_C_ARR'),
    'C_AT': ('LTP_CART_MES_ATUAL', 'ORI_C_AT'),
    'PREV_VEN': ('LTP_SALDO_PREV_PCS', 'ORI_PREV_VEN'),
    'PREV_PROX': ('LTP_SALDO_PREV_PROX_MES_PCS', 'ORI_PREV_PROX'),
    'EST_SEG': ('LTP_EST_SEG_PCS', 'ORI_EST_SEG'),
}


def _aplicar_cortes_matriz_m1(bd_entrada, matriz_cortes, pool_cortavel, validacao_de_onde, aplicar_regras_industriais=False):
    """Materializa PCS_CORTE nos cinco componentes permitidos, sem dupla contagem entre RECs.

    Regra preservada:
    - o plano DE ONDE CORTAR continua sendo calculado por REC;
    - quando a mesma origem+componente aparece em mais de uma REC, o corte fisico ocorre uma unica vez;
    - a quantidade fisica a aplicar e o MAIOR PCS_CORTE requerido entre as RECs, nunca a soma;
    - esse unico corte libera horas simultaneamente em todas as RECs impactadas pela origem.
    """
    tol_pcs = max(TOL * 100, 1e-6)
    tol_hr = max(TOL * 100, 1e-6)

    if not isinstance(matriz_cortes, pd.DataFrame) or matriz_cortes.empty:
        raise RuntimeError('MATRIZ_CORTES_M1_VAZIA: nao existem cortes validados para aplicar.')
    if not isinstance(bd_entrada, pd.DataFrame) or bd_entrada.empty:
        raise RuntimeError('BASE_M1_VAZIA: nao existe base de entrada para aplicar os cortes.')
    if not isinstance(pool_cortavel, pd.DataFrame) or pool_cortavel.empty:
        raise RuntimeError('POOL_CORTAVEL_M1_VAZIO: necessario para validar o alivio global entre RECs.')
    if not isinstance(validacao_de_onde, pd.DataFrame) or validacao_de_onde.empty:
        raise RuntimeError('VALIDACAO_DE_ONDE_M1_VAZIA: necessario para validar o alivio global entre RECs.')

    cortes = matriz_cortes.copy()
    cortes['MES_REF'] = pd.to_datetime(cortes['MES_REF'], errors='coerce').dt.normalize()
    cortes['ID_PROD_UNID_FAT_ORIGEM'] = cortes['ID_PROD_UNID_FAT_ORIGEM'].astype(str)
    cortes['ALOC_REC'] = cortes['ALOC_REC'].astype(str)
    cortes['PCS_CORTE'] = to_num(cortes['PCS_CORTE']).clip(lower=0.0)
    cortes['HR_CORTE'] = to_num(cortes['HR_CORTE']).clip(lower=0.0)
    cortes = cortes[cortes['PCS_CORTE'] > TOL].copy()
    if cortes.empty:
        raise RuntimeError('MATRIZ_CORTES_M1_SEM_PCS_CORTE_POSITIVO')

    # ------------------------------------------------------------
    # 0. Consolidacao fisica global sem dupla contagem entre RECs
    # ------------------------------------------------------------
    cols_chave = ['MES_REF', 'ID_PROD_UNID_FAT_ORIGEM', 'COMPONENTE', 'COLUNA_LTP_CORTE']

    # Diagnostico por REC: preserva o maior corte de pecas pedido por aquela REC.
    cortes_por_rec = (
        cortes.groupby(cols_chave + ['ALOC_REC'], as_index=False)
        .agg(
            PCS_CORTE_REC=('PCS_CORTE', 'max'),
            HR_CORTE_REC=('HR_CORTE', 'sum'),
        )
    )

    # Regra fisica: a mesma origem+componente e reduzida uma unica vez.
    # Se varias RECs pedirem corte desse mesmo volume, aplica-se o MAIOR PCS_CORTE requerido.
    cortes_cons = (
        cortes_por_rec.groupby(cols_chave, as_index=False)
        .agg(
            PCS_CORTE_PLANEJADO=('PCS_CORTE_REC', 'max'),
            PCS_CORTE_SOMA_RECS_DIAG=('PCS_CORTE_REC', 'sum'),
            HR_CORTE_SOMA_RECS_DIAG=('HR_CORTE_REC', 'sum'),
            QTD_RECS_ENVOLVIDAS=('ALOC_REC', 'nunique'),
            RECS_ENVOLVIDAS=('ALOC_REC', lambda s: ';'.join(sorted(set(map(str, s))))),
        )
    )
    cortes_cons['PCS_DUPLA_CONTAGEM_ELIMINADA'] = (
        cortes_cons['PCS_CORTE_SOMA_RECS_DIAG'] - cortes_cons['PCS_CORTE_PLANEJADO']
    ).clip(lower=0.0)

    # ------------------------------------------------------------
    # 0B. Validacao do alivio global por REC antes de alterar a base
    # ------------------------------------------------------------
    pool = pool_cortavel.copy()
    pool['MES_REF'] = pd.to_datetime(pool['MES_REF'], errors='coerce').dt.normalize()
    pool['ID_PROD_UNID_FAT_ORIGEM'] = pool['ID_PROD_UNID_FAT_ORIGEM'].astype(str)
    pool['ALOC_REC'] = pool['ALOC_REC'].astype(str)
    pool['HR_POR_PCA_ORIGEM_NO_REC'] = to_num(pool['HR_POR_PCA_ORIGEM_NO_REC'])

    alivio = pool.merge(
        cortes_cons[['MES_REF', 'ID_PROD_UNID_FAT_ORIGEM', 'COMPONENTE', 'PCS_CORTE_PLANEJADO']],
        on=['MES_REF', 'ID_PROD_UNID_FAT_ORIGEM', 'COMPONENTE'],
        how='inner',
        validate='many_to_one',
    )
    alivio['HR_ALIVIO_CORTE_UNICO'] = (
        to_num(alivio['PCS_CORTE_PLANEJADO']) * to_num(alivio['HR_POR_PCA_ORIGEM_NO_REC'])
    )
    alivio_rec = (
        alivio.groupby(['MES_REF', 'ALOC_REC'], as_index=False)
        .agg(
            HR_ALIVIO_GLOBAL=('HR_ALIVIO_CORTE_UNICO', 'sum'),
            QTD_ORIGENS_COMPONENTES=('ID_PROD_UNID_FAT_ORIGEM', 'nunique'),
        )
    )

    val_rec = validacao_de_onde.copy()
    val_rec['MES_REF'] = pd.to_datetime(val_rec['MES_REF'], errors='coerce').dt.normalize()
    val_rec['ALOC_REC'] = val_rec['ALOC_REC'].astype(str)
    val_rec['CORTE_HR_ALVO'] = to_num(val_rec['CORTE_HR_ALVO'])
    val_rec = val_rec[['MES_REF', 'ALOC_REC', 'CORTE_HR_ALVO']].drop_duplicates()
    validacao_alivio = val_rec.merge(alivio_rec, on=['MES_REF', 'ALOC_REC'], how='left')
    validacao_alivio['HR_ALIVIO_GLOBAL'] = to_num(validacao_alivio['HR_ALIVIO_GLOBAL'])
    validacao_alivio['QTD_ORIGENS_COMPONENTES'] = pd.to_numeric(
        validacao_alivio['QTD_ORIGENS_COMPONENTES'], errors='coerce'
    ).fillna(0).astype(int)
    validacao_alivio['DEFICIT_ALIVIO_HR'] = (
        validacao_alivio['CORTE_HR_ALVO'] - validacao_alivio['HR_ALIVIO_GLOBAL']
    ).clip(lower=0.0)
    validacao_alivio['EXCESSO_ALIVIO_HR'] = (
        validacao_alivio['HR_ALIVIO_GLOBAL'] - validacao_alivio['CORTE_HR_ALVO']
    ).clip(lower=0.0)
    validacao_alivio['STATUS_ALIVIO_GLOBAL'] = np.where(
        validacao_alivio['DEFICIT_ALIVIO_HR'] <= tol_hr,
        'PASSOU',
        'FALHOU_DEFICIT_ALIVIO',
    )

    falhas_alivio = validacao_alivio[~validacao_alivio['STATUS_ALIVIO_GLOBAL'].eq('PASSOU')].copy()
    if not falhas_alivio.empty:
        display(falhas_alivio)
        raise RuntimeError(
            f'ALIVIO_GLOBAL_MULTI_REC_INSUFICIENTE: {len(falhas_alivio)} REC(s) com deficit apos consolidar corte unico.'
        )

    base = bd_entrada.copy()
    base['MES_REF'] = pd.to_datetime(base['MES_REF'], errors='coerce').dt.normalize()
    base['ID_PROD_UNID_FAT'] = base['ID_PROD_UNID_FAT'].astype(str)

    mes_ref = cortes_cons['MES_REF'].min()
    base = base[base['MES_REF'].eq(mes_ref)].copy().reset_index(drop=True)
    if base.empty:
        raise RuntimeError(f'BASE_M1_SEM_LINHAS_PARA_MES_{pd.Timestamp(mes_ref):%Y-%m}')

    # ------------------------------------------------------------
    # 1. Congelamento das origens antes de qualquer corte
    # ------------------------------------------------------------
    base_ori_nec = calcular_nec_pcs_se_necessario(base.copy(), aplicar_regras_industriais=aplicar_regras_industriais)
    for componente, (col_ltp, col_ori) in _MAPA_ORI_COMPONENTE_M1.items():
        if col_ltp not in base.columns:
            base[col_ltp] = 0.0
        base[col_ltp] = to_num(base[col_ltp]).clip(lower=0.0)
        if col_ori not in base.columns:
            base[col_ori] = base[col_ltp]
        else:
            base[col_ori] = to_num(base[col_ori])

    if 'ORI_NEC' not in base.columns:
        base['ORI_NEC'] = to_num(base_ori_nec['NEC_PCS'])
    else:
        base['ORI_NEC'] = to_num(base['ORI_NEC'])

    # ------------------------------------------------------------
    # 2. Saldo real disponivel por origem + componente
    # ------------------------------------------------------------
    registros_disp = []
    for componente, (col_ltp, _) in _MAPA_ORI_COMPONENTE_M1.items():
        tmp = (
            base.groupby(['MES_REF', 'ID_PROD_UNID_FAT'], as_index=False)
            .agg(
                PCS_DISP_MIN=(col_ltp, 'min'),
                PCS_DISP_MAX=(col_ltp, 'max'),
                QTD_ROTAS_ORIGEM=(col_ltp, 'size'),
            )
        )
        tmp['COMPONENTE'] = componente
        tmp['COLUNA_LTP_CORTE'] = col_ltp
        registros_disp.append(tmp)

    disp = pd.concat(registros_disp, ignore_index=True, sort=False)
    disp = disp.rename(columns={'ID_PROD_UNID_FAT': 'ID_PROD_UNID_FAT_ORIGEM'})
    disp['DIF_VALOR_ENTRE_ROTAS'] = (to_num(disp['PCS_DISP_MAX']) - to_num(disp['PCS_DISP_MIN'])).abs()

    validacao = cortes_cons.merge(
        disp,
        on=['MES_REF', 'ID_PROD_UNID_FAT_ORIGEM', 'COMPONENTE', 'COLUNA_LTP_CORTE'],
        how='left',
        validate='one_to_one',
    )
    for c in ['PCS_DISP_MIN', 'PCS_DISP_MAX', 'DIF_VALOR_ENTRE_ROTAS']:
        validacao[c] = pd.to_numeric(validacao[c], errors='coerce')

    validacao['DIF_CORTE_VS_DISP'] = validacao['PCS_CORTE_PLANEJADO'] - validacao['PCS_DISP_MIN']
    validacao['STATUS_SOBRECORTE'] = np.select(
        [
            validacao['PCS_DISP_MIN'].isna(),
            validacao['DIF_VALOR_ENTRE_ROTAS'] > tol_pcs,
            validacao['DIF_CORTE_VS_DISP'] > tol_pcs,
        ],
        [
            'ORIGEM_NAO_ENCONTRADA',
            'VALOR_LTP_DIVERGENTE_ENTRE_ROTAS',
            'SOBRECORTE_PCS',
        ],
        default='PASSOU',
    )

    falhas = validacao[~validacao['STATUS_SOBRECORTE'].eq('PASSOU')].copy()
    if not falhas.empty:
        display(falhas)
        raise RuntimeError(
            f'APLICACAO_MATRIZ_SOBRECORTE_OU_DIVERGENCIA: {len(falhas)} origem(ns)+componente(s) com falha.'
        )

    # ------------------------------------------------------------
    # 3. Aplicacao vetorizada do corte unico em todas as rotas da origem
    # ------------------------------------------------------------
    antes_item = {}
    for componente, (col_ltp, _) in _MAPA_ORI_COMPONENTE_M1.items():
        antes_item[componente] = (
            base.groupby(['MES_REF', 'ID_PROD_UNID_FAT'], as_index=False)[col_ltp]
            .max()
            .rename(columns={col_ltp: 'PCS_ANTES'})
        )

    for componente, (col_ltp, _) in _MAPA_ORI_COMPONENTE_M1.items():
        corte_comp = cortes_cons[cortes_cons['COMPONENTE'].eq(componente)][
            ['MES_REF', 'ID_PROD_UNID_FAT_ORIGEM', 'PCS_CORTE_PLANEJADO']
        ].copy()
        if corte_comp.empty:
            continue
        mapa_corte = {
            (pd.Timestamp(m), str(i)): float(q)
            for m, i, q in corte_comp.itertuples(index=False, name=None)
        }
        chaves = list(zip(base['MES_REF'], base['ID_PROD_UNID_FAT']))
        corte_linha = pd.Series(
            [mapa_corte.get((pd.Timestamp(m), str(i)), 0.0) for m, i in chaves],
            index=base.index,
        )
        base[col_ltp] = (to_num(base[col_ltp]) - corte_linha).clip(lower=0.0)

    # ------------------------------------------------------------
    # 4. Auditoria planejado x aplicado no nivel da origem
    # ------------------------------------------------------------
    registros_aud = []
    for componente, (col_ltp, _) in _MAPA_ORI_COMPONENTE_M1.items():
        depois = (
            base.groupby(['MES_REF', 'ID_PROD_UNID_FAT'], as_index=False)[col_ltp]
            .max()
            .rename(columns={col_ltp: 'PCS_DEPOIS'})
        )
        aud = antes_item[componente].merge(depois, on=['MES_REF', 'ID_PROD_UNID_FAT'], how='outer')
        aud['COMPONENTE'] = componente
        aud['PCS_CORTE_APLICADO'] = (to_num(aud['PCS_ANTES']) - to_num(aud['PCS_DEPOIS'])).clip(lower=0.0)
        aud = aud.rename(columns={'ID_PROD_UNID_FAT': 'ID_PROD_UNID_FAT_ORIGEM'})
        registros_aud.append(aud)

    auditoria = pd.concat(registros_aud, ignore_index=True, sort=False)
    auditoria = cortes_cons.merge(
        auditoria,
        on=['MES_REF', 'ID_PROD_UNID_FAT_ORIGEM', 'COMPONENTE'],
        how='left',
        validate='one_to_one',
    )
    auditoria['DIF_PCS_PLANEJADO_VS_APLICADO'] = (
        auditoria['PCS_CORTE_APLICADO'] - auditoria['PCS_CORTE_PLANEJADO']
    )
    auditoria['STATUS_APLICACAO'] = np.where(
        auditoria['DIF_PCS_PLANEJADO_VS_APLICADO'].abs() <= tol_pcs,
        'PASSOU',
        'FALHOU',
    )

    falhas_aplicacao = auditoria[~auditoria['STATUS_APLICACAO'].eq('PASSOU')].copy()
    if not falhas_aplicacao.empty:
        display(falhas_aplicacao)
        raise RuntimeError(
            f'APLICACAO_MATRIZ_DIVERGENTE: {len(falhas_aplicacao)} origem(ns)+componente(s) nao reconciliaram.'
        )

    # ------------------------------------------------------------
    # 5. Recalculo da NEC direta apos alterar as origens
    # ------------------------------------------------------------
    cols_derived = [
        'NEC_PCS', 'NEC_ESTRUTURA_PAI_PCS', 'NEC_DIRETA_BRUTA_PCS',
        'NEC_PCS_ORIGINAL', 'NEC_DIRETA_PCS', 'NEC_COMP_PCS',
        'NEC_PCS_CONSOLIDADA', 'NEC_EXPLODIDA_BRUTA_PCS',
        'ESTOQUE_CONSUMIDO_EXPLOSAO_PCS', 'NEC_EXPLODIDA_PCS',
    ]
    base = base.drop(columns=[c for c in cols_derived if c in base.columns], errors='ignore')
    base = calcular_nec_pcs_se_necessario(base, aplicar_regras_industriais=aplicar_regras_industriais)

    return base, auditoria, validacao, cortes_cons, validacao_alivio


# ------------------------------------------------------------
# A. Precondicao: DE ONDE CORTAR precisa estar 100% validado
# ------------------------------------------------------------
if not isinstance(VALIDACAO_DE_ONDE_CORTAR_M1, pd.DataFrame) or VALIDACAO_DE_ONDE_CORTAR_M1.empty:
    raise RuntimeError('DE_ONDE_CORTAR_M1_NAO_VALIDADO')

_falhas_de_onde_v66_14 = VALIDACAO_DE_ONDE_CORTAR_M1[
    ~VALIDACAO_DE_ONDE_CORTAR_M1['STATUS_VALIDACAO'].eq('PASSOU')
].copy()
if not _falhas_de_onde_v66_14.empty:
    raise RuntimeError(
        f'DE_ONDE_CORTAR_M1_COM_FALHAS: {len(_falhas_de_onde_v66_14)} restricao(oes) ainda nao fecham.'
    )

_mes_m1_v66_14 = pd.to_datetime(bd_entrada_completa['MES_REF'], errors='coerce').min()
_BD_M1_ORIGINAL_V66_14 = bd_entrada_completa[
    pd.to_datetime(bd_entrada_completa['MES_REF'], errors='coerce').eq(_mes_m1_v66_14)
].copy().reset_index(drop=True)

_TEMPO_APLICACAO_ROD1_M1_INICIO = time.perf_counter()
BD_M1_POS_CORTE_MATRIZ, AUDITORIA_APLICACAO_CORTES_M1, VALIDACAO_SOBRECORTE_M1, CORTES_CONSOLIDADOS_ORIGEM_M1, VALIDACAO_ALIVIO_GLOBAL_REC_M1 = _aplicar_cortes_matriz_m1(
    _BD_M1_ORIGINAL_V66_14,
    MAT_DE_ONDE_CORTAR_M1,
    MAT_POOL_CORTAVEL_M1,
    VALIDACAO_DE_ONDE_CORTAR_M1,
    aplicar_regras_industriais=False,
)
_TEMPO_APLICACAO_ROD1_M1 = time.perf_counter() - _TEMPO_APLICACAO_ROD1_M1_INICIO

_qtd_aplic_v66_14 = len(AUDITORIA_APLICACAO_CORTES_M1)
_qtd_aplic_ok_v66_14 = int(AUDITORIA_APLICACAO_CORTES_M1['STATUS_APLICACAO'].eq('PASSOU').sum())
_pcs_soma_rec_diag_v66_14 = float(AUDITORIA_APLICACAO_CORTES_M1['PCS_CORTE_SOMA_RECS_DIAG'].sum())
_pcs_plan_v66_14 = float(AUDITORIA_APLICACAO_CORTES_M1['PCS_CORTE_PLANEJADO'].sum())
_pcs_apl_v66_14 = float(AUDITORIA_APLICACAO_CORTES_M1['PCS_CORTE_APLICADO'].sum())
_pcs_dup_elim_v66_14 = float(AUDITORIA_APLICACAO_CORTES_M1['PCS_DUPLA_CONTAGEM_ELIMINADA'].sum())
_dif_apl_v66_14 = float(AUDITORIA_APLICACAO_CORTES_M1['DIF_PCS_PLANEJADO_VS_APLICADO'].abs().max())
print(
    '       APLICACAO GLOBAL CORTES M1 - origem+componente:', _qtd_aplic_v66_14,
    '| passou:', _qtd_aplic_ok_v66_14,
    '| pcs_soma_rec_diag:', round(_pcs_soma_rec_diag_v66_14, 6),
    '| pcs_unico_aplicar:', round(_pcs_plan_v66_14, 6),
    '| dupla_contagem_eliminada:', round(_pcs_dup_elim_v66_14, 6),
    '| pcs_aplicado:', round(_pcs_apl_v66_14, 6),
    '| dif_max:', round(_dif_apl_v66_14, 9),
)

_qtd_alivio_v66_14 = len(VALIDACAO_ALIVIO_GLOBAL_REC_M1)
_qtd_alivio_ok_v66_14 = int(VALIDACAO_ALIVIO_GLOBAL_REC_M1['STATUS_ALIVIO_GLOBAL'].eq('PASSOU').sum())
_hr_alvo_global_v66_14 = float(VALIDACAO_ALIVIO_GLOBAL_REC_M1['CORTE_HR_ALVO'].sum())
_hr_alivio_global_v66_14 = float(VALIDACAO_ALIVIO_GLOBAL_REC_M1['HR_ALIVIO_GLOBAL'].sum())
_hr_excesso_global_v66_14 = float(VALIDACAO_ALIVIO_GLOBAL_REC_M1['EXCESSO_ALIVIO_HR'].sum())
print(
    '       ALIVIO GLOBAL MULTI-REC - restricoes:', _qtd_alivio_v66_14,
    '| passou:', _qtd_alivio_ok_v66_14,
    '| falhou:', _qtd_alivio_v66_14 - _qtd_alivio_ok_v66_14,
    '| corte_hr_alvo:', round(_hr_alvo_global_v66_14, 6),
    '| hr_alivio_global:', round(_hr_alivio_global_v66_14, 6),
    '| excesso_intrinseco:', round(_hr_excesso_global_v66_14, 6),
)

# ------------------------------------------------------------
# B. Rerodagem unica da V66 com as origens ajustadas
# ------------------------------------------------------------
_TEMPO_V66_ROD1_M1_INICIO = time.perf_counter()
RESULTADO_V66_POS_CORTE_M1 = _executar_motor_periodo_v58(BD_M1_POS_CORTE_MATRIZ.copy(), aplicar_regras_industriais=False)
_TEMPO_V66_ROD1_M1 = time.perf_counter() - _TEMPO_V66_ROD1_M1_INICIO

NEC_CONSOLIDADA_POS_CORTE_M1 = RESULTADO_V66_POS_CORTE_M1.get('nec_consolidada_solver', pd.DataFrame()).copy()
ESTRUTURA_EXPLODIDA_POS_CORTE_M1 = RESULTADO_V66_POS_CORTE_M1.get('estrutura_explodida', pd.DataFrame()).copy()
RASTRO_RATEIO_POS_CORTE_M1 = RESULTADO_V66_POS_CORTE_M1.get('rastro_rateio', pd.DataFrame()).copy()
DIAGNOSTICO_MATRIZ_POS_CORTE_M1 = RESULTADO_V66_POS_CORTE_M1.get('diagnostico_matriz', pd.DataFrame()).copy()
DIAGNOSTICO_MATRIZ_ROTAS_POS_CORTE_M1 = RESULTADO_V66_POS_CORTE_M1.get('diagnostico_matriz_rotas', pd.DataFrame()).copy()

VALIDACAO_CORTE_RESTRICAO_POS_CORTE_M1 = validar_corte_hr_por_restricao_m1(
    DIAGNOSTICO_MATRIZ_POS_CORTE_M1,
    DIAGNOSTICO_MATRIZ_ROTAS_POS_CORTE_M1,
    RASTRO_RATEIO_POS_CORTE_M1,
)

# ------------------------------------------------------------
# C. Auditoria NEC antes x depois
# ------------------------------------------------------------
_base_nec_pos_v66_13 = (
    BD_M1_POS_CORTE_MATRIZ.groupby(['MES_REF', 'ID_PROD_UNID_FAT'], as_index=False)
    .agg(
        ORI_NEC=('ORI_NEC', 'max'),
        NEC_DIRETA_RECALCULADA_PCS=('NEC_PCS', 'max'),
    )
)

if isinstance(NEC_CONSOLIDADA_POS_CORTE_M1, pd.DataFrame) and not NEC_CONSOLIDADA_POS_CORTE_M1.empty:
    _nec_cons_v66_13 = NEC_CONSOLIDADA_POS_CORTE_M1.copy()
    _nec_cons_v66_13['MES_REF'] = pd.to_datetime(_nec_cons_v66_13['MES_REF'], errors='coerce').dt.normalize()
    _nec_cons_v66_13 = (
        _nec_cons_v66_13.groupby(['MES_REF', 'ID_PROD_UNID_FAT'], as_index=False)
        .agg(
            NEC_DIRETA_POS_CORTE_PCS=('NEC_DIRETA_PCS', 'max'),
            NEC_COMP_POS_CORTE_PCS=('NEC_COMP_PCS', 'max'),
            NEC_CONSOLIDADA_POS_CORTE_PCS=('NEC_PCS_CONSOLIDADA', 'max'),
        )
    )
    AUDITORIA_NEC_POS_CORTE_M1 = _base_nec_pos_v66_13.merge(
        _nec_cons_v66_13,
        on=['MES_REF', 'ID_PROD_UNID_FAT'],
        how='left',
        validate='one_to_one',
    )
else:
    AUDITORIA_NEC_POS_CORTE_M1 = _base_nec_pos_v66_13.copy()

# ------------------------------------------------------------
# D. Resumo do estado residual de REC apos a primeira aplicacao
# ------------------------------------------------------------
if isinstance(DIAGNOSTICO_MATRIZ_POS_CORTE_M1, pd.DataFrame) and not DIAGNOSTICO_MATRIZ_POS_CORTE_M1.empty:
    _diag_rec_res_v66_13 = DIAGNOSTICO_MATRIZ_POS_CORTE_M1[
        DIAGNOSTICO_MATRIZ_POS_CORTE_M1['STATUS_DIAGNOSTICO'].eq('CORTE_CANDIDATO_OBJETIVO')
        & DIAGNOSTICO_MATRIZ_POS_CORTE_M1['TIPO_RESTRICAO_CANDIDATA'].eq('REC')
    ].copy()
else:
    _diag_rec_res_v66_13 = pd.DataFrame()

if _diag_rec_res_v66_13.empty:
    _qtd_rec_res_v66_13 = 0
    _hr_rec_res_v66_13 = 0.0
else:
    _qtd_rec_res_v66_13 = int(_diag_rec_res_v66_13['RESTRICAO_CANDIDATA'].astype(str).nunique())
    _hr_rec_res_v66_13 = float(to_num(_diag_rec_res_v66_13['CORTE_HR_CANDIDATO']).sum())

_status_pos_v66_13 = 'SEM_RESTRICAO_REC_RESIDUAL' if _qtd_rec_res_v66_13 == 0 else 'RESTRICAO_REC_RESIDUAL_A_ANALISAR'

RESUMO_VALIDACAO_POS_CORTE_M1 = pd.DataFrame([{
    'MES_REF': _mes_m1_v66_14,
    'QTD_ORIGEM_COMPONENTE_CORTE': _qtd_aplic_v66_14,
    'QTD_APLICACAO_PASSOU': _qtd_aplic_ok_v66_14,
    'PCS_CORTE_PLANEJADO': _pcs_plan_v66_14,
    'PCS_CORTE_APLICADO': _pcs_apl_v66_14,
    'DIF_MAX_PCS_APLICACAO': _dif_apl_v66_14,
    'QTD_RESTRICOES_REC_OBJETIVAS_POS_CORTE': _qtd_rec_res_v66_13,
    'CORTE_HR_REC_RESIDUAL_POS_CORTE': _hr_rec_res_v66_13,
    'STATUS_POS_CORTE': _status_pos_v66_13,
    'PCS_SOMA_RECS_DIAG': _pcs_soma_rec_diag_v66_14,
    'PCS_DUPLA_CONTAGEM_ELIMINADA': _pcs_dup_elim_v66_14,
    'HR_ALIVIO_GLOBAL_TOTAL': _hr_alivio_global_v66_14,
    'HR_EXCESSO_ALIVIO_GLOBAL_TOTAL': _hr_excesso_global_v66_14,
}])

print(
    '       V66 POS-CORTE M1 - candidatos REC objetivos:', _qtd_rec_res_v66_13,
    '| corte_hr_rec_residual:', round(_hr_rec_res_v66_13, 6),
    '| status:', _status_pos_v66_13,
)

if _qtd_rec_res_v66_13 > 0:
    print('       ATENCAO - existe restricao REC residual apos a primeira aplicacao real; analisar antes de automatizar nova rodada')
    display(_diag_rec_res_v66_13)
_TEMPO_RODADA1_M1 = time.perf_counter() - _TEMPO_RODADA1_M1_INICIO


# 04F. Diagnóstico comparativo das RECs — original x pós-corte

Etapa somente diagnóstica. Compara as restrições REC objetivas do M1 original com o estado após a primeira aplicação física dos cortes e a rerodagem da V66. Não altera a base, os cortes, a NEC nem o Excel final.


In [ ]:
# ============================================================
# 04F. DIAGNOSTICO COMPARATIVO REC ORIGINAL X POS-CORTE - M1
# ============================================================
# Objetivo:
# - comparar as RECs objetivas antes e depois da primeira aplicacao real;
# - classificar cada recurso como ELIMINADA, RESIDUAL ou NOVA;
# - medir a variacao do CORTE_HR por recurso;
# - nao alterar nenhuma regra ou estado operacional.


def _comparar_restricoes_rec_original_x_pos_corte_m1(
    validacao_original,
    validacao_pos_corte,
    tol_hr=None,
):
    if tol_hr is None:
        tol_hr = max(TOL * 100, 1e-6)

    cols_saida = [
        'MES_REF', 'ALOC_REC',
        'CORTE_HR_ORIGINAL', 'CORTE_HR_POS_CORTE',
        'REDUCAO_HR', 'VARIACAO_HR', 'PERC_REDUCAO_HR',
        'STATUS_REC_COMPARATIVO',
        'STATUS_VALIDACAO_ORIGINAL', 'STATUS_VALIDACAO_POS_CORTE',
        'QTD_ITENS_ORIGINAL', 'QTD_ITENS_POS_CORTE',
    ]

    def _preparar(df, sufixo):
        if not isinstance(df, pd.DataFrame) or df.empty:
            return pd.DataFrame(columns=[
                'MES_REF', 'ALOC_REC', f'CORTE_HR_{sufixo}',
                f'STATUS_VALIDACAO_{sufixo}', f'QTD_ITENS_{sufixo}'
            ])

        tmp = df.copy()
        if 'TIPO_RESTRICAO_CANDIDATA' in tmp.columns:
            tmp = tmp[tmp['TIPO_RESTRICAO_CANDIDATA'].astype(str).eq('REC')].copy()
        if tmp.empty:
            return pd.DataFrame(columns=[
                'MES_REF', 'ALOC_REC', f'CORTE_HR_{sufixo}',
                f'STATUS_VALIDACAO_{sufixo}', f'QTD_ITENS_{sufixo}'
            ])

        tmp['MES_REF'] = pd.to_datetime(tmp['MES_REF'], errors='coerce').dt.normalize()
        tmp['RESTRICAO_CANDIDATA'] = tmp['RESTRICAO_CANDIDATA'].astype(str)
        tmp['CORTE_HR_CANDIDATO'] = to_num(tmp['CORTE_HR_CANDIDATO'])
        if 'QTD_ITENS' not in tmp.columns:
            tmp['QTD_ITENS'] = 0
        tmp['QTD_ITENS'] = pd.to_numeric(tmp['QTD_ITENS'], errors='coerce').fillna(0).astype(int)
        if 'STATUS_VALIDACAO' not in tmp.columns:
            tmp['STATUS_VALIDACAO'] = ''

        # A funcao de validacao ja entrega uma linha por REC, mas o groupby
        # deixa o diagnostico robusto contra eventuais duplicidades futuras.
        agg = (
            tmp.groupby(['MES_REF', 'RESTRICAO_CANDIDATA'], as_index=False)
            .agg(
                CORTE_HR_CANDIDATO=('CORTE_HR_CANDIDATO', 'sum'),
                QTD_ITENS=('QTD_ITENS', 'sum'),
                STATUS_VALIDACAO=('STATUS_VALIDACAO', lambda s: 'PASSOU' if s.astype(str).eq('PASSOU').all() else 'FALHOU'),
            )
            .rename(columns={
                'RESTRICAO_CANDIDATA': 'ALOC_REC',
                'CORTE_HR_CANDIDATO': f'CORTE_HR_{sufixo}',
                'QTD_ITENS': f'QTD_ITENS_{sufixo}',
                'STATUS_VALIDACAO': f'STATUS_VALIDACAO_{sufixo}',
            })
        )
        return agg

    orig = _preparar(validacao_original, 'ORIGINAL')
    pos = _preparar(validacao_pos_corte, 'POS_CORTE')

    comp = orig.merge(pos, on=['MES_REF', 'ALOC_REC'], how='outer', validate='one_to_one')
    if comp.empty:
        return pd.DataFrame(columns=cols_saida), pd.DataFrame()

    for c in ['CORTE_HR_ORIGINAL', 'CORTE_HR_POS_CORTE']:
        comp[c] = to_num(comp[c])
    for c in ['QTD_ITENS_ORIGINAL', 'QTD_ITENS_POS_CORTE']:
        comp[c] = pd.to_numeric(comp[c], errors='coerce').fillna(0).astype(int)
    for c in ['STATUS_VALIDACAO_ORIGINAL', 'STATUS_VALIDACAO_POS_CORTE']:
        comp[c] = comp[c].astype('string').fillna('NAO_EXISTE').astype(str)

    tem_orig = comp['CORTE_HR_ORIGINAL'] > tol_hr
    tem_pos = comp['CORTE_HR_POS_CORTE'] > tol_hr

    comp['STATUS_REC_COMPARATIVO'] = np.select(
        [
            tem_orig & ~tem_pos,
            tem_orig & tem_pos,
            ~tem_orig & tem_pos,
        ],
        [
            'ORIGINAL_ELIMINADA',
            'ORIGINAL_RESIDUAL',
            'NOVA_POS_REOTIMIZACAO',
        ],
        default='SEM_CORTE_RELEVANTE',
    )

    comp['REDUCAO_HR'] = comp['CORTE_HR_ORIGINAL'] - comp['CORTE_HR_POS_CORTE']
    comp['VARIACAO_HR'] = comp['CORTE_HR_POS_CORTE'] - comp['CORTE_HR_ORIGINAL']
    comp['PERC_REDUCAO_HR'] = np.divide(
        comp['REDUCAO_HR'],
        comp['CORTE_HR_ORIGINAL'],
        out=np.full(len(comp), np.nan, dtype=float),
        where=comp['CORTE_HR_ORIGINAL'].to_numpy() > tol_hr,
    )

    ordem = {
        'ORIGINAL_RESIDUAL': 1,
        'NOVA_POS_REOTIMIZACAO': 2,
        'ORIGINAL_ELIMINADA': 3,
        'SEM_CORTE_RELEVANTE': 4,
    }
    comp['_ORDEM_STATUS'] = comp['STATUS_REC_COMPARATIVO'].map(ordem).fillna(9)
    comp = comp.sort_values(
        ['_ORDEM_STATUS', 'CORTE_HR_POS_CORTE', 'CORTE_HR_ORIGINAL'],
        ascending=[True, False, False],
    ).drop(columns=['_ORDEM_STATUS']).reset_index(drop=True)

    qtd_orig = int((comp['CORTE_HR_ORIGINAL'] > tol_hr).sum())
    qtd_pos = int((comp['CORTE_HR_POS_CORTE'] > tol_hr).sum())
    qtd_elim = int(comp['STATUS_REC_COMPARATIVO'].eq('ORIGINAL_ELIMINADA').sum())
    qtd_res = int(comp['STATUS_REC_COMPARATIVO'].eq('ORIGINAL_RESIDUAL').sum())
    qtd_nova = int(comp['STATUS_REC_COMPARATIVO'].eq('NOVA_POS_REOTIMIZACAO').sum())

    hr_orig = float(comp['CORTE_HR_ORIGINAL'].sum())
    hr_pos = float(comp['CORTE_HR_POS_CORTE'].sum())
    hr_reducao_liquida = hr_orig - hr_pos
    perc_reducao_total = (hr_reducao_liquida / hr_orig) if hr_orig > tol_hr else np.nan

    resumo = pd.DataFrame([{
        'MES_REF': comp['MES_REF'].dropna().min() if comp['MES_REF'].notna().any() else pd.NaT,
        'QTD_RECS_ORIGINAIS': qtd_orig,
        'QTD_RECS_POS_CORTE': qtd_pos,
        'QTD_ORIGINAIS_ELIMINADAS': qtd_elim,
        'QTD_ORIGINAIS_RESIDUAIS': qtd_res,
        'QTD_RECS_NOVAS_POS_REOTIMIZACAO': qtd_nova,
        'CORTE_HR_ORIGINAL_TOTAL': hr_orig,
        'CORTE_HR_POS_CORTE_TOTAL': hr_pos,
        'REDUCAO_LIQUIDA_HR': hr_reducao_liquida,
        'PERC_REDUCAO_TOTAL': perc_reducao_total,
    }])

    return comp[[c for c in cols_saida if c in comp.columns]], resumo


DIAGNOSTICO_COMPARATIVO_REC_M1, RESUMO_COMPARATIVO_REC_M1 = _comparar_restricoes_rec_original_x_pos_corte_m1(
    validacao_corte_restricao_m1,
    VALIDACAO_CORTE_RESTRICAO_POS_CORTE_M1,
)

RECS_ORIGINAIS_ELIMINADAS_M1 = DIAGNOSTICO_COMPARATIVO_REC_M1[
    DIAGNOSTICO_COMPARATIVO_REC_M1['STATUS_REC_COMPARATIVO'].eq('ORIGINAL_ELIMINADA')
].copy()

RECS_ORIGINAIS_RESIDUAIS_M1 = DIAGNOSTICO_COMPARATIVO_REC_M1[
    DIAGNOSTICO_COMPARATIVO_REC_M1['STATUS_REC_COMPARATIVO'].eq('ORIGINAL_RESIDUAL')
].copy()

RECS_NOVAS_POS_CORTE_M1 = DIAGNOSTICO_COMPARATIVO_REC_M1[
    DIAGNOSTICO_COMPARATIVO_REC_M1['STATUS_REC_COMPARATIVO'].eq('NOVA_POS_REOTIMIZACAO')
].copy()

if RESUMO_COMPARATIVO_REC_M1.empty:
    print('       COMPARATIVO REC ORIGINAL X POS-CORTE - sem restricoes para comparar')
else:
    _r = RESUMO_COMPARATIVO_REC_M1.iloc[0]
    print(
        '       COMPARATIVO REC ORIGINAL X POS-CORTE - originais:', int(_r['QTD_RECS_ORIGINAIS']),
        '| pos_corte:', int(_r['QTD_RECS_POS_CORTE']),
        '| eliminadas:', int(_r['QTD_ORIGINAIS_ELIMINADAS']),
        '| residuais:', int(_r['QTD_ORIGINAIS_RESIDUAIS']),
        '| novas:', int(_r['QTD_RECS_NOVAS_POS_REOTIMIZACAO']),
    )
    print(
        '       CORTE_HR COMPARATIVO - original:', round(float(_r['CORTE_HR_ORIGINAL_TOTAL']), 6),
        '| pos_corte:', round(float(_r['CORTE_HR_POS_CORTE_TOTAL']), 6),
        '| reducao_liquida:', round(float(_r['REDUCAO_LIQUIDA_HR']), 6),
        '| perc_reducao:', round(float(_r['PERC_REDUCAO_TOTAL']) * 100, 4), '%',
    )

    if not RECS_ORIGINAIS_RESIDUAIS_M1.empty:
        print('       RECS ORIGINAIS RESIDUAIS:')
        display(RECS_ORIGINAIS_RESIDUAIS_M1)

    if not RECS_NOVAS_POS_CORTE_M1.empty:
        print('       RECS NOVAS POS-REOTIMIZACAO:')
        display(RECS_NOVAS_POS_CORTE_M1)

    if not RECS_ORIGINAIS_ELIMINADAS_M1.empty:
        print('       RECS ORIGINAIS ELIMINADAS:')
        display(RECS_ORIGINAIS_ELIMINADAS_M1)


# 04G. Loop automatico controlado REC -> FER - M1

Respeita a ordem oficial: fecha restricoes de recurso (`ALOC_REC`) primeiro, depois ferramenta (`COD_FER_UNID`), retornando a REC se a rerodagem voltar a gerar restricao de recurso.


In [ ]:
# ============================================================
# 04G. LOOP AUTOMATICO CONTROLADO REC -> FER - M1
# ============================================================
# Objetivo:
# - preservar a primeira aplicacao REC ja validada;
# - continuar fechando REC enquanto existir CORTE_HR objetivo de recurso;
# - quando REC zerar, processar FER com os mesmos 9 criterios;
# - rerodar a V66 apos cada corte fisico e voltar para REC se necessario;
# - encerrar somente quando nao restar CORTE_HR objetivo em REC nem FER.
#
# Regras preservadas:
# - ordem oficial: ALOC_REC primeiro, COD_FER_UNID depois;
# - mesmos 9 criterios da Matriz para REC e FER;
# - corte fisico unico: nunca somar duas vezes o mesmo volume apenas porque
#   recurso e ferramenta estao simultaneamente envolvidos;
# - ORI_* nunca e sobrescrito;
# - NEC_PCS nunca e cortada diretamente;
# - MAX_ITER e somente blindagem de seguranca.
# - LIMIT_PCS/LOTE_MIN/QTD_EMB ficam fora das iteracoes principais da Matriz.

MATRIZ_CORTES_M1_MAX_ITER = 100
MATRIZ_CORTES_M1_TOL_HR = max(TOL * 100, 1e-6)

_LOG_TEMPO_PROCESSAMENTO_M1 = []
_TEMPO_LOOP_M1_INICIO = time.perf_counter()
if '_TEMPO_APLICACAO_ROD1_M1' in globals():
    _LOG_TEMPO_PROCESSAMENTO_M1.append({
        'ETAPA': 'APLICACAO_CORTES', 'RODADA_MATRIZ': 1, 'FASE': 'REC',
        'TEMPO_SEG': float(_TEMPO_APLICACAO_ROD1_M1), 'DETALHES': 'primeira aplicacao',
    })
if '_TEMPO_V66_ROD1_M1' in globals():
    _LOG_TEMPO_PROCESSAMENTO_M1.append({
        'ETAPA': 'V66_RERODAGEM_CONTINUA', 'RODADA_MATRIZ': 1, 'FASE': 'REC',
        'TEMPO_SEG': float(_TEMPO_V66_ROD1_M1), 'DETALHES': 'primeira rerodagem',
    })
if '_TEMPO_RODADA1_M1' in globals():
    _LOG_TEMPO_PROCESSAMENTO_M1.append({
        'ETAPA': 'RODADA_TOTAL', 'RODADA_MATRIZ': 1, 'FASE': 'REC',
        'TEMPO_SEG': float(_TEMPO_RODADA1_M1), 'DETALHES': 'rodada 1 completa',
    })

def _registrar_tempo_m1(etapa, inicio, rodada=0, fase='', detalhes=''):
    segundos = time.perf_counter() - inicio
    _LOG_TEMPO_PROCESSAMENTO_M1.append({
        'ETAPA': str(etapa),
        'RODADA_MATRIZ': int(rodada),
        'FASE': str(fase),
        'TEMPO_SEG': float(segundos),
        'DETALHES': str(detalhes),
    })
    return segundos


def _tag_rodada_m1(df, rodada, estado='', tipo_restricao=''):
    if not isinstance(df, pd.DataFrame) or df.empty:
        return pd.DataFrame()
    out = df.copy()
    out.insert(0, 'RODADA_MATRIZ', int(rodada))
    if estado:
        out.insert(1, 'ESTADO_MATRIZ', estado)
    if tipo_restricao:
        pos = 2 if estado else 1
        out.insert(pos, 'TIPO_RESTRICAO_PROCESSADA', str(tipo_restricao))
    return out


def _resumo_validacao_tipo_m1(validacao):
    if not isinstance(validacao, pd.DataFrame) or validacao.empty:
        return 0, 0.0, set()
    val = validacao.copy()
    if 'CORTE_HR_CANDIDATO' not in val.columns:
        return 0, 0.0, set()
    val['CORTE_HR_CANDIDATO'] = to_num(val['CORTE_HR_CANDIDATO'])
    val = val[val['CORTE_HR_CANDIDATO'] > MATRIZ_CORTES_M1_TOL_HR].copy()
    if val.empty:
        return 0, 0.0, set()
    restricoes = set(val['RESTRICAO_CANDIDATA'].astype(str))
    return len(restricoes), float(val['CORTE_HR_CANDIDATO'].sum()), restricoes


def _fingerprint_objetivos_m1(diag):
    if not isinstance(diag, pd.DataFrame) or diag.empty:
        return tuple()
    obj = diag[
        diag['STATUS_DIAGNOSTICO'].eq('CORTE_CANDIDATO_OBJETIVO')
        & diag['TIPO_RESTRICAO_CANDIDATA'].astype(str).str.upper().isin(['REC', 'FER'])
    ].copy()
    if obj.empty:
        return tuple()
    obj['CORTE_HR_CANDIDATO'] = to_num(obj['CORTE_HR_CANDIDATO'])
    fp = (
        obj.groupby(['TIPO_RESTRICAO_CANDIDATA', 'RESTRICAO_CANDIDATA'], as_index=False)
        .agg(CORTE_HR=('CORTE_HR_CANDIDATO', 'sum'))
        .sort_values(['TIPO_RESTRICAO_CANDIDATA', 'RESTRICAO_CANDIDATA'])
    )
    return tuple(
        (str(t).upper(), str(r), round(float(h), 9))
        for t, r, h in fp.itertuples(index=False, name=None)
    )


def _validar_status_nao_suportado_m1(diag, rodada):
    if not isinstance(diag, pd.DataFrame) or diag.empty:
        return

    obj = diag[diag['STATUS_DIAGNOSTICO'].eq('CORTE_CANDIDATO_OBJETIVO')].copy()
    if not obj.empty:
        tipos = set(obj['TIPO_RESTRICAO_CANDIDATA'].astype(str).str.upper())
        tipos_invalidos = sorted(t for t in tipos if t not in {'REC', 'FER'})
        if tipos_invalidos:
            raise RuntimeError(
                f'LOOP_MATRIZ_M1_TIPO_RESTRICAO_NAO_SUPORTADO_RODADA_{rodada}: {tipos_invalidos}'
            )

    # Bloqueio duplo e ambiguidade nao podem ser convertidos em dois cortes independentes.
    # Se aparecerem, parar para diagnostico especifico antes de materializar qualquer nova rodada.
    especiais = diag[
        diag['STATUS_DIAGNOSTICO'].isin([
            'CORTE_CANDIDATO_BLOQUEIO_DUPLO',
            'AMBIGUO_MULTIPLAS_RESTRICOES',
        ])
        & (to_num(diag['QTD_PENDENTE_FINAL']) > TOL)
    ].copy()
    if not especiais.empty:
        display(especiais)
        raise RuntimeError(
            f'LOOP_MATRIZ_M1_RESTRICAO_ESPECIAL_RODADA_{rodada}: '
            f'{sorted(especiais["STATUS_DIAGNOSTICO"].astype(str).unique().tolist())}. '
            'Nao gerar cortes independentes para o mesmo volume.'
        )


def validar_corte_hr_fer_m1(diagnostico_m1, rotas_m1, rastro):
    """Valida QUANTO CORTAR para restricoes objetivas de ferramenta (COD_FER_UNID).

    Espelha a validacao REC, trocando o eixo de capacidade:
    - ferramenta precisa estar esgotada;
    - recurso da rota final precisa continuar nao limitante;
    - CORTE_HR precisa reconciliar com falta final e excesso reconstruido.
    """
    cols_saida = [
        'MES_REF', 'TIPO_RESTRICAO_CANDIDATA', 'RESTRICAO_CANDIDATA',
        'QTD_ITENS', 'QTD_ITENS_RASTRO', 'QTD_ROTAS_FINAIS_RASTRO',
        'HOR_FER_RAW_RASTRO', 'HR_PRODUZIR_RAW_RASTRO',
        'CAP_FER_FINAL_RECONSTRUIDA', 'MAX_CAP_FER_FINAL',
        'DIF_CAP_FINAL_RECON_VS_SOLVER',
        'HR_FALTA_RASTRO', 'HR_DEMANDA_EFETIVA_RECONSTRUIDA',
        'EXCESSO_HR_RECONSTRUIDO', 'CORTE_HR_CANDIDATO',
        'DIF_CORTE_VS_RASTRO', 'DIF_CORTE_VS_EXCESSO',
        'MIN_CAP_REC_FINAL',
        'VALIDA_QTD_ITENS', 'VALIDA_ROTA_UNICA_FINAL',
        'VALIDA_CAP_FINAL_RECONSTRUIDA', 'VALIDA_FERRAMENTA_ESGOTADA',
        'VALIDA_RECURSO_NAO_LIMITANTE',
        'VALIDA_CORTE_VS_RASTRO', 'VALIDA_CORTE_VS_EXCESSO',
        'STATUS_VALIDACAO',
    ]

    if not isinstance(diagnostico_m1, pd.DataFrame) or diagnostico_m1.empty:
        return pd.DataFrame(columns=cols_saida)

    diag = diagnostico_m1.copy()
    diag = diag[
        diag['STATUS_DIAGNOSTICO'].eq('CORTE_CANDIDATO_OBJETIVO')
        & diag['TIPO_RESTRICAO_CANDIDATA'].astype(str).str.upper().eq('FER')
    ].copy()
    if diag.empty:
        return pd.DataFrame(columns=cols_saida)

    mes_ref = pd.to_datetime(diag['MES_REF'], errors='coerce').min()
    tol_hr = max(TOL * 100, 1e-6)

    cand = (
        diag.groupby(['TIPO_RESTRICAO_CANDIDATA', 'RESTRICAO_CANDIDATA'], as_index=False)
        .agg(
            QTD_ITENS=('ID_PROD_UNID_FAT', 'nunique'),
            CORTE_HR_CANDIDATO=('CORTE_HR_CANDIDATO', 'sum'),
        )
    )
    cand.insert(0, 'MES_REF', mes_ref)

    ids = set(diag['ID_PROD_UNID_FAT'].astype(str))

    ras = rastro.copy()
    ras['MES_REF'] = pd.to_datetime(ras['MES_REF'], errors='coerce')
    ras_m1 = ras[ras['MES_REF'].eq(mes_ref)].copy()
    if ras_m1.empty:
        raise RuntimeError('VALIDACAO_CORTE_FER_M1_SEM_RASTRO')

    ras_cand = ras_m1[ras_m1['ID_PROD_UNID_FAT'].astype(str).isin(ids)].copy()
    ras_cand['_RODADA_FINAL_ITEM'] = ras_cand.groupby('ID_PROD_UNID_FAT')['RODADA'].transform('max')
    ras_cand = ras_cand[ras_cand['RODADA'].eq(ras_cand['_RODADA_FINAL_ITEM'])].copy()
    ras_cand['QTD_FALTA_RASTRO'] = (
        to_num(ras_cand['QTD_SOLICITADA']) - to_num(ras_cand['QTD_PRODUZIR_SOLVER'])
    ).clip(lower=0.0)
    ras_cand['HR_FALTA_RASTRO'] = np.divide(
        ras_cand['QTD_FALTA_RASTRO'],
        to_num(ras_cand['PCS_HORA']),
        out=np.zeros(len(ras_cand), dtype=float),
        where=to_num(ras_cand['PCS_HORA']).to_numpy() > TOL,
    )
    ras_cand = ras_cand[ras_cand['QTD_FALTA_RASTRO'] > TOL].copy()

    ras_fer = (
        ras_cand.groupby('COD_FER_UNID', as_index=False)
        .agg(
            QTD_ITENS_RASTRO=('ID_PROD_UNID_FAT', 'nunique'),
            QTD_ROTAS_FINAIS_RASTRO=('ID_PROD_UNID_FAT', 'size'),
            HR_FALTA_RASTRO=('HR_FALTA_RASTRO', 'sum'),
        )
        .rename(columns={'COD_FER_UNID': 'RESTRICAO_CANDIDATA'})
    )

    ras_m1['CAP_RESTANTE_COD_FER_ANTES'] = to_num(ras_m1['CAP_RESTANTE_COD_FER_ANTES'])
    ras_m1['HR_PRODUZIR_SOLVER'] = to_num(ras_m1['HR_PRODUZIR_SOLVER'])
    fer_raw = (
        ras_m1.groupby('COD_FER_UNID', as_index=False)
        .agg(
            HOR_FER_RAW_RASTRO=('CAP_RESTANTE_COD_FER_ANTES', 'max'),
            HR_PRODUZIR_RAW_RASTRO=('HR_PRODUZIR_SOLVER', 'sum'),
        )
        .rename(columns={'COD_FER_UNID': 'RESTRICAO_CANDIDATA'})
    )
    fer_raw['CAP_FER_FINAL_RECONSTRUIDA'] = (
        fer_raw['HOR_FER_RAW_RASTRO'] - fer_raw['HR_PRODUZIR_RAW_RASTRO']
    ).clip(lower=0.0)

    rot = rotas_m1.copy()
    rot['MES_REF'] = pd.to_datetime(rot['MES_REF'], errors='coerce')
    rot = rot[
        rot['MES_REF'].eq(mes_ref)
        & rot['ID_PROD_UNID_FAT'].astype(str).isin(ids)
        & rot['FOI_ULTIMA_RODADA'].fillna(False)
        & (to_num(rot['QTD_FALTA_ROTA']) > TOL)
    ].copy()
    rot_estado = (
        rot.groupby('COD_FER_UNID', as_index=False)
        .agg(
            MIN_CAP_REC_FINAL=('CAP_RESTANTE_ALOC_REC_FINAL', 'min'),
            MAX_CAP_FER_FINAL=('CAP_RESTANTE_COD_FER_FINAL', 'max'),
        )
        .rename(columns={'COD_FER_UNID': 'RESTRICAO_CANDIDATA'})
    )

    val = (
        cand
        .merge(ras_fer, on='RESTRICAO_CANDIDATA', how='left', validate='one_to_one')
        .merge(fer_raw, on='RESTRICAO_CANDIDATA', how='left', validate='one_to_one')
        .merge(rot_estado, on='RESTRICAO_CANDIDATA', how='left', validate='one_to_one')
    )

    for c in [
        'QTD_ITENS_RASTRO', 'QTD_ROTAS_FINAIS_RASTRO', 'HR_FALTA_RASTRO',
        'HOR_FER_RAW_RASTRO', 'HR_PRODUZIR_RAW_RASTRO', 'CAP_FER_FINAL_RECONSTRUIDA',
        'MIN_CAP_REC_FINAL', 'MAX_CAP_FER_FINAL',
    ]:
        val[c] = to_num(val[c])

    val['DIF_CAP_FINAL_RECON_VS_SOLVER'] = (
        val['CAP_FER_FINAL_RECONSTRUIDA'] - val['MAX_CAP_FER_FINAL']
    )
    val['HR_DEMANDA_EFETIVA_RECONSTRUIDA'] = (
        val['HR_PRODUZIR_RAW_RASTRO'] + val['HR_FALTA_RASTRO']
    )
    val['EXCESSO_HR_RECONSTRUIDO'] = (
        val['HR_DEMANDA_EFETIVA_RECONSTRUIDA'] - val['HOR_FER_RAW_RASTRO']
    ).clip(lower=0.0)
    val['DIF_CORTE_VS_RASTRO'] = val['CORTE_HR_CANDIDATO'] - val['HR_FALTA_RASTRO']
    val['DIF_CORTE_VS_EXCESSO'] = val['CORTE_HR_CANDIDATO'] - val['EXCESSO_HR_RECONSTRUIDO']

    val['VALIDA_QTD_ITENS'] = val['QTD_ITENS'].eq(val['QTD_ITENS_RASTRO'])
    val['VALIDA_ROTA_UNICA_FINAL'] = val['QTD_ROTAS_FINAIS_RASTRO'].eq(val['QTD_ITENS_RASTRO'])
    val['VALIDA_CAP_FINAL_RECONSTRUIDA'] = val['DIF_CAP_FINAL_RECON_VS_SOLVER'].abs() <= tol_hr
    val['VALIDA_FERRAMENTA_ESGOTADA'] = val['MAX_CAP_FER_FINAL'].abs() <= tol_hr
    val['VALIDA_RECURSO_NAO_LIMITANTE'] = val['MIN_CAP_REC_FINAL'] > TOL
    val['VALIDA_CORTE_VS_RASTRO'] = val['DIF_CORTE_VS_RASTRO'].abs() <= tol_hr
    val['VALIDA_CORTE_VS_EXCESSO'] = val['DIF_CORTE_VS_EXCESSO'].abs() <= tol_hr

    checks = [
        'VALIDA_QTD_ITENS', 'VALIDA_ROTA_UNICA_FINAL',
        'VALIDA_CAP_FINAL_RECONSTRUIDA', 'VALIDA_FERRAMENTA_ESGOTADA',
        'VALIDA_RECURSO_NAO_LIMITANTE',
        'VALIDA_CORTE_VS_RASTRO', 'VALIDA_CORTE_VS_EXCESSO',
    ]
    val['STATUS_VALIDACAO'] = np.where(val[checks].all(axis=1), 'PASSOU', 'FALHOU')
    val = val.sort_values('CORTE_HR_CANDIDATO', ascending=False).reset_index(drop=True)
    return val[[c for c in cols_saida if c in val.columns]]


def gerar_de_onde_cortar_fer_m1(
    validacao_fer,
    diagnostico_m1,
    rastro_m1,
    bd_entrada,
    nec_consolidada,
    estrutura_exp,
):
    """Reutiliza a Matriz REC no eixo COD_FER_UNID, preservando os mesmos 9 criterios."""
    if not isinstance(validacao_fer, pd.DataFrame) or validacao_fer.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    val_fake = validacao_fer.copy()
    val_fake['TIPO_RESTRICAO_CANDIDATA'] = 'REC'

    ras_fake = rastro_m1.copy()
    if 'COD_FER_UNID' not in ras_fake.columns:
        raise RuntimeError('DE_ONDE_CORTAR_FER_SEM_COD_FER_UNID_NO_RASTRO')
    ras_fake['ALOC_REC'] = ras_fake['COD_FER_UNID'].astype(str)

    mat, validacao, pool = gerar_de_onde_cortar_m1(
        val_fake,
        diagnostico_m1,
        ras_fake,
        bd_entrada,
        nec_consolidada,
        estrutura_exp,
    )

    for df in [mat, validacao, pool]:
        if isinstance(df, pd.DataFrame) and not df.empty:
            df['TIPO_RESTRICAO_PROCESSADA'] = 'FER'
            df['COD_FER_UNID_ALVO'] = df['ALOC_REC'].astype(str)
    return mat, validacao, pool


def _validacoes_estado_m1(diag, diag_rotas, rastro):
    val_rec = validar_corte_hr_por_restricao_m1(diag, diag_rotas, rastro)
    val_fer = validar_corte_hr_fer_m1(diag, diag_rotas, rastro)
    return val_rec, val_fer


def _combinar_validacoes_m1(val_rec, val_fer):
    partes = []
    if isinstance(val_rec, pd.DataFrame) and not val_rec.empty:
        partes.append(val_rec.copy())
    if isinstance(val_fer, pd.DataFrame) and not val_fer.empty:
        partes.append(val_fer.copy())
    return pd.concat(partes, ignore_index=True, sort=False) if partes else pd.DataFrame()


# ------------------------------------------------------------
# 1. Estado inicial = fotografia apos a primeira aplicacao REC
# ------------------------------------------------------------
_loop_bd = BD_M1_POS_CORTE_MATRIZ.copy()
_loop_resultado = RESULTADO_V66_POS_CORTE_M1
_loop_nec = NEC_CONSOLIDADA_POS_CORTE_M1.copy()
_loop_estrutura = ESTRUTURA_EXPLODIDA_POS_CORTE_M1.copy()
_loop_rastro = RASTRO_RATEIO_POS_CORTE_M1.copy()
_loop_diag = DIAGNOSTICO_MATRIZ_POS_CORTE_M1.copy()
_loop_diag_rotas = DIAGNOSTICO_MATRIZ_ROTAS_POS_CORTE_M1.copy()
_loop_validacao_rec, _loop_validacao_fer = _validacoes_estado_m1(
    _loop_diag,
    _loop_diag_rotas,
    _loop_rastro,
)

_validar_status_nao_suportado_m1(_loop_diag, 1)

_logs_resumo = []
_logs_de_onde = []
_logs_aplicacao = []
_logs_alivio = []
_logs_restricoes = []

# Fotografia original REC antes de qualquer corte = rodada 0.
_logs_restricoes.append(_tag_rodada_m1(validacao_corte_restricao_m1, 0, 'ANTES_DOS_CORTES', 'REC'))
# Fotografia apos a primeira aplicacao = rodada 1.
_logs_restricoes.append(_tag_rodada_m1(_loop_validacao_rec, 1, 'POS_RODADA', 'REC'))
_logs_restricoes.append(_tag_rodada_m1(_loop_validacao_fer, 1, 'POS_RODADA', 'FER'))

_qtd_rec_orig, _hr_rec_orig, _recs_orig = _resumo_validacao_tipo_m1(validacao_corte_restricao_m1)
_qtd_rec_pos1, _hr_rec_pos1, _recs_pos1 = _resumo_validacao_tipo_m1(_loop_validacao_rec)
_qtd_fer_pos1, _hr_fer_pos1, _fers_pos1 = _resumo_validacao_tipo_m1(_loop_validacao_fer)
_pcs_apl_rod1 = float(to_num(AUDITORIA_APLICACAO_CORTES_M1['PCS_CORTE_APLICADO']).sum()) if not AUDITORIA_APLICACAO_CORTES_M1.empty else 0.0
_pcs_dup_rod1 = float(to_num(AUDITORIA_APLICACAO_CORTES_M1['PCS_DUPLA_CONTAGEM_ELIMINADA']).sum()) if not AUDITORIA_APLICACAO_CORTES_M1.empty else 0.0

_logs_resumo.append({
    'RODADA_MATRIZ': 1,
    'TIPO_RESTRICAO_PROCESSADA': 'REC',
    'QTD_REC_ANTES': _qtd_rec_orig,
    'QTD_REC_DEPOIS': _qtd_rec_pos1,
    'QTD_FER_DEPOIS': _qtd_fer_pos1,
    'CORTE_HR_REC_ANTES': _hr_rec_orig,
    'CORTE_HR_REC_DEPOIS': _hr_rec_pos1,
    'CORTE_HR_FER_DEPOIS': _hr_fer_pos1,
    'CORTE_HR_TOTAL_DEPOIS': _hr_rec_pos1 + _hr_fer_pos1,
    'PCS_CORTE_APLICADO': _pcs_apl_rod1,
    'PCS_DUPLA_CONTAGEM_ELIMINADA': _pcs_dup_rod1,
    'STATUS_RODADA': 'CONCLUIDA',
})

# Fingerprints impedem ciclo de estado repetido sem exigir monotonicidade estrita
# entre REC e FER durante a estabilizacao.
_estados_vistos = {_fingerprint_objetivos_m1(_loop_diag)}

# ------------------------------------------------------------
# 2. Rodadas automaticas: sempre REC antes de FER
# ------------------------------------------------------------
STATUS_LOOP_MATRIZ_M1 = 'EM_PROCESSAMENTO'
RODADA_FINAL_MATRIZ_M1 = 1

for _rodada in range(2, MATRIZ_CORTES_M1_MAX_ITER + 1):
    _tempo_rodada_inicio = time.perf_counter()
    _tempo_quanto_inicio = time.perf_counter()
    _validar_status_nao_suportado_m1(_loop_diag, _rodada - 1)

    _qtd_rec_antes, _hr_rec_antes, _recs_antes = _resumo_validacao_tipo_m1(_loop_validacao_rec)
    _qtd_fer_antes, _hr_fer_antes, _fers_antes = _resumo_validacao_tipo_m1(_loop_validacao_fer)
    _hr_total_antes = _hr_rec_antes + _hr_fer_antes

    if _hr_total_antes <= MATRIZ_CORTES_M1_TOL_HR or (_qtd_rec_antes + _qtd_fer_antes) == 0:
        STATUS_LOOP_MATRIZ_M1 = 'FECHOU_CORTE_HR_ZERO'
        break

    # Ordem oficial: enquanto houver REC, FER aguarda.
    if _qtd_rec_antes > 0 and _hr_rec_antes > MATRIZ_CORTES_M1_TOL_HR:
        _tipo_processar = 'REC'
        _validacao_atual = _loop_validacao_rec.copy()
    else:
        _tipo_processar = 'FER'
        _validacao_atual = _loop_validacao_fer.copy()

    if _validacao_atual.empty or not _validacao_atual['STATUS_VALIDACAO'].eq('PASSOU').all():
        _falhas = _validacao_atual[
            ~_validacao_atual['STATUS_VALIDACAO'].eq('PASSOU')
        ].copy() if not _validacao_atual.empty else pd.DataFrame()
        if not _falhas.empty:
            display(_falhas)
        raise RuntimeError(
            f'LOOP_MATRIZ_M1_VALIDACAO_QUANTO_CORTAR_{_tipo_processar}_FALHOU_RODADA_{_rodada}'
        )

    _tempo_quanto = _registrar_tempo_m1(
        'QUANTO_CORTAR_VALIDACAO', _tempo_quanto_inicio, _rodada, _tipo_processar,
        f'rec={_qtd_rec_antes};fer={_qtd_fer_antes};hr_total={_hr_total_antes:.9f}',
    )

    print(
        '       LOOP MATRIZ M1 - rodada:', _rodada,
        '| fase:', _tipo_processar,
        '| recs_antes:', _qtd_rec_antes,
        '| fers_antes:', _qtd_fer_antes,
        '| corte_hr_rec:', round(_hr_rec_antes, 6),
        '| corte_hr_fer:', round(_hr_fer_antes, 6),
    )

    # --------------------------------------------------------
    # A. DE ONDE CORTAR da fase atual
    # --------------------------------------------------------
    _tempo_de_onde_inicio = time.perf_counter()
    if _tipo_processar == 'REC':
        _mat_corte, _val_de_onde, _pool_cortavel = gerar_de_onde_cortar_m1(
            _validacao_atual,
            _loop_diag,
            _loop_rastro,
            _loop_bd,
            _loop_nec,
            _loop_estrutura,
        )
    else:
        _mat_corte, _val_de_onde, _pool_cortavel = gerar_de_onde_cortar_fer_m1(
            _validacao_atual,
            _loop_diag,
            _loop_rastro,
            _loop_bd,
            _loop_nec,
            _loop_estrutura,
        )

    if _val_de_onde.empty:
        raise RuntimeError(
            f'LOOP_MATRIZ_M1_DE_ONDE_CORTAR_{_tipo_processar}_VAZIO_RODADA_{_rodada}'
        )

    _qtd_de_onde = len(_val_de_onde)
    _qtd_de_onde_ok = int(_val_de_onde['STATUS_VALIDACAO'].eq('PASSOU').sum())
    _hr_alvo_de_onde = float(to_num(_val_de_onde['CORTE_HR_ALVO']).sum())
    _hr_atrib_de_onde = float(to_num(_val_de_onde['HR_CORTE_ATRIBUIDO']).sum())
    _dif_de_onde = float(to_num(_val_de_onde['DIF_HR_CORTE']).abs().max())

    print(
        '       DE ONDE CORTAR LOOP - rodada:', _rodada,
        '| fase:', _tipo_processar,
        '| restricoes:', _qtd_de_onde,
        '| passou:', _qtd_de_onde_ok,
        '| falhou:', _qtd_de_onde - _qtd_de_onde_ok,
        '| corte_hr_alvo:', round(_hr_alvo_de_onde, 6),
        '| hr_atribuido:', round(_hr_atrib_de_onde, 6),
        '| dif_max:', round(_dif_de_onde, 9),
    )

    if _qtd_de_onde_ok != _qtd_de_onde:
        display(_val_de_onde.loc[~_val_de_onde['STATUS_VALIDACAO'].eq('PASSOU')])
        raise RuntimeError(
            f'LOOP_MATRIZ_M1_DE_ONDE_CORTAR_{_tipo_processar}_NAO_FECHOU_RODADA_{_rodada}'
        )

    _logs_de_onde.append(_tag_rodada_m1(_val_de_onde, _rodada, 'DE_ONDE_CORTAR', _tipo_processar))
    _tempo_de_onde = _registrar_tempo_m1('DE_ONDE_CORTAR', _tempo_de_onde_inicio, _rodada, _tipo_processar, f'restricoes={_qtd_de_onde}')

    # --------------------------------------------------------
    # B. Aplicacao fisica unica no conjunto de restricoes da fase
    # --------------------------------------------------------
    _tempo_aplicacao_inicio = time.perf_counter()
    _bd_novo, _aud_aplic, _val_sobrecorte, _cortes_cons, _val_alivio = _aplicar_cortes_matriz_m1(
        _loop_bd,
        _mat_corte,
        _pool_cortavel,
        _val_de_onde,
        aplicar_regras_industriais=False,
    )

    _qtd_aplic = len(_aud_aplic)
    _qtd_aplic_ok = int(_aud_aplic['STATUS_APLICACAO'].eq('PASSOU').sum()) if _qtd_aplic else 0
    _pcs_aplic = float(to_num(_aud_aplic['PCS_CORTE_APLICADO']).sum()) if _qtd_aplic else 0.0
    _pcs_dup = float(to_num(_aud_aplic['PCS_DUPLA_CONTAGEM_ELIMINADA']).sum()) if _qtd_aplic else 0.0
    _dif_aplic = float(to_num(_aud_aplic['DIF_PCS_PLANEJADO_VS_APLICADO']).abs().max()) if _qtd_aplic else 0.0

    print(
        '       APLICACAO GLOBAL LOOP - rodada:', _rodada,
        '| fase:', _tipo_processar,
        '| origem+componente:', _qtd_aplic,
        '| passou:', _qtd_aplic_ok,
        '| pcs_aplicado:', round(_pcs_aplic, 6),
        '| dupla_contagem_eliminada:', round(_pcs_dup, 6),
        '| dif_max:', round(_dif_aplic, 9),
    )

    if _qtd_aplic_ok != _qtd_aplic:
        display(_aud_aplic.loc[~_aud_aplic['STATUS_APLICACAO'].eq('PASSOU')])
        raise RuntimeError(
            f'LOOP_MATRIZ_M1_APLICACAO_{_tipo_processar}_NAO_FECHOU_RODADA_{_rodada}'
        )

    _logs_aplicacao.append(_tag_rodada_m1(_aud_aplic, _rodada, 'APLICACAO', _tipo_processar))
    _logs_alivio.append(_tag_rodada_m1(_val_alivio, _rodada, 'ALIVIO_GLOBAL', _tipo_processar))
    _tempo_aplicacao = _registrar_tempo_m1('APLICACAO_CORTES', _tempo_aplicacao_inicio, _rodada, _tipo_processar, f'pcs={_pcs_aplic:.6f}')

    # --------------------------------------------------------
    # C. Rerodar V66 e recalcular os dois eixos de restricao
    # --------------------------------------------------------
    _tempo_v66_inicio = time.perf_counter()
    _resultado_novo = _executar_motor_periodo_v58(_bd_novo.copy(), aplicar_regras_industriais=False)
    _nec_novo = _resultado_novo.get('nec_consolidada_solver', pd.DataFrame()).copy()
    _estrutura_nova = _resultado_novo.get('estrutura_explodida', pd.DataFrame()).copy()
    _rastro_novo = _resultado_novo.get('rastro_rateio', pd.DataFrame()).copy()
    _diag_novo = _resultado_novo.get('diagnostico_matriz', pd.DataFrame()).copy()
    _diag_rotas_novo = _resultado_novo.get('diagnostico_matriz_rotas', pd.DataFrame()).copy()

    _tempo_v66 = _registrar_tempo_m1('V66_RERODAGEM_CONTINUA', _tempo_v66_inicio, _rodada, _tipo_processar)
    _tempo_validacao_pos_inicio = time.perf_counter()

    _validar_status_nao_suportado_m1(_diag_novo, _rodada)
    _validacao_rec_nova, _validacao_fer_nova = _validacoes_estado_m1(
        _diag_novo,
        _diag_rotas_novo,
        _rastro_novo,
    )

    for _tipo_val, _val_nova in [('REC', _validacao_rec_nova), ('FER', _validacao_fer_nova)]:
        if not _val_nova.empty and not _val_nova['STATUS_VALIDACAO'].eq('PASSOU').all():
            _falhas_novas = _val_nova[~_val_nova['STATUS_VALIDACAO'].eq('PASSOU')].copy()
            display(_falhas_novas)
            raise RuntimeError(
                f'LOOP_MATRIZ_M1_VALIDACAO_{_tipo_val}_POS_RODADA_FALHOU_{_rodada}: '
                f'{len(_falhas_novas)} restricao(oes).'
            )

    _qtd_rec_depois, _hr_rec_depois, _recs_depois = _resumo_validacao_tipo_m1(_validacao_rec_nova)
    _qtd_fer_depois, _hr_fer_depois, _fers_depois = _resumo_validacao_tipo_m1(_validacao_fer_nova)
    _hr_total_depois = _hr_rec_depois + _hr_fer_depois

    _logs_restricoes.append(_tag_rodada_m1(_validacao_rec_nova, _rodada, 'POS_RODADA', 'REC'))
    _logs_restricoes.append(_tag_rodada_m1(_validacao_fer_nova, _rodada, 'POS_RODADA', 'FER'))
    _tempo_validacao_pos = _registrar_tempo_m1(
        'VALIDACAO_POS_V66', _tempo_validacao_pos_inicio, _rodada, _tipo_processar,
        f'rec={_qtd_rec_depois};fer={_qtd_fer_depois};hr_total={_hr_total_depois:.9f}',
    )
    _tempo_rodada = _registrar_tempo_m1(
        'RODADA_TOTAL', _tempo_rodada_inicio, _rodada, _tipo_processar,
        f'hr_antes={_hr_total_antes:.9f};hr_depois={_hr_total_depois:.9f}',
    )

    _logs_resumo.append({
        'RODADA_MATRIZ': _rodada,
        'TIPO_RESTRICAO_PROCESSADA': _tipo_processar,
        'QTD_REC_ANTES': _qtd_rec_antes,
        'QTD_FER_ANTES': _qtd_fer_antes,
        'QTD_REC_DEPOIS': _qtd_rec_depois,
        'QTD_FER_DEPOIS': _qtd_fer_depois,
        'CORTE_HR_REC_ANTES': _hr_rec_antes,
        'CORTE_HR_FER_ANTES': _hr_fer_antes,
        'CORTE_HR_REC_DEPOIS': _hr_rec_depois,
        'CORTE_HR_FER_DEPOIS': _hr_fer_depois,
        'CORTE_HR_TOTAL_ANTES': _hr_total_antes,
        'CORTE_HR_TOTAL_DEPOIS': _hr_total_depois,
        'PCS_CORTE_APLICADO': _pcs_aplic,
        'PCS_DUPLA_CONTAGEM_ELIMINADA': _pcs_dup,
        'STATUS_RODADA': 'CONCLUIDA',
        'TEMPO_QUANTO_CORTAR_SEG': _tempo_quanto,
        'TEMPO_DE_ONDE_CORTAR_SEG': _tempo_de_onde,
        'TEMPO_APLICACAO_SEG': _tempo_aplicacao,
        'TEMPO_V66_SEG': _tempo_v66,
        'TEMPO_VALIDACAO_POS_SEG': _tempo_validacao_pos,
        'TEMPO_RODADA_SEG': _tempo_rodada,
    })

    print(
        '       RESULTADO LOOP M1 - rodada:', _rodada,
        '| fase:', _tipo_processar,
        '| recs_depois:', _qtd_rec_depois,
        '| fers_depois:', _qtd_fer_depois,
        '| corte_hr_rec:', round(_hr_rec_depois, 6),
        '| corte_hr_fer:', round(_hr_fer_depois, 6),
        '| corte_hr_total:', round(_hr_total_depois, 6),
        '| tempo_rodada_s:', round(_tempo_rodada, 2),
        '| tempo_v66_s:', round(_tempo_v66, 2),
    )

    # Atualiza o estado somente depois de todas as validacoes da rodada.
    _loop_bd = _bd_novo
    _loop_resultado = _resultado_novo
    _loop_nec = _nec_novo
    _loop_estrutura = _estrutura_nova
    _loop_rastro = _rastro_novo
    _loop_diag = _diag_novo
    _loop_diag_rotas = _diag_rotas_novo
    _loop_validacao_rec = _validacao_rec_nova
    _loop_validacao_fer = _validacao_fer_nova
    RODADA_FINAL_MATRIZ_M1 = _rodada

    if _hr_total_depois <= MATRIZ_CORTES_M1_TOL_HR or (_qtd_rec_depois + _qtd_fer_depois) == 0:
        STATUS_LOOP_MATRIZ_M1 = 'FECHOU_CORTE_HR_ZERO'
        break

    _fp_novo = _fingerprint_objetivos_m1(_loop_diag)
    if _fp_novo in _estados_vistos:
        STATUS_LOOP_MATRIZ_M1 = 'ESTADO_REPETIDO_SEM_CONVERGENCIA'
        raise RuntimeError(
            f'LOOP_MATRIZ_M1_ESTADO_REPETIDO_RODADA_{_rodada}: '
            'o conjunto de restricoes objetivas REC/FER repetiu um estado anterior.'
        )
    _estados_vistos.add(_fp_novo)
else:
    STATUS_LOOP_MATRIZ_M1 = 'MAX_ITER_ATINGIDO'

# ------------------------------------------------------------
# 3. Consolida logs e publica o estado final do M1
# ------------------------------------------------------------
LOG_TEMPO_PROCESSAMENTO_M1 = pd.DataFrame(_LOG_TEMPO_PROCESSAMENTO_M1)
LOG_MATRIZ_CORTES_M1 = pd.DataFrame(_logs_resumo)
LOG_DE_ONDE_CORTAR_M1 = pd.concat(_logs_de_onde, ignore_index=True, sort=False) if _logs_de_onde else pd.DataFrame()
LOG_APLICACAO_CORTES_M1 = pd.concat(_logs_aplicacao, ignore_index=True, sort=False) if _logs_aplicacao else pd.DataFrame()
LOG_ALIVIO_GLOBAL_M1 = pd.concat(_logs_alivio, ignore_index=True, sort=False) if _logs_alivio else pd.DataFrame()
LOG_RESTRICOES_M1 = pd.concat(
    [x for x in _logs_restricoes if isinstance(x, pd.DataFrame) and not x.empty],
    ignore_index=True,
    sort=False,
) if any(isinstance(x, pd.DataFrame) and not x.empty for x in _logs_restricoes) else pd.DataFrame()

BD_M1_FINAL_MATRIZ = _loop_bd.copy()
RESULTADO_V66_FINAL_M1 = _loop_resultado
NEC_CONSOLIDADA_FINAL_M1 = _loop_nec.copy()
ESTRUTURA_EXPLODIDA_FINAL_M1 = _loop_estrutura.copy()
RASTRO_RATEIO_FINAL_M1 = _loop_rastro.copy()
DIAGNOSTICO_MATRIZ_FINAL_M1 = _loop_diag.copy()
DIAGNOSTICO_MATRIZ_ROTAS_FINAL_M1 = _loop_diag_rotas.copy()
VALIDACAO_CORTE_REC_FINAL_M1 = _loop_validacao_rec.copy()
VALIDACAO_CORTE_FER_FINAL_M1 = _loop_validacao_fer.copy()
VALIDACAO_CORTE_RESTRICAO_FINAL_M1 = _combinar_validacoes_m1(
    VALIDACAO_CORTE_REC_FINAL_M1,
    VALIDACAO_CORTE_FER_FINAL_M1,
)

_QTD_RECS_FINAL_M1, _CORTE_HR_REC_FINAL_M1, _RECS_FINAL_M1 = _resumo_validacao_tipo_m1(
    VALIDACAO_CORTE_REC_FINAL_M1
)
_QTD_FERS_FINAL_M1, _CORTE_HR_FER_FINAL_M1, _FERS_FINAL_M1 = _resumo_validacao_tipo_m1(
    VALIDACAO_CORTE_FER_FINAL_M1
)
_CORTE_HR_FINAL_M1 = _CORTE_HR_REC_FINAL_M1 + _CORTE_HR_FER_FINAL_M1


_tempo_loop_total_m1 = time.perf_counter() - _TEMPO_LOOP_M1_INICIO
_LOG_TEMPO_PROCESSAMENTO_M1.append({
    'ETAPA': 'LOOP_MATRIZ_TOTAL',
    'RODADA_MATRIZ': int(RODADA_FINAL_MATRIZ_M1),
    'FASE': 'M1',
    'TEMPO_SEG': float(_tempo_loop_total_m1),
    'DETALHES': f'status={STATUS_LOOP_MATRIZ_M1};hr_final={_CORTE_HR_FINAL_M1:.9f}',
})
LOG_TEMPO_PROCESSAMENTO_M1 = pd.DataFrame(_LOG_TEMPO_PROCESSAMENTO_M1)
_TEMPO_MATRIZ_CORTES_M1 = _finalizar_etapa_v58('MATRIZ_CORTES_M1')

print(
    '       TEMPO LOOP MATRIZ M1 - total_s:', round(_tempo_loop_total_m1, 2),
    '| rodadas_executadas:', RODADA_FINAL_MATRIZ_M1,
)
if not LOG_TEMPO_PROCESSAMENTO_M1.empty:
    _tempo_por_etapa_m1 = (
        LOG_TEMPO_PROCESSAMENTO_M1
        .groupby('ETAPA', as_index=False)['TEMPO_SEG']
        .sum()
        .sort_values('TEMPO_SEG', ascending=False)
    )
    display(_tempo_por_etapa_m1)

print(
    '       LOOP AUTOMATICO M1 FINAL - status:', STATUS_LOOP_MATRIZ_M1,
    '| rodada_final:', RODADA_FINAL_MATRIZ_M1,
    '| recs_residuais:', _QTD_RECS_FINAL_M1,
    '| fers_residuais:', _QTD_FERS_FINAL_M1,
    '| corte_hr_rec:', round(_CORTE_HR_REC_FINAL_M1, 9),
    '| corte_hr_fer:', round(_CORTE_HR_FER_FINAL_M1, 9),
    '| corte_hr_total:', round(_CORTE_HR_FINAL_M1, 9),
)

if STATUS_LOOP_MATRIZ_M1 == 'FECHOU_CORTE_HR_ZERO':
    print('       M1 - LOOP REC/FER FECHOU CORTE_HR = 0; seguir para bateria final de validacao')
elif STATUS_LOOP_MATRIZ_M1 == 'MAX_ITER_ATINGIDO' and _CORTE_HR_FINAL_M1 > MATRIZ_CORTES_M1_TOL_HR:
    raise RuntimeError(
        f'LOOP_MATRIZ_M1_MAX_ITER_ATINGIDO: rodadas={MATRIZ_CORTES_M1_MAX_ITER}; '
        f'recs_residuais={_QTD_RECS_FINAL_M1}; fers_residuais={_QTD_FERS_FINAL_M1}; '
        f'corte_hr_residual={_CORTE_HR_FINAL_M1:.9f}'
    )


# 04. Montagem e exportação do resultado


In [ ]:
print("[03/04] Gravando arquivo final...")
_iniciar_etapa_v58("EXPORTACAO_EXCEL")
# ============================================================
# 07. EXPORTAÇÃO
# ============================================================



# Simplificação rotina:
# - mantém a limpeza da guia 15: não leva os fatores FATOR_FINAL_VIAVEL_ORIGEM e FATOR_AJUSTE_SOBRE_SOLVER;
# - as guias 17_RESUMO_REALOCACAO e 17_RESUMO_REALOCACAO_POOL seguem removidas do arquivo final;
capacidade_liberada_pf_export = capacidade_liberada_pf.drop(
    columns=[col for col in ["FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER"] if col in capacidade_liberada_pf.columns]
).copy()

OUTPUT_SOLVER_FILE.parent.mkdir(parents=True, exist_ok=True)

_GUIAS_DADOS_EXPORTAR = {
    "01_RESUMO_RECURSOS": resumo_recursos,
    "02_RESUMO_FERRAMENTAS": resumo_ferramentas,
    "03_PLANO_PRODUCAO": plano_producao,
    "04_NAO_ATEND_ITEM": nao_atend_item,
    "05_AUDITORIA_CAPACIDADE": auditoria_capacidade,
    "06_ALTERNATIVAS_ITEM": alternativas_item,
    "07_RASTRO_RATEIO": rastro_rateio,
    "08_DIAGNOSTICO_ITEM": diagnostico_item,
    "09_DIAGNOSTICO_ALTERNATIVAS": diagnostico_alternativas,
    "10_ESTRUTURA_EXPLODIDA": estrutura_explodida,
    "11_NEC_EXPLODIDA_COMPONENTES": nec_explodida_componentes,
    "12_NEC_CONSOLIDADA_SOLVER": nec_consolidada_solver,
    "13_AJUSTE_PAI_FILHO": ajuste_pai_filho,
    "14_RESUMO_PAI_FILHO": resumo_pai_filho,
    "15_CAPACIDADE_LIBERADA_PF": capacidade_liberada_pf_export,
    "16_REALOCACAO_CAP_LIBERADA": realocacao_cap_liberada,
    "18_RESUMO_ITEM_REALOCACAO": resumo_item_realocacao,
}

_guias_ativas = [nome for nome, ativo in GUIAS_EXPORTAR.items() if ativo]
if not _guias_ativas:
    raise RuntimeError("NENHUMA_GUIA_ATIVA_PARA_EXPORTACAO")

with pd.ExcelWriter(OUTPUT_SOLVER_FILE, engine="openpyxl") as writer:
    for _nome_guia in _guias_ativas:
        _GUIAS_DADOS_EXPORTAR[_nome_guia].to_excel(writer, sheet_name=_nome_guia, index=False)

# Formatação básica
wb = load_workbook(OUTPUT_SOLVER_FILE)
header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)
for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for col_cells in ws.columns:
        max_len = 0
        col_letter = col_cells[0].column_letter
        for cell in col_cells[:1000]:
            if cell.value is not None:
                max_len = max(max_len, min(len(str(cell.value)), 60))
        ws.column_dimensions[col_letter].width = min(max(max_len + 2, 10), 45)
wb.save(OUTPUT_SOLVER_FILE)


_finalizar_etapa_v58("EXPORTACAO_EXCEL")
if 'CURVA_ABC' in plano_producao.columns and plano_producao['CURVA_ABC'].isna().all():
    print('       ATENCAO - CURVA_ABC nao encontrada/preenchida na base de entrada')
if 'CURVA_123' in plano_producao.columns and plano_producao['CURVA_123'].isna().all():
    print('       ATENCAO - CURVA_123 nao encontrada/preenchida na base de entrada')
print(f"       OK - Arquivo gravado: {OUTPUT_SOLVER_FILE.name}")


# 05. Finalização


In [ ]:
print("[04/04] Finalizando e consolidando tempos...")
tempo_total = time.perf_counter() - _TEMPO_TOTAL_INICIO_V58
segundos_total = int(round(tempo_total))
horas, resto = divmod(segundos_total, 3600)
minutos, segundos = divmod(resto, 60)
print("=" * 60)
print(" PROCESSAMENTO CONCLUIDO COM SUCESSO")
print(f" Tempo total: {horas:02d}:{minutos:02d}:{segundos:02d}")
for _nome_v58, _seg_v58 in TEMPOS_ETAPAS_V58.items():
    print(f" {_nome_v58}: {_seg_v58:.2f}s")

if 'LOG_TEMPO_PROCESSAMENTO_M1' in globals() and isinstance(LOG_TEMPO_PROCESSAMENTO_M1, pd.DataFrame) and not LOG_TEMPO_PROCESSAMENTO_M1.empty:
    _tempo_loop_total = LOG_TEMPO_PROCESSAMENTO_M1.loc[LOG_TEMPO_PROCESSAMENTO_M1['ETAPA'].eq('LOOP_MATRIZ_TOTAL'), 'TEMPO_SEG']
    if not _tempo_loop_total.empty:
        print(f" LOOP_MATRIZ_M1: {_tempo_loop_total.iloc[-1]:.2f}s")
print("=" * 60)
